In [0]:
# =============================================================================
# LOAD DATA ONCE
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the full table
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

print(f"Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# =============================================================================
# DATA PREPARATION
# =============================================================================

# Convert GMV to float
df['gmv_l12mo'] = df['gmv_l12mo'].astype(float)

# Create 4 GMV tiers using quartiles (P25, P50, P75, P100)
active = df[df['gmv_l12mo'] > 0]
p25 = active['gmv_l12mo'].quantile(0.25)
p50 = active['gmv_l12mo'].quantile(0.50)
p75 = active['gmv_l12mo'].quantile(0.75)

print(f"\nGMV Quartiles (L12M):")
print(f"  P25: €{p25:,.2f}")
print(f"  P50: €{p50:,.2f}")
print(f"  P75: €{p75:,.2f}")

def get_gmv_tier(gmv):
    if gmv == 0:
        return 'No Revenue'
    elif gmv <= p25:
        return 'Q1: Bottom 25%'
    elif gmv <= p50:
        return 'Q2: 25-50%'
    elif gmv <= p75:
        return 'Q3: 50-75%'
    else:
        return 'Q4: Top 25%'

df['gmv_tier'] = df['gmv_l12mo'].apply(get_gmv_tier)

# Clean managed field
df['managed'] = df['is_managed'].astype(str).str.lower().replace({
    'yes': 'Managed', 'true': 'Managed', '1': 'Managed',
    'no': 'Not Managed', 'false': 'Not Managed', '0': 'Not Managed',
    'none': 'Not Managed', 'nan': 'Not Managed'
})

print(f"\nGMV Tier Distribution:")
print(df['gmv_tier'].value_counts().sort_index())

# =============================================================================
# DEFINE FEATURES AS BINARY COMPARISONS
# =============================================================================

# Create binary feature flags
df['has_individual'] = (df['has_individual_pricing'] == 1).astype(int)
df['has_group'] = (df['has_group_pricing'] == 1).astype(int)
df['has_addons'] = (df['has_addons'] == 1).astype(int)
df['has_options'] = (df['num_pricing_configs'] > 1).astype(int)  # Multiple pricing configs = options

# Price tiers / scales
df['has_any_scales'] = ((df['has_native_scales'] == 1) | (df['has_api_scales'] == 1)).astype(int)
df['no_scales'] = ((df['has_native_scales'] == 0) & (df['has_api_scales'] == 0)).astype(int)

# Prices over API
df['uses_api'] = (df['uses_api_pricing'] == 1).astype(int)
df['no_api'] = (df['uses_api_pricing'] == 0).astype(int)

# Scales over API
df['scales_via_api'] = (df['has_api_scales'] == 1).astype(int)
df['no_scales_via_api'] = (df['has_api_scales'] == 0).astype(int)

# Live pricing
df['has_live_pricing'] = (df['has_live_dynamic_pricing'] == 1).astype(int)
df['no_live_pricing'] = (df['has_live_dynamic_pricing'] == 0).astype(int)

# Define all features
FEATURES = {
    'Individual Pricing': 'has_individual',
    'Group Pricing': 'has_group',
    'Add-ons': 'has_addons',
    'Multiple Options': 'has_options',
    'Any Price Scales': 'has_any_scales',
    'No Price Scales': 'no_scales',
    'Uses API Pricing': 'uses_api',
    'No API Pricing': 'no_api',
    'Scales via API': 'scales_via_api',
    'No Scales via API': 'no_scales_via_api',
    'Live Pricing & Avail': 'has_live_pricing',
    'No Live Pricing': 'no_live_pricing'
}

# =============================================================================
# COLOR SCHEME: Orange to Grey
# =============================================================================

COLORS = ['#D35400', '#E67E22', '#F39C12', '#F8B500', '#555555', '#808080', '#AAAAAA', '#CCCCCC']

# =============================================================================
# ANALYSIS: GMV TIER BREAKDOWN
# =============================================================================

gmv_tiers = ['No Revenue', 'Q1: Bottom 25%', 'Q2: 25-50%', 'Q3: 50-75%', 'Q4: Top 25%']

print("\n" + "="*100)
print("GMV TIER ANALYSIS")
print("="*100)

for gmv_tier in gmv_tiers:
    tier_df = df[df['gmv_tier'] == gmv_tier]
    
    if len(tier_df) == 0:
        continue
    
    print(f"\n{'='*100}")
    print(f"GMV TIER: {gmv_tier}")
    print(f"{'='*100}")
    print(f"Total Tours: {len(tier_df):,}")
    print(f"Total Suppliers: {tier_df['supplier_id'].nunique():,}")
    print(f"Total GMV: €{tier_df['gmv_l12mo'].sum():,.2f}")
    
    # =============================================================================
    # BY SUPPLIER SEGMENT
    # =============================================================================
    
    print(f"\n--- BY SUPPLIER SEGMENT ---")
    
    segment_data = []
    for segment in sorted(tier_df['supplier_segment'].dropna().unique()):
        seg_df = tier_df[tier_df['supplier_segment'] == segment]
        row = {'segment': segment, 'tours': len(seg_df)}
        for feat_name, feat_col in FEATURES.items():
            if feat_col in seg_df.columns:
                pct = 100 * seg_df[feat_col].sum() / len(seg_df)
                row[feat_name] = round(pct, 1)
        segment_data.append(row)
    
    seg_df_results = pd.DataFrame(segment_data)
    print(seg_df_results.to_string(index=False))
    
    # Chart: Segment comparison
    if len(segment_data) > 0:
        fig, ax = plt.subplots(figsize=(14, 6))
        
        # Plot key features only for clarity
        key_features = ['Individual Pricing', 'Group Pricing', 'Add-ons', 'Any Price Scales', 'Uses API Pricing', 'Live Pricing & Avail']
        
        x = np.arange(len(seg_df_results))
        width = 0.12
        
        for i, feat in enumerate(key_features):
            if feat in seg_df_results.columns:
                offset = (i - 2.5) * width
                ax.bar(x + offset, seg_df_results[feat], width, label=feat, color=COLORS[i])
        
        ax.set_xlabel('Supplier Segment', fontsize=12, fontweight='bold')
        ax.set_ylabel('Adoption Rate (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'{gmv_tier}: Feature Adoption by Segment', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(seg_df_results['segment'], rotation=45, ha='right')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    # =============================================================================
    # BY ACTIVITY TYPE
    # =============================================================================
    
    print(f"\n--- BY ACTIVITY TYPE ---")
    
    activity_data = []
    for activity in sorted(tier_df['activity_type'].dropna().unique()):
        act_df = tier_df[tier_df['activity_type'] == activity]
        if len(act_df) >= 10:  # Only show activities with 10+ tours
            row = {'activity': activity, 'tours': len(act_df)}
            for feat_name, feat_col in FEATURES.items():
                if feat_col in act_df.columns:
                    pct = 100 * act_df[feat_col].sum() / len(act_df)
                    row[feat_name] = round(pct, 1)
            activity_data.append(row)
    
    if len(activity_data) > 0:
        act_df_results = pd.DataFrame(activity_data).sort_values('tours', ascending=False).head(10)
        print(act_df_results.to_string(index=False))
        
        # Chart: Activity type comparison
        fig, ax = plt.subplots(figsize=(14, 6))
        
        x = np.arange(len(act_df_results))
        width = 0.12
        
        for i, feat in enumerate(key_features):
            if feat in act_df_results.columns:
                offset = (i - 2.5) * width
                ax.bar(x + offset, act_df_results[feat], width, label=feat, color=COLORS[i])
        
        ax.set_xlabel('Activity Type', fontsize=12, fontweight='bold')
        ax.set_ylabel('Adoption Rate (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'{gmv_tier}: Feature Adoption by Activity Type (Top 10)', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(act_df_results['activity'], rotation=45, ha='right')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    # =============================================================================
    # BY MANAGED STATUS
    # =============================================================================
    
    print(f"\n--- BY MANAGED STATUS ---")
    
    managed_data = []
    for managed_status in ['Managed', 'Not Managed']:
        mgd_df = tier_df[tier_df['managed'] == managed_status]
        if len(mgd_df) > 0:
            row = {'managed': managed_status, 'tours': len(mgd_df)}
            for feat_name, feat_col in FEATURES.items():
                if feat_col in mgd_df.columns:
                    pct = 100 * mgd_df[feat_col].sum() / len(mgd_df)
                    row[feat_name] = round(pct, 1)
            managed_data.append(row)
    
    mgd_df_results = pd.DataFrame(managed_data)
    print(mgd_df_results.to_string(index=False))
    
    # Chart: Managed vs Not Managed
    if len(managed_data) == 2:
        fig, ax = plt.subplots(figsize=(12, 6))
        
        x = np.arange(len(key_features))
        width = 0.35
        
        managed_vals = [mgd_df_results[mgd_df_results['managed'] == 'Managed'][feat].values[0] 
                       if feat in mgd_df_results.columns else 0 
                       for feat in key_features]
        not_managed_vals = [mgd_df_results[mgd_df_results['managed'] == 'Not Managed'][feat].values[0] 
                           if feat in mgd_df_results.columns else 0 
                           for feat in key_features]
        
        ax.bar(x - width/2, managed_vals, width, label='Managed', color='#E67E22')
        ax.bar(x + width/2, not_managed_vals, width, label='Not Managed', color='#808080')
        
        ax.set_xlabel('Feature', fontsize=12, fontweight='bold')
        ax.set_ylabel('Adoption Rate (%)', fontsize=12, fontweight='bold')
        ax.set_title(f'{gmv_tier}: Managed vs Not Managed', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(key_features, rotation=45, ha='right')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

print("\n✓ GMV Tier Analysis Complete")


In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ============================================================
# 0) FILTERS (set these before running)
# ============================================================
FILTERS = {
    "connected_status": None,      # "Connected" / "Not connected" / None
    "managed_status": None,        # "Managed" / "Not Managed" / None (kept separate in case later you split them)
    "supplier_segment": None,      # e.g. ["Enterprise", "SMB"] or None
    "market": None,                # location_name values, e.g. ["Paris", "Rome"] or None
    "activity_type": None,         # e.g. ["Walking tour"] or None
}

# ============================================================
# 1) LOAD FULL DATA ONCE
# ============================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()
print(f"Loaded rows (tours): {len(df):,} | suppliers: {df['supplier_id'].nunique():,}")

# ============================================================
# 2) TYPE FIXES (Databricks often brings decimals)
# ============================================================
NUM_COLS = [
    "gmv_l12mo", "bookings_l12mo", "nr_l12mo", "tickets_l12mo",
    "num_pricing_configs", "num_individual_categories", "num_individual_tiers",
    "num_group_configs", "num_group_tiers", "max_group_tiers",
    "num_addons", "num_addon_tiers",
    "avg_cutoff_hours", "avg_pct_dates_with_vacancy",
]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# ============================================================
# 3) STANDARDIZE CONNECTED / MANAGED TEXT FIELDS
#    You said "connected vs not connected" is the same field as managed (yes/no text).
#    We'll map is_managed -> both fields for now (kept separate so you can change later).
# ============================================================
def normalize_yes_no(x):
    s = str(x).strip().lower()
    if s in {"yes", "y", "true", "1"}:
        return "Yes"
    if s in {"no", "n", "false", "0", "none", "nan"}:
        return "No"
    return "Unknown"

df["is_managed_norm"] = df["is_managed"].apply(normalize_yes_no) if "is_managed" in df.columns else "Unknown"

# Connected / not connected (as you defined it)
df["connected_status"] = df["is_managed_norm"].map({"Yes": "Connected", "No": "Not connected"}).fillna("Unknown")

# Managed / not managed (same source for now)
df["managed_status"] = df["is_managed_norm"].map({"Yes": "Managed", "No": "Not Managed"}).fillna("Unknown")

# ============================================================
# 4) APPLY FILTERS (optional)
# ============================================================
def apply_filters(d):
    out = d.copy()

    if FILTERS["connected_status"] is not None:
        out = out[out["connected_status"].isin([FILTERS["connected_status"]] if isinstance(FILTERS["connected_status"], str) else FILTERS["connected_status"])]

    if FILTERS["managed_status"] is not None:
        out = out[out["managed_status"].isin([FILTERS["managed_status"]] if isinstance(FILTERS["managed_status"], str) else FILTERS["managed_status"])]

    if FILTERS["supplier_segment"] is not None and "supplier_segment" in out.columns:
        segs = FILTERS["supplier_segment"]
        out = out[out["supplier_segment"].isin([segs] if isinstance(segs, str) else segs)]

    if FILTERS["market"] is not None and "location_name" in out.columns:
        mkts = FILTERS["market"]
        out = out[out["location_name"].isin([mkts] if isinstance(mkts, str) else mkts)]

    if FILTERS["activity_type"] is not None and "activity_type" in out.columns:
        acts = FILTERS["activity_type"]
        out = out[out["activity_type"].isin([acts] if isinstance(acts, str) else acts)]

    return out

df = apply_filters(df)
print(f"After filters: rows (tours): {len(df):,} | suppliers: {df['supplier_id'].nunique():,}")

# ============================================================
# 5) BUILD 4 GMV TIERS (percentiles among tours with GMV>0)
#    You asked for tiers that represent what they are: percentile buckets.
# ============================================================
df["gmv_l12mo"] = df["gmv_l12mo"].fillna(0.0)

active = df[df["gmv_l12mo"] > 0].copy()
if len(active) == 0:
    raise ValueError("No tours with gmv_l12mo > 0 after filters. Remove filters or check gmv_l12mo.")

p25 = active["gmv_l12mo"].quantile(0.25)
p50 = active["gmv_l12mo"].quantile(0.50)
p75 = active["gmv_l12mo"].quantile(0.75)

print("GMV (L12M) cutoffs among tours with GMV > 0:")
print(f"  25th percentile: {p25:,.2f}")
print(f"  50th percentile: {p50:,.2f}")
print(f"  75th percentile: {p75:,.2f}")

def gmv_tier_4(g):
    if g <= 0:
        return "No GMV in last 12 months"
    if g <= p25:
        return "GMV percentile 0-25 (among GMV>0)"
    if g <= p50:
        return "GMV percentile 25-50 (among GMV>0)"
    if g <= p75:
        return "GMV percentile 50-75 (among GMV>0)"
    return "GMV percentile 75-100 (among GMV>0)"

df["gmv_tier_4"] = df["gmv_l12mo"].apply(gmv_tier_4)

TIER_ORDER = [
    "No GMV in last 12 months",
    "GMV percentile 0-25 (among GMV>0)",
    "GMV percentile 25-50 (among GMV>0)",
    "GMV percentile 50-75 (among GMV>0)",
    "GMV percentile 75-100 (among GMV>0)",
]

# ============================================================
# 6) DEFINE FEATURES (explicit, separate areas like you described)
#    Everything is a separate feature flag; we do NOT collapse individual/group.
# ============================================================
# If you have a better "tour options" definition, swap it here.
# Using num_pricing_configs > 1 as a proxy for multiple options/configurations.
df["has_multiple_pricing_configurations"] = (
    (pd.to_numeric(df.get("num_pricing_configs", 0), errors="coerce").fillna(0) > 1).astype(int)
)

FEATURES = [
    ("Individual pricing configured", "has_individual_pricing"),
    ("Group pricing configured", "has_group_pricing"),
    ("Add-ons configured", "has_addons"),
    ("Multiple pricing configurations (proxy for options)", "has_multiple_pricing_configurations"),
    ("Individual pricing has multiple price tiers (scales)", "has_individual_scales"),
    ("Group pricing has multiple price tiers (scales)", "has_group_scales"),
    ("Add-ons have multiple price tiers (scales)", "has_addon_scales"),
    ("Any pricing scales (native or API)", "has_scale_pricing"),
    ("Pricing scales over API", "has_api_scales"),
    ("Pricing scales not over API (native)", "has_native_scales"),
    ("Prices provided over API (any)", "uses_api_pricing"),
    ("Static API pricing (not live)", "has_static_api_pricing"),
    ("Live pricing and availability", "has_live_dynamic_pricing"),
]

# Ensure missing columns become 0 (so code doesn't break)
for _, col in FEATURES:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

FEATURE_DEFINITIONS = {
    "Individual pricing configured": "Tour has per-ticket-category pricing configured (for example Adult, Child).",
    "Group pricing configured": "Tour has group pricing configured (prices depend on group size or group rules).",
    "Add-ons configured": "Tour offers paid add-ons.",
    "Multiple pricing configurations (proxy for options)": "Proxy: more than 1 pricing configuration snapshot/config on the tour (num_pricing_configs > 1).",
    "Individual pricing has multiple price tiers (scales)": "At least one individual ticket category has more than one tier.",
    "Group pricing has multiple price tiers (scales)": "At least one group config has more than one tier.",
    "Add-ons have multiple price tiers (scales)": "At least one add-on has more than one tier.",
    "Any pricing scales (native or API)": "Any multi-tier pricing exists (native or via API).",
    "Pricing scales over API": "Multi-tier pricing is provided via API.",
    "Pricing scales not over API (native)": "Multi-tier pricing exists and is not provided via API.",
    "Prices provided over API (any)": "Tour’s prices are provided via API.",
    "Static API pricing (not live)": "Prices via API but not live availability and price.",
    "Live pricing and availability": "Live availability and price is enabled.",
}

# Print definitions once (meant to be copy-pastable into a dashboard tooltip/definitions panel)
print("\nMetric / Feature definitions:")
for k, v in FEATURE_DEFINITIONS.items():
    print(f" - {k}: {v}")

# ============================================================
# 7) AGGREGATE: ADOPTION RATE BY GMV TIER (the core of Analysis 1)
# ============================================================
rows = []
for tier in TIER_ORDER:
    d = df[df["gmv_tier_4"] == tier]
    if len(d) == 0:
        continue

    row = {
        "GMV tier": tier,
        "Tours": len(d),
        "Suppliers": d["supplier_id"].nunique(),
        "Total GMV (L12M)": float(d["gmv_l12mo"].sum()),
    }
    for feat_name, col in FEATURES:
        row[feat_name] = 100.0 * d[col].mean()  # mean of 0/1 => adoption %
    rows.append(row)

adopt = pd.DataFrame(rows)

# Keep tiers in order
adopt["GMV tier"] = pd.Categorical(adopt["GMV tier"], categories=TIER_ORDER, ordered=True)
adopt = adopt.sort_values("GMV tier")

# ============================================================
# 8) VISUALS (orange <-> grey only)
# ============================================================
# Custom grey -> orange colormap for heatmap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    ["#D0D0D0", "#9E9E9E", "#F6B26B", "#F39C12", "#D35400"],
    N=256
)

# --- Chart A: Distribution context (Tours by GMV tier)
fig = plt.figure(figsize=(12, 4))
x = np.arange(len(adopt))
plt.bar(x, adopt["Tours"], color="#9E9E9E")
plt.xticks(x, adopt["GMV tier"], rotation=25, ha="right")
plt.ylabel("Number of tours")
plt.title("Tours by GMV tier (last 12 months)")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

# --- Chart B: Heatmap of adoption by feature x GMV tier
feature_names = [name for name, _ in FEATURES]
mat = adopt[feature_names].to_numpy().T  # rows=features, cols=tiers

fig = plt.figure(figsize=(12, 7))
im = plt.imshow(mat, aspect="auto", cmap=cmap, vmin=0, vmax=100)

plt.yticks(np.arange(len(feature_names)), feature_names)
plt.xticks(np.arange(len(adopt)), adopt["GMV tier"], rotation=25, ha="right")
plt.title("Feature adoption rate (%) by GMV tier (last 12 months)")

# annotate values
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat[i, j]
        txt_color = "black" if val < 60 else "white"
        plt.text(j, i, f"{val:.0f}", ha="center", va="center", color=txt_color, fontsize=9)

cbar = plt.colorbar(im)
cbar.set_label("Adoption rate (%)")
plt.tight_layout()
plt.show()

# --- Chart C: Focus view for leadership (only the 6 most important features)
focus_features = [
    "Individual pricing configured",
    "Group pricing configured",
    "Add-ons configured",
    "Any pricing scales (native or API)",
    "Prices provided over API (any)",
    "Live pricing and availability",
]
focus = adopt[["GMV tier"] + focus_features].copy()

fig = plt.figure(figsize=(12, 5))
x = np.arange(len(focus))
width = 0.13

palette = ["#D35400", "#F39C12", "#F6B26B", "#9E9E9E", "#808080", "#B0B0B0"]  # orange -> greys

for i, f in enumerate(focus_features):
    plt.bar(x + (i - (len(focus_features)-1)/2) * width, focus[f], width=width, color=palette[i], label=f)

plt.xticks(x, focus["GMV tier"], rotation=25, ha="right")
plt.ylabel("Adoption rate (%)")
plt.title("Adoption rate (%) by GMV tier for key pricing features")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# PARAMETERS (edit these, re-run)
# =============================================================================
DASH = {
    # IMPORTANT: pick ONE supplier segment at a time for interpretation.
    # Set to None to auto-pick top segments by GMV for plotting.
    "segments_to_plot": None,     # e.g. ["SMB", "Enterprise"] or ["Segment A"] or None
    "top_k_segments": 3,          # only used when segments_to_plot is None

    # Drill-down selectors (used in Market/Activity charts)
    "segment_for_drill": None,    # if None, uses first from segments_to_plot
    "supplier_size_bucket_for_drill": "Supplier GMV percentile 75-100 (among GMV>0)",
    "feature_for_drill": "Prices provided over API",

    # Top-N for drilldowns
    "top_n_markets": 15,
    "top_n_activity_types": 15,

    # Minimum group size to chart (avoid noise)
    "min_tours_per_group": 50,
}

# =============================================================================
# STYLE: orange ↔ grey only
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#F39C12"
ORANGE_LIGHT = "#F6B26B"
GREY_DARK    = "#555555"
GREY         = "#808080"
GREY_LIGHT   = "#B0B0B0"
GREY_PALE    = "#D0D0D0"

GREY_TO_ORANGE = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

def _grid_y(ax): ax.grid(axis="y", alpha=0.2)
def _rotate_xticks(ax, deg=25):
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(deg); lbl.set_ha("right")

# =============================================================================
# 1) LOAD ONCE
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()
print(f"Loaded tours: {len(df):,} | suppliers: {df['supplier_id'].nunique():,}")

# =============================================================================
# 2) TYPE FIXES (Decimals -> numeric)
# =============================================================================
NUM_COLS = [
    "gmv_l12mo", "bookings_l12mo", "nr_l12mo", "tickets_l12mo",
    "total_options",
    "num_addons", "num_addon_tiers",
    "num_individual_categories", "num_individual_tiers",
    "num_group_configs", "num_group_tiers", "max_group_tiers",
    "avg_cutoff_hours", "avg_pct_dates_with_vacancy",
    "num_pricing_configs",
]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

df["gmv_l12mo"] = df["gmv_l12mo"].fillna(0.0)
df["total_options"] = pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0).astype(int)
df["num_addons"] = pd.to_numeric(df.get("num_addons", 0), errors="coerce").fillna(0).astype(int)

# =============================================================================
# 3) CONNECTED / MANAGED (your current rule: connected == is_managed yes/no)
# =============================================================================
def normalize_yes_no(x):
    s = str(x).strip().lower()
    if s in {"yes", "y", "true", "1"}: return "Yes"
    if s in {"no", "n", "false", "0", "none", "nan"}: return "No"
    return "Unknown"

df["is_managed_norm"] = df["is_managed"].apply(normalize_yes_no) if "is_managed" in df.columns else "Unknown"
df["connected_status"] = df["is_managed_norm"].map({"Yes": "Connected", "No": "Not connected"}).fillna("Unknown")
df["managed_status"]   = df["is_managed_norm"].map({"Yes": "Managed", "No": "Not Managed"}).fillna("Unknown")

# =============================================================================
# 4) SUPPLIER SIZE BUCKETS (supplier GMV L12M percentiles)
#    This matches "supplier size means GMV" better than tour-level buckets.
# =============================================================================
supplier_gmv = (
    df.groupby("supplier_id", dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

active_sup = supplier_gmv[supplier_gmv["supplier_gmv_l12mo"] > 0].copy()
if len(active_sup) == 0:
    raise ValueError("No suppliers with supplier_gmv_l12mo > 0. Check data.")

q25 = active_sup["supplier_gmv_l12mo"].quantile(0.25)
q50 = active_sup["supplier_gmv_l12mo"].quantile(0.50)
q75 = active_sup["supplier_gmv_l12mo"].quantile(0.75)

def supplier_size_bucket(x):
    if x <= 0: return "No supplier GMV in last 12 months"
    if x <= q25: return "Supplier GMV percentile 0-25 (among GMV>0)"
    if x <= q50: return "Supplier GMV percentile 25-50 (among GMV>0)"
    if x <= q75: return "Supplier GMV percentile 50-75 (among GMV>0)"
    return "Supplier GMV percentile 75-100 (among GMV>0)"

supplier_gmv["supplier_size_bucket"] = supplier_gmv["supplier_gmv_l12mo"].apply(supplier_size_bucket)

SIZE_BUCKET_ORDER = [
    "No supplier GMV in last 12 months",
    "Supplier GMV percentile 0-25 (among GMV>0)",
    "Supplier GMV percentile 25-50 (among GMV>0)",
    "Supplier GMV percentile 50-75 (among GMV>0)",
    "Supplier GMV percentile 75-100 (among GMV>0)",
]

df = df.merge(supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "supplier_size_bucket"]], on="supplier_id", how="left")
df["supplier_size_bucket"] = df["supplier_size_bucket"].fillna("No supplier GMV in last 12 months")

print("Supplier GMV bucket cutoffs:")
print(f"  P25: {q25:,.2f} | P50: {q50:,.2f} | P75: {q75:,.2f}")

# =============================================================================
# 5) FEATURES (your full pool) + complements
#    Changes:
#      - tour options uses total_options (no proxy)
#      - add "not provided over API" for pricing and for scales
# =============================================================================
df["Tour options configured"] = (df["total_options"] > 0).astype(int)

df["Pricing scales NOT provided over API"] = (df["has_api_scales"] == 0).astype(int)   # complement (includes no-scales)
df["Prices NOT provided over API"] = (df["uses_api_pricing"] == 0).astype(int)         # complement

FEATURES = [
    ("Individual pricing configured", "has_individual_pricing"),
    ("Group pricing configured", "has_group_pricing"),
    ("Add-ons configured", "has_addons"),
    ("Tour options configured", "Tour options configured"),

    ("Individual pricing has multiple price tiers", "has_individual_scales"),
    ("Group pricing has multiple price tiers", "has_group_scales"),
    ("Add-ons have multiple price tiers", "has_addon_scales"),

    ("Any pricing scales (native or over API)", "has_scale_pricing"),
    ("Pricing scales provided over API", "has_api_scales"),
    ("Pricing scales not provided over API (native)", "has_native_scales"),
    ("Pricing scales NOT provided over API", "Pricing scales NOT provided over API"),

    ("Prices provided over API", "uses_api_pricing"),
    ("Prices NOT provided over API", "Prices NOT provided over API"),
    ("Static API pricing (not live)", "has_static_api_pricing"),
    ("Live pricing and availability", "has_live_dynamic_pricing"),
]

# Make safe
for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

FEATURE_NAMES = [n for n, _ in FEATURES]

# =============================================================================
# 6) HELPERS: segment selection + aggregation
# =============================================================================
def top_segments_by_gmv(d, k):
    seg = (
        d.groupby("supplier_segment", dropna=False)["gmv_l12mo"]
         .sum()
         .reset_index()
         .sort_values("gmv_l12mo", ascending=False)
    )
    seg_list = seg["supplier_segment"].dropna().astype(str).head(k).tolist()
    return seg_list

def adoption_matrix(d, group_col):
    """
    Returns:
      - matrix (features x buckets) with adoption rates
      - counts by bucket
    """
    buckets = SIZE_BUCKET_ORDER
    rows = []
    counts = []

    for b in buckets:
        g = d[d[group_col] == b]
        counts.append({"bucket": b, "tours": len(g), "suppliers": g["supplier_id"].nunique(), "total_gmv": float(g["gmv_l12mo"].sum())})
        row = []
        for _, col in FEATURES:
            row.append(100.0 * float(g[col].mean()) if len(g) > 0 else np.nan)
        rows.append(row)

    mat = np.array(rows).T  # features x buckets
    return mat, pd.DataFrame(counts)

def plot_heatmap(mat, buckets, title):
    fig, ax = plt.subplots(figsize=(13, 7))
    im = ax.imshow(mat, aspect="auto", cmap=GREY_TO_ORANGE, vmin=0, vmax=100)

    ax.set_yticks(np.arange(len(FEATURE_NAMES)))
    ax.set_yticklabels(FEATURE_NAMES)

    ax.set_xticks(np.arange(len(buckets)))
    ax.set_xticklabels(buckets, rotation=25, ha="right")

    ax.set_title(title)

    # annotate
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if np.isnan(v):
                continue
            ax.text(j, i, f"{v:.0f}", ha="center", va="center",
                    color=("white" if v >= 60 else "black"), fontsize=8)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Adoption rate (%)")
    plt.tight_layout()
    plt.show()

def plot_bucket_context(counts_df, title_prefix):
    c = counts_df.copy()
    c["bucket"] = pd.Categorical(c["bucket"], categories=SIZE_BUCKET_ORDER, ordered=True)
    c = c.sort_values("bucket")

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(c["bucket"].astype(str), c["total_gmv"], color=GREY)
    ax.set_title(f"{title_prefix} — Total GMV (L12M) by supplier-size bucket")
    ax.set_xlabel("Supplier-size bucket (by supplier GMV percentiles)")
    ax.set_ylabel("Total GMV (L12M)")
    _rotate_xticks(ax, 25); _grid_y(ax)
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(c["bucket"].astype(str), c["tours"], color=GREY_LIGHT)
    ax.set_title(f"{title_prefix} — Tours by supplier-size bucket")
    ax.set_xlabel("Supplier-size bucket (by supplier GMV percentiles)")
    ax.set_ylabel("Tours")
    _rotate_xticks(ax, 25); _grid_y(ax)
    plt.tight_layout(); plt.show()

def topN_bar_by_dim(d, dim_col, feature_col, title, top_n, min_tours):
    """
    Bar chart for top-N groups by tour count.
    Adoption computed as mean(feature) * 100.
    """
    g = (
        d.groupby(dim_col, dropna=False)
         .agg(tours=("tour_id", "count"), adoption=(feature_col, "mean"))
         .reset_index()
    )
    g = g[g["tours"] >= min_tours].copy()
    if len(g) == 0:
        print(f"Skipped: not enough data for {title} (after min_tours={min_tours}).")
        return

    g["adoption_pct"] = 100.0 * g["adoption"]
    g = g.sort_values("tours", ascending=False).head(top_n).sort_values("adoption_pct", ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(g[dim_col].astype(str), g["adoption_pct"], color=ORANGE)
    ax.set_title(title)
    ax.set_xlabel("Adoption rate (%)")
    ax.set_ylabel(dim_col)
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.show()

# =============================================================================
# 7) SEGMENT-FIRST DASHBOARD PAGES
#    For each segment: context + heatmaps split by managed_status
# =============================================================================
segments_to_plot = DASH["segments_to_plot"]
if segments_to_plot is None:
    segments_to_plot = top_segments_by_gmv(df, DASH["top_k_segments"])

print("Segments to plot (do not interpret cross-segment differences):")
for s in segments_to_plot:
    print(" -", s)

for seg in segments_to_plot:
    ds = df[df["supplier_segment"].astype(str) == str(seg)].copy()
    if len(ds) == 0:
        continue

    seg_title = f"Segment: {seg}"

    # Context charts (stakeholders need this to interpret adoption)
    mat_all, counts_all = adoption_matrix(ds, "supplier_size_bucket")
    plot_bucket_context(counts_all.rename(columns={"bucket":"bucket"}), seg_title)

    # Heatmap: Managed
    d_m = ds[ds["managed_status"] == "Managed"].copy()
    mat_m, _ = adoption_matrix(d_m, "supplier_size_bucket")
    plot_heatmap(
        mat_m,
        SIZE_BUCKET_ORDER,
        f"{seg_title} | Managed — Feature adoption (%) by supplier-size bucket"
    )

    # Heatmap: Not Managed
    d_nm = ds[ds["managed_status"] == "Not Managed"].copy()
    mat_nm, _ = adoption_matrix(d_nm, "supplier_size_bucket")
    plot_heatmap(
        mat_nm,
        SIZE_BUCKET_ORDER,
        f"{seg_title} | Not Managed — Feature adoption (%) by supplier-size bucket"
    )

# =============================================================================
# 8) DRILLDOWN PAGE (Market + Activity type) WITHIN ONE SEGMENT & ONE SIZE BUCKET
#    This is what stakeholders learn that they usually do not know:
#    which markets or activity types are lagging for a specific feature within a comparable cohort.
# =============================================================================
segment_for_drill = DASH["segment_for_drill"] or (segments_to_plot[0] if len(segments_to_plot) else None)
if segment_for_drill is None:
    raise ValueError("No segment available for drilldown.")

bucket_for_drill = DASH["supplier_size_bucket_for_drill"]
feature_for_drill = DASH["feature_for_drill"]

# Map feature name -> column
feat_map = {n: c for n, c in FEATURES}
if feature_for_drill not in feat_map:
    raise ValueError(f"feature_for_drill must be one of: {list(feat_map.keys())}")

feat_col = feat_map[feature_for_drill]

dd = df[
    (df["supplier_segment"].astype(str) == str(segment_for_drill)) &
    (df["supplier_size_bucket"] == bucket_for_drill)
].copy()

print(f"Drilldown cohort: segment={segment_for_drill} | bucket={bucket_for_drill} | tours={len(dd):,}")

# Market drilldown
topN_bar_by_dim(
    dd, "location_name", feat_col,
    title=f"{segment_for_drill} | {bucket_for_drill} — {feature_for_drill} adoption by market (top {DASH['top_n_markets']} by tours)",
    top_n=DASH["top_n_markets"],
    min_tours=DASH["min_tours_per_group"]
)

# Activity type drilldown
topN_bar_by_dim(
    dd, "activity_type", feat_col,
    title=f"{segment_for_drill} | {bucket_for_drill} — {feature_for_drill} adoption by activity type (top {DASH['top_n_activity_types']} by tours)",
    top_n=DASH["top_n_activity_types"],
    min_tours=DASH["min_tours_per_group"]
)

print("Done.")


In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# SETTINGS (you MUST pick one segment to interpret within-segment)
# =============================================================================
SEGMENT = None  # e.g. "SMB", "Enterprise", "Long tail" (set this)
SIZE_BUCKET_FOR_GAPS = "Supplier GMV percentile 75-100 (among GMV>0)"  # for managed vs not managed gap chart
DRILL_FEATURE = "Prices provided over API"  # for market/activity outlier charts
DRILL_SIZE_BUCKET = "Supplier GMV percentile 75-100 (among GMV>0)"
TOP_N = 15
MIN_TOURS_PER_GROUP = 80

# =============================================================================
# STYLE (orange ↔ grey only)
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#F39C12"
ORANGE_LIGHT = "#F6B26B"
GREY_DARK    = "#555555"
GREY         = "#808080"
GREY_LIGHT   = "#B0B0B0"
GREY_PALE    = "#D0D0D0"

CMAP = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

def rotate_xticks(ax, deg=25):
    for t in ax.get_xticklabels():
        t.set_rotation(deg); t.set_ha("right")

def grid_y(ax):
    ax.grid(axis="y", alpha=0.2)

# =============================================================================
# 1) LOAD ONCE
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()
print(f"Loaded tours: {len(df):,} | suppliers: {df['supplier_id'].nunique():,}")

# =============================================================================
# 2) TYPE FIXES
# =============================================================================
NUM_COLS = [
    "gmv_l12mo", "bookings_l12mo", "nr_l12mo", "tickets_l12mo",
    "total_options", "num_addons", "num_addon_tiers",
    "num_individual_categories", "num_individual_tiers",
    "num_group_configs", "num_group_tiers", "max_group_tiers",
]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

df["gmv_l12mo"] = pd.to_numeric(df.get("gmv_l12mo", 0), errors="coerce").fillna(0.0)
df["total_options"] = pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0).astype(int)
df["num_addons"] = pd.to_numeric(df.get("num_addons", 0), errors="coerce").fillna(0).astype(int)

# =============================================================================
# 3) CONNECTED / MANAGED (your current rule)
# =============================================================================
def normalize_yes_no(x):
    s = str(x).strip().lower()
    if s in {"yes", "y", "true", "1"}: return "Yes"
    if s in {"no", "n", "false", "0", "none", "nan"}: return "No"
    return "Unknown"

df["is_managed_norm"] = df["is_managed"].apply(normalize_yes_no) if "is_managed" in df.columns else "Unknown"
df["connected_status"] = df["is_managed_norm"].map({"Yes": "Connected", "No": "Not connected"}).fillna("Unknown")
df["managed_status"]   = df["is_managed_norm"].map({"Yes": "Managed", "No": "Not Managed"}).fillna("Unknown")

# =============================================================================
# 4) SUPPLIER SIZE BUCKETS (supplier GMV L12M percentiles)
# =============================================================================
supplier_gmv = (
    df.groupby("supplier_id", dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)
active_sup = supplier_gmv[supplier_gmv["supplier_gmv_l12mo"] > 0].copy()
if len(active_sup) == 0:
    raise ValueError("No suppliers with GMV>0 in last 12 months.")

q25 = active_sup["supplier_gmv_l12mo"].quantile(0.25)
q50 = active_sup["supplier_gmv_l12mo"].quantile(0.50)
q75 = active_sup["supplier_gmv_l12mo"].quantile(0.75)

def supplier_bucket(x):
    if x <= 0: return "No supplier GMV in last 12 months"
    if x <= q25: return "Supplier GMV percentile 0-25 (among GMV>0)"
    if x <= q50: return "Supplier GMV percentile 25-50 (among GMV>0)"
    if x <= q75: return "Supplier GMV percentile 50-75 (among GMV>0)"
    return "Supplier GMV percentile 75-100 (among GMV>0)"

supplier_gmv["supplier_size_bucket"] = supplier_gmv["supplier_gmv_l12mo"].apply(supplier_bucket)

SIZE_BUCKET_ORDER = [
    "No supplier GMV in last 12 months",
    "Supplier GMV percentile 0-25 (among GMV>0)",
    "Supplier GMV percentile 25-50 (among GMV>0)",
    "Supplier GMV percentile 50-75 (among GMV>0)",
    "Supplier GMV percentile 75-100 (among GMV>0)",
]

df = df.merge(
    supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "supplier_size_bucket"]],
    on="supplier_id",
    how="left"
)
df["supplier_size_bucket"] = df["supplier_size_bucket"].fillna("No supplier GMV in last 12 months")

# =============================================================================
# 5) FEATURES (your list) + REQUIRED complements
#    - Tour options uses total_options (real)
#    - Add-ons uses has_addons (adoption) and num_addons (depth)
#    - Complements:
#        * Prices NOT provided over API
#        * Pricing scales NOT provided over API
# =============================================================================
df["Tour options configured"] = (df["total_options"] > 0).astype(int)

# Complements (simple complements, as you requested)
df["Prices NOT provided over API"] = (df["uses_api_pricing"] == 0).astype(int)
df["Pricing scales NOT provided over API"] = (df["has_api_scales"] == 0).astype(int)

FEATURES = [
    ("Individual pricing configured", "has_individual_pricing"),
    ("Group pricing configured", "has_group_pricing"),
    ("Add-ons configured", "has_addons"),
    ("Tour options configured", "Tour options configured"),

    ("Individual pricing has multiple price tiers", "has_individual_scales"),
    ("Group pricing has multiple price tiers", "has_group_scales"),
    ("Add-ons have multiple price tiers", "has_addon_scales"),

    ("Any pricing scales (native or over API)", "has_scale_pricing"),
    ("Pricing scales provided over API", "has_api_scales"),
    ("Pricing scales not provided over API (native)", "has_native_scales"),
    ("Pricing scales NOT provided over API", "Pricing scales NOT provided over API"),

    ("Prices provided over API", "uses_api_pricing"),
    ("Prices NOT provided over API", "Prices NOT provided over API"),
    ("Static API pricing (not live)", "has_static_api_pricing"),
    ("Live pricing and availability", "has_live_dynamic_pricing"),
]

feat_map = {n: c for n, c in FEATURES}
for n, c in FEATURES:
    if c not in df.columns:
        df[c] = 0
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# 6) ENFORCE WITHIN-SEGMENT VIEW
# =============================================================================
if SEGMENT is None:
    # Auto-pick top segment by GMV to prevent accidental cross-segment comparison
    seg_rank = (df.groupby("supplier_segment")["gmv_l12mo"].sum().sort_values(ascending=False))
    SEGMENT = str(seg_rank.index[0])
    print(f"SEGMENT not set. Using top-GMV segment automatically: {SEGMENT}")

ds = df[df["supplier_segment"].astype(str) == str(SEGMENT)].copy()
print(f"Working cohort (within segment): {SEGMENT} | tours={len(ds):,} | suppliers={ds['supplier_id'].nunique():,}")

# =============================================================================
# 7) ADVANCED, DECISION-USEFUL CHARTS
# =============================================================================

# ---------- Helper: Reach + Activation (director-friendly) ----------
# Reach: % of tours that are eligible for a feature
# Activation: % of eligible tours that have it configured
#
# Eligibility definitions:
# - Scales features are only eligible if the base pricing type exists
# - API features are eligible if there is pricing configured (individual or group); adjust if needed
# - Tour options & add-ons: eligible = all tours
def eligibility_mask(d, feature_name):
    if feature_name == "Individual pricing has multiple price tiers":
        return d["has_individual_pricing"] == 1
    if feature_name == "Group pricing has multiple price tiers":
        return d["has_group_pricing"] == 1
    if feature_name == "Add-ons have multiple price tiers":
        return d["has_addons"] == 1
    if feature_name in ["Pricing scales provided over API", "Pricing scales not provided over API (native)",
                        "Any pricing scales (native or over API)", "Pricing scales NOT provided over API"]:
        # eligible if any scale concept can apply -> if individual or group pricing exists
        return (d["has_individual_pricing"] == 1) | (d["has_group_pricing"] == 1)
    if feature_name in ["Prices provided over API", "Prices NOT provided over API",
                        "Static API pricing (not live)", "Live pricing and availability"]:
        # eligible if tour has some pricing configured at all
        return (d["has_individual_pricing"] == 1) | (d["has_group_pricing"] == 1)
    # default: all tours eligible
    return pd.Series(True, index=d.index)

def compute_reach_activation(d, group_cols):
    rows = []
    for key, g in d.groupby(group_cols, dropna=False):
        key_dict = dict(zip(group_cols, key if isinstance(key, tuple) else (key,)))
        total = len(g)
        total_gmv = float(g["gmv_l12mo"].sum())

        for feat_name, col in FEATURES:
            elig = eligibility_mask(g, feat_name)
            elig_n = int(elig.sum())
            adop_n = int(g.loc[elig, col].sum()) if elig_n > 0 else 0

            reach = 100.0 * elig_n / total if total > 0 else np.nan
            activation = 100.0 * adop_n / elig_n if elig_n > 0 else np.nan

            # GMV-weighted activation among eligible (better for prioritization)
            if elig_n > 0:
                w = g.loc[elig, "gmv_l12mo"].to_numpy()
                w_sum = float(w.sum())
                if w_sum > 0:
                    gmv_weighted_activation = 100.0 * float(np.average(g.loc[elig, col].to_numpy(), weights=w))
                else:
                    gmv_weighted_activation = np.nan
            else:
                gmv_weighted_activation = np.nan

            rows.append({
                **key_dict,
                "tours": total,
                "suppliers": g["supplier_id"].nunique(),
                "total_gmv_l12mo": total_gmv,
                "feature_name": feat_name,
                "reach_pct": reach,
                "activation_pct": activation,
                "activation_gmv_weighted_pct": gmv_weighted_activation,
            })
    return pd.DataFrame(rows)

# =============================================================================
# CHART 1 (Most important): Feature Adoption Funnel Map
# Question: For each feature, is the problem reach (eligibility) or activation (setup/usage)?
# Visualization: bubble scatter reach vs activation, within this segment and one supplier-size bucket.
# =============================================================================
bucket_focus = SIZE_BUCKET_FOR_GAPS
focus = ds[ds["supplier_size_bucket"] == bucket_focus].copy()
if len(focus) < 200:
    print(f"Warning: focus cohort is small (tours={len(focus):,}). Consider another bucket.")

ra = compute_reach_activation(focus, group_cols=["managed_status"])
# Keep only meaningful managed statuses
ra = ra[ra["managed_status"].isin(["Managed", "Not Managed"])].copy()

for ms in ["Managed", "Not Managed"]:
    sub = ra[ra["managed_status"] == ms].copy()
    sub = sub.dropna(subset=["reach_pct", "activation_pct"])
    # bubble size by GMV-weighted activation (so big bubbles highlight revenue-relevant adoption)
    sizes = np.nan_to_num(sub["activation_gmv_weighted_pct"].to_numpy(), nan=0.0)
    sizes = 50 + 6 * sizes  # scale

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(sub["reach_pct"], sub["activation_pct"], s=sizes, c=sub["activation_gmv_weighted_pct"],
               cmap=CMAP, vmin=0, vmax=100, edgecolors=GREY_DARK, linewidths=0.4)

    for _, r in sub.iterrows():
        ax.text(r["reach_pct"] + 0.6, r["activation_pct"] + 0.2, r["feature_name"], fontsize=8, color=GREY_DARK)

    ax.set_title(f"{SEGMENT} | {bucket_focus} | {ms}\nChart 1: Feature funnel map (Reach vs Activation)")
    ax.set_xlabel("Reach (% of tours eligible for the feature)")
    ax.set_ylabel("Activation (% of eligible tours configured)")
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 105)
    ax.grid(alpha=0.2)

    cbar = plt.colorbar(ax.collections[0], ax=ax)
    cbar.set_label("GMV-weighted activation among eligible (%)")
    plt.tight_layout()
    plt.show()

# =============================================================================
# CHART 2: Managed vs Not Managed gap (at SAME supplier-size bucket)
# Question: Which features show the largest managed uplift in activation (not raw adoption)?
# Visualization: sorted horizontal bar of activation delta.
# =============================================================================
ra2 = compute_reach_activation(focus, group_cols=["managed_status"])
ra2 = ra2[ra2["managed_status"].isin(["Managed", "Not Managed"])]

pivot = ra2.pivot(index="feature_name", columns="managed_status", values="activation_pct").reset_index()
pivot["delta_managed_minus_not"] = pivot["Managed"] - pivot["Not Managed"]
pivot = pivot.sort_values("delta_managed_minus_not", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(pivot["feature_name"], pivot["delta_managed_minus_not"], color=ORANGE)
ax.axvline(0, color=GREY_DARK, linewidth=1)
ax.set_title(f"{SEGMENT} | {bucket_focus}\nChart 2: Managed uplift (Activation delta, percentage points)")
ax.set_xlabel("Managed activation minus Not Managed activation (pp)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

# =============================================================================
# CHART 3: Pricing model mix by supplier-size bucket
# Question: Are suppliers in different size buckets using different pricing models (Individual only / Group only / Both / Neither)?
# Visualization: 100% stacked bars by supplier-size bucket.
# =============================================================================
mix = ds.copy()
mix["pricing_model"] = np.select(
    [
        (mix["has_individual_pricing"] == 1) & (mix["has_group_pricing"] == 1),
        (mix["has_individual_pricing"] == 1) & (mix["has_group_pricing"] == 0),
        (mix["has_individual_pricing"] == 0) & (mix["has_group_pricing"] == 1),
    ],
    ["Both individual and group pricing", "Individual pricing only", "Group pricing only"],
    default="Neither individual nor group pricing"
)

mix_g = (
    mix.groupby(["supplier_size_bucket", "pricing_model"], dropna=False)
       .size()
       .reset_index(name="tours")
)
mix_tot = mix_g.groupby("supplier_size_bucket")["tours"].sum().reset_index(name="total")
mix_g = mix_g.merge(mix_tot, on="supplier_size_bucket", how="left")
mix_g["share_pct"] = 100.0 * mix_g["tours"] / mix_g["total"]

# Order buckets
mix_g["supplier_size_bucket"] = pd.Categorical(mix_g["supplier_size_bucket"], categories=SIZE_BUCKET_ORDER, ordered=True)
mix_g = mix_g.sort_values("supplier_size_bucket")

bucket_list = [b for b in SIZE_BUCKET_ORDER if b in mix_g["supplier_size_bucket"].astype(str).unique()]
models = [
    "Neither individual nor group pricing",
    "Individual pricing only",
    "Group pricing only",
    "Both individual and group pricing",
]

colors = [GREY_LIGHT, ORANGE_LIGHT, ORANGE, ORANGE_DARK]

fig, ax = plt.subplots(figsize=(12, 5))
bottom = np.zeros(len(bucket_list))
for m, col in zip(models, colors):
    vals = []
    for b in bucket_list:
        v = mix_g[(mix_g["supplier_size_bucket"].astype(str) == b) & (mix_g["pricing_model"] == m)]["share_pct"]
        vals.append(float(v.iloc[0]) if len(v) else 0.0)
    ax.bar(bucket_list, vals, bottom=bottom, label=m, color=col, edgecolor=GREY_DARK, linewidth=0.2)
    bottom += np.array(vals)

ax.set_title(f"{SEGMENT}\nChart 3: Pricing model mix by supplier-size bucket (100% stacked)")
ax.set_ylabel("Share of tours (%)")
ax.set_xlabel("Supplier-size bucket")
rotate_xticks(ax, 25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
grid_y(ax)
plt.tight_layout()
plt.show()

# =============================================================================
# CHART 4: API split (conditional, not misleading complements)
# Question: Among eligible tours, what share uses API pricing vs not? Same for scales.
# Visualization: 100% stacked bars by supplier-size bucket.
# =============================================================================
def stacked_api_split(title, eligible_mask, api_col, native_col=None):
    d = ds[eligible_mask].copy()
    if len(d) == 0:
        print(f"Skipped: no eligible tours for {title}")
        return

    g = d.groupby("supplier_size_bucket").agg(
        tours=("tour_id", "count"),
        api_rate=(api_col, "mean"),
    ).reset_index()

    g["supplier_size_bucket"] = pd.Categorical(g["supplier_size_bucket"], categories=SIZE_BUCKET_ORDER, ordered=True)
    g = g.sort_values("supplier_size_bucket")

    # Share = api vs not api among eligible
    buckets = g["supplier_size_bucket"].astype(str).tolist()
    api_pct = 100.0 * g["api_rate"].to_numpy()
    not_api_pct = 100.0 - api_pct

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(buckets, api_pct, color=ORANGE_DARK, label="Provided over API", edgecolor=GREY_DARK, linewidth=0.2)
    ax.bar(buckets, not_api_pct, bottom=api_pct, color=GREY_LIGHT, label="Not provided over API", edgecolor=GREY_DARK, linewidth=0.2)

    ax.set_title(f"{SEGMENT}\nChart 4: {title} (conditional on eligibility, 100% stacked)")
    ax.set_ylabel("Share of eligible tours (%)")
    ax.set_xlabel("Supplier-size bucket")
    rotate_xticks(ax, 25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    grid_y(ax)
    plt.tight_layout()
    plt.show()

# Pricing API split: eligible if any pricing configured
eligible_pricing = (ds["has_individual_pricing"] == 1) | (ds["has_group_pricing"] == 1)
stacked_api_split(
    title="Prices provided over API",
    eligible_mask=eligible_pricing,
    api_col="uses_api_pricing"
)

# Scales API split: eligible if any pricing configured AND has scales (to avoid mixing no-scales)
eligible_scales = ((ds["has_individual_pricing"] == 1) | (ds["has_group_pricing"] == 1)) & (ds["has_scale_pricing"] == 1)
stacked_api_split(
    title="Pricing scales provided over API (among tours that have scales)",
    eligible_mask=eligible_scales,
    api_col="has_api_scales"
)

# =============================================================================
# CHART 5: Market outliers (within segment + size bucket)
# Question: Within a comparable cohort, which markets are lagging/leading on a chosen feature?
# Visualization: dot plot with overall mean reference line.
# =============================================================================
if DRILL_FEATURE not in feat_map:
    raise ValueError(f"DRILL_FEATURE must be one of: {list(feat_map.keys())}")
drill_col = feat_map[DRILL_FEATURE]

drill = ds[ds["supplier_size_bucket"] == DRILL_SIZE_BUCKET].copy()
g = (
    drill.groupby("location_name", dropna=False)
         .agg(tours=("tour_id", "count"), adoption=(drill_col, "mean"))
         .reset_index()
)
g = g[g["tours"] >= MIN_TOURS_PER_GROUP].copy()
g = g.sort_values("tours", ascending=False).head(TOP_N)
g["adoption_pct"] = 100.0 * g["adoption"]
g = g.sort_values("adoption_pct", ascending=True)

overall = 100.0 * float(drill[drill_col].mean()) if len(drill) else 0.0

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(g["adoption_pct"], g["location_name"].astype(str), color=ORANGE_DARK)
ax.axvline(overall, color=GREY, linestyle="--", linewidth=1, label="Cohort average")
ax.set_title(f"{SEGMENT} | {DRILL_SIZE_BUCKET}\nChart 5: {DRILL_FEATURE} adoption by market (top markets by tours)")
ax.set_xlabel("Adoption rate (%)")
ax.set_ylabel("Market (location)")
ax.legend()
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

# =============================================================================
# CHART 6: Activity type outliers (within segment + size bucket)
# =============================================================================
ga = (
    drill.groupby("activity_type", dropna=False)
         .agg(tours=("tour_id", "count"), adoption=(drill_col, "mean"))
         .reset_index()
)
ga = ga[ga["tours"] >= MIN_TOURS_PER_GROUP].copy()
ga = ga.sort_values("tours", ascending=False).head(TOP_N)
ga["adoption_pct"] = 100.0 * ga["adoption"]
ga = ga.sort_values("adoption_pct", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(ga["adoption_pct"], ga["activity_type"].astype(str), color=ORANGE)
ax.axvline(overall, color=GREY, linestyle="--", linewidth=1, label="Cohort average")
ax.set_title(f"{SEGMENT} | {DRILL_SIZE_BUCKET}\nChart 6: {DRILL_FEATURE} adoption by activity type (top activity types by tours)")
ax.set_xlabel("Adoption rate (%)")
ax.set_ylabel("Activity type")
ax.legend()
ax.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

# =============================================================================
# CHART 7: Depth distributions that actually explain supplier pain
# Question: Do bigger suppliers configure more options/add-ons (complexity)?
# Visualization: ECDF curves of total_options and num_addons by supplier-size bucket.
# =============================================================================
def plot_ecdf_by_bucket(value_col, title):
    # Choose 4 buckets (exclude "No supplier GMV..." to keep it readable)
    buckets = [
        "Supplier GMV percentile 0-25 (among GMV>0)",
        "Supplier GMV percentile 25-50 (among GMV>0)",
        "Supplier GMV percentile 50-75 (among GMV>0)",
        "Supplier GMV percentile 75-100 (among GMV>0)",
    ]
    colors = [GREY_LIGHT, ORANGE_LIGHT, ORANGE, ORANGE_DARK]

    fig, ax = plt.subplots(figsize=(10, 5))
    for b, c in zip(buckets, colors):
        d = ds[ds["supplier_size_bucket"] == b][value_col].dropna().to_numpy()
        if len(d) < 50:
            continue
        d = np.sort(d)
        y = np.arange(1, len(d) + 1) / len(d)
        ax.plot(d, y, color=c, linewidth=2, label=b)

    ax.set_title(f"{SEGMENT}\nChart 7: {title} (ECDF by supplier-size bucket)")
    ax.set_xlabel(value_col)
    ax.set_ylabel("Cumulative share of tours")
    ax.grid(alpha=0.2)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_ecdf_by_bucket("total_options", "Tour options depth")
plot_ecdf_by_bucket("num_addons", "Add-ons depth")

print("Dashboard charts complete (within-segment).")


In [0]:
# =============================================================================
# ANALYSIS 1: Feature Adoption Correlation with Supplier Success
# Question: Which pricing features are most adopted by high-performing suppliers?
# Framework: Feature Value → Adoption (attribution lens for roadmap prioritization)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# =============================================================================
# STYLE: Orange to Grey Scale Only
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_MED   = "#F39C12"
ORANGE_LIGHT = "#F8B500"
ORANGE_PALE  = "#FDC68A"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_MED     = "#95A5A6"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# Color palette for quartiles (bottom to top performance)
QUARTILE_COLORS = {
    "Q1 (Bottom 25%)": GREY_LIGHT,
    "Q2 (25-50%)": ORANGE_PALE,
    "Q3 (50-75%)": ORANGE_LIGHT,
    "Q4 (Top 25%)": ORANGE_DARK,
}

QUARTILE_ORDER = ["Q1 (Bottom 25%)", "Q2 (25-50%)", "Q3 (50-75%)", "Q4 (Top 25%)"]

# Custom colormap for heatmaps
CMAP_GREY_ORANGE = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE_MED, ORANGE_DARK],
    N=256
)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10

# =============================================================================
# 1) LOAD DATA ONCE
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()
print(f"✓ Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# =============================================================================
# 2) TYPE FIXES
# =============================================================================
NUM_COLS = ["gmv_l12mo", "bookings_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# 3) FEATURE POOL (full list as you specified)
# =============================================================================
df["Tour options configured"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 0).astype(int)
df["Pricing scales NOT provided over API"] = (pd.to_numeric(df.get("has_api_scales", 0), errors="coerce").fillna(0) == 0).astype(int)
df["Prices NOT provided over API"] = (pd.to_numeric(df.get("uses_api_pricing", 0), errors="coerce").fillna(0) == 0).astype(int)

FEATURES = [
    ("Individual pricing configured", "has_individual_pricing"),
    ("Group pricing configured", "has_group_pricing"),
    ("Add-ons configured", "has_addons"),
    ("Tour options configured", "Tour options configured"),
    ("Individual pricing has multiple price tiers", "has_individual_scales"),
    ("Group pricing has multiple price tiers", "has_group_scales"),
    ("Add-ons have multiple price tiers", "has_addon_scales"),
    ("Any pricing scales (native or over API)", "has_scale_pricing"),
    ("Pricing scales provided over API", "has_api_scales"),
    ("Pricing scales not provided over API (native)", "has_native_scales"),
    ("Pricing scales NOT provided over API", "Pricing scales NOT provided over API"),
    ("Prices provided over API", "uses_api_pricing"),
    ("Prices NOT provided over API", "Prices NOT provided over API"),
    ("Static API pricing (not live)", "has_static_api_pricing"),
    ("Live pricing and availability", "has_live_dynamic_pricing"),
]

# Ensure all columns exist
for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

FEATURE_NAMES = [n for n, _ in FEATURES]

# =============================================================================
# 4) CALCULATE SUPPLIER-LEVEL GMV (aggregate tours per supplier)
# =============================================================================
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({"gmv_l12mo": "sum"})
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

print(f"✓ Computed GMV for {len(supplier_gmv):,} suppliers")

# =============================================================================
# 5) CREATE GMV QUARTILES WITHIN EACH SEGMENT
# =============================================================================
def assign_quartile(group):
    """Assign quartile labels within a segment based on supplier GMV"""
    active = group[group["supplier_gmv_l12mo"] > 0].copy()
    if len(active) == 0:
        group["supplier_gmv_quartile"] = "No GMV"
        return group
    
    q25 = active["supplier_gmv_l12mo"].quantile(0.25)
    q50 = active["supplier_gmv_l12mo"].quantile(0.50)
    q75 = active["supplier_gmv_l12mo"].quantile(0.75)
    
    def quartile_label(gmv):
        if gmv <= 0:
            return "No GMV"
        elif gmv <= q25:
            return "Q1 (Bottom 25%)"
        elif gmv <= q50:
            return "Q2 (25-50%)"
        elif gmv <= q75:
            return "Q3 (50-75%)"
        else:
            return "Q4 (Top 25%)"
    
    group["supplier_gmv_quartile"] = group["supplier_gmv_l12mo"].apply(quartile_label)
    return group

supplier_gmv = supplier_gmv.groupby("supplier_segment", group_keys=False).apply(assign_quartile)

# Merge quartiles back to tour-level data
df = df.merge(
    supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "supplier_gmv_quartile"]],
    on="supplier_id",
    how="left"
)

df["supplier_gmv_quartile"] = df["supplier_gmv_quartile"].fillna("No GMV")

print(f"✓ Created GMV quartiles within each segment")

# =============================================================================
# 6) CALCULATE ADOPTION RATES BY QUARTILE FOR EACH SEGMENT
# =============================================================================
def calculate_adoption_by_quartile(segment_data, segment_name):
    """Calculate feature adoption rates by GMV quartile"""
    results = []
    
    for quartile in QUARTILE_ORDER + ["No GMV"]:
        q_data = segment_data[segment_data["supplier_gmv_quartile"] == quartile]
        
        if len(q_data) == 0:
            continue
        
        row = {
            "segment": segment_name,
            "quartile": quartile,
            "suppliers": q_data["supplier_id"].nunique(),
            "tours": len(q_data),
            "total_gmv": float(q_data["gmv_l12mo"].sum()),
        }
        
        for feat_name, col in FEATURES:
            adoption_rate = 100.0 * float(q_data[col].mean())
            row[feat_name] = adoption_rate
        
        results.append(row)
    
    return pd.DataFrame(results)

# Calculate for all segments
segments = df["supplier_segment"].dropna().unique()
all_adoption = []

for segment in segments:
    seg_data = df[df["supplier_segment"] == segment]
    adoption_df = calculate_adoption_by_quartile(seg_data, segment)
    all_adoption.append(adoption_df)

adoption_by_quartile = pd.concat(all_adoption, ignore_index=True)

# Order quartiles properly
adoption_by_quartile["quartile"] = pd.Categorical(
    adoption_by_quartile["quartile"],
    categories=QUARTILE_ORDER + ["No GMV"],
    ordered=True
)
adoption_by_quartile = adoption_by_quartile.sort_values(["segment", "quartile"])

print(f"✓ Calculated adoption rates across {len(segments)} segments")

# =============================================================================
# 7) IDENTIFY HIGH-CORRELATION FEATURES (biggest Q4 vs Q1 gap)
# =============================================================================
def calculate_correlation_strength(segment_data):
    """Calculate Q4 vs Q1 adoption gap for each feature"""
    q1_data = segment_data[segment_data["quartile"] == "Q1 (Bottom 25%)"]
    q4_data = segment_data[segment_data["quartile"] == "Q4 (Top 25%)"]
    
    if len(q1_data) == 0 or len(q4_data) == 0:
        return None
    
    gaps = []
    for feat_name in FEATURE_NAMES:
        q1_adoption = float(q1_data[feat_name].iloc[0]) if len(q1_data) > 0 else 0
        q4_adoption = float(q4_data[feat_name].iloc[0]) if len(q4_data) > 0 else 0
        gap = q4_adoption - q1_adoption
        
        gaps.append({
            "feature": feat_name,
            "Q1_adoption": q1_adoption,
            "Q4_adoption": q4_adoption,
            "gap": gap
        })
    
    return pd.DataFrame(gaps)

# =============================================================================
# VISUALIZATION 1: Q4 vs Q1 GAP (Waterfall/Diverging Bar)
# Shows which features have the strongest correlation with supplier success
# =============================================================================
for segment in segments[:3]:  # Show top 3 segments by GMV
    seg_adoption = adoption_by_quartile[adoption_by_quartile["segment"] == segment]
    correlation_df = calculate_correlation_strength(seg_adoption)
    
    if correlation_df is None or len(correlation_df) == 0:
        continue
    
    # Sort by gap size (largest gaps = strongest correlation with success)
    correlation_df = correlation_df.sort_values("gap", ascending=True)
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Create diverging bar chart
    y_pos = np.arange(len(correlation_df))
    colors = [ORANGE_DARK if gap > 0 else GREY_LIGHT for gap in correlation_df["gap"]]
    
    bars = ax.barh(y_pos, correlation_df["gap"], color=colors, edgecolor=GREY_DARK, linewidth=0.5)
    
    # Add value labels
    for i, (idx, row) in enumerate(correlation_df.iterrows()):
        gap_val = row["gap"]
        label = f"+{gap_val:.0f}pp" if gap_val >= 0 else f"{gap_val:.0f}pp"
        x_pos = gap_val + (2 if gap_val > 0 else -2)
        ha = "left" if gap_val > 0 else "right"
        ax.text(x_pos, i, label, va="center", ha=ha, fontsize=9, fontweight="bold", color=GREY_DARK)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(correlation_df["feature"], fontsize=10)
    ax.set_xlabel("Adoption Gap: Top 25% GMV minus Bottom 25% GMV (percentage points)", fontsize=11, fontweight="bold")
    ax.set_title(
        f"Q1: Feature Correlation with Supplier Success\n{segment} — Which features are adopted more by high-performers?",
        fontsize=13,
        fontweight="bold",
        pad=20
    )
    ax.axvline(0, color=GREY_DARK, linewidth=1.5, linestyle="-")
    ax.grid(axis="x", alpha=0.3, linestyle="--")
    
    # Add interpretation box
    top_3_features = correlation_df.nlargest(3, "gap")["feature"].tolist()
    interp_text = f"Features with largest gap (strongest success correlation):\n" + "\n".join([f"• {f}" for f in top_3_features[:3]])
    ax.text(
        0.98, 0.02, interp_text,
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment="bottom",
        horizontalalignment="right",
        bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.3, edgecolor=GREY)
    )
    
    plt.tight_layout()
    plt.show()

# =============================================================================
# VISUALIZATION 2: ADOPTION PROGRESSION (Slope Chart)
# Shows how adoption increases from Q1 → Q4 for top features
# =============================================================================
for segment in segments[:3]:
    seg_adoption = adoption_by_quartile[adoption_by_quartile["segment"] == segment]
    seg_adoption = seg_adoption[seg_adoption["quartile"].isin(QUARTILE_ORDER)]
    
    if len(seg_adoption) == 0:
        continue
    
    # Pick top 8 features by Q4 adoption rate
    q4_data = seg_adoption[seg_adoption["quartile"] == "Q4 (Top 25%)"]
    if len(q4_data) == 0:
        continue
    
    top_features = []
    for feat in FEATURE_NAMES:
        top_features.append((feat, float(q4_data[feat].iloc[0])))
    top_features = sorted(top_features, key=lambda x: x[1], reverse=True)[:8]
    selected_features = [f[0] for f in top_features]
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Plot slope lines for each feature
    for i, feat in enumerate(selected_features):
        values = []
        for q in QUARTILE_ORDER:
            q_row = seg_adoption[seg_adoption["quartile"] == q]
            if len(q_row) > 0:
                values.append(float(q_row[feat].iloc[0]))
            else:
                values.append(0)
        
        # Color gradient from grey to orange based on Q4 value
        q4_val = values[3]
        color = CMAP_GREY_ORANGE(q4_val / 100.0)
        
        x_pos = np.arange(len(QUARTILE_ORDER))
        ax.plot(x_pos, values, marker="o", linewidth=2.5, markersize=8, color=color, label=feat, alpha=0.85)
        
        # Label at end
        ax.text(
            len(QUARTILE_ORDER) - 0.5, values[-1],
            feat,
            fontsize=9,
            va="center",
            ha="left",
            color=color,
            fontweight="bold"
        )
    
    ax.set_xticks(np.arange(len(QUARTILE_ORDER)))
    ax.set_xticklabels(QUARTILE_ORDER, fontsize=11)
    ax.set_ylabel("Adoption Rate (%)", fontsize=11, fontweight="bold")
    ax.set_xlabel("Supplier Performance Quartile (by GMV within segment)", fontsize=11, fontweight="bold")
    ax.set_title(
        f"Q1: Adoption Maturity Curve by Supplier Performance\n{segment} — Top 8 features by Q4 adoption",
        fontsize=13,
        fontweight="bold",
        pad=20
    )
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    
    # Add shaded regions
    ax.axvspan(-0.5, 0.5, alpha=0.05, color=GREY)
    ax.axvspan(2.5, 3.5, alpha=0.1, color=ORANGE_DARK)
    
    plt.tight_layout()
    plt.show()

# =============================================================================
# VISUALIZATION 3: HEATMAP (Feature x Quartile)
# Comprehensive view of all features across quartiles
# =============================================================================
for segment in segments[:3]:
    seg_adoption = adoption_by_quartile[adoption_by_quartile["segment"] == segment]
    seg_adoption = seg_adoption[seg_adoption["quartile"].isin(QUARTILE_ORDER)]
    
    if len(seg_adoption) == 0:
        continue
    
    # Build matrix: features (rows) x quartiles (columns)
    matrix = []
    for feat_name in FEATURE_NAMES:
        row = []
        for q in QUARTILE_ORDER:
            q_data = seg_adoption[seg_adoption["quartile"] == q]
            if len(q_data) > 0:
                row.append(float(q_data[feat_name].iloc[0]))
            else:
                row.append(0)
        matrix.append(row)
    
    matrix = np.array(matrix)
    
    fig, ax = plt.subplots(figsize=(10, 10))
    im = ax.imshow(matrix, aspect="auto", cmap=CMAP_GREY_ORANGE, vmin=0, vmax=100)
    
    # Set ticks
    ax.set_xticks(np.arange(len(QUARTILE_ORDER)))
    ax.set_xticklabels(QUARTILE_ORDER, fontsize=10, rotation=20, ha="right")
    
    ax.set_yticks(np.arange(len(FEATURE_NAMES)))
    ax.set_yticklabels(FEATURE_NAMES, fontsize=9)
    
    ax.set_xlabel("Supplier Performance Quartile (by GMV within segment)", fontsize=11, fontweight="bold")
    ax.set_title(
        f"Q1: Feature Adoption Heatmap by Supplier Performance\n{segment} — All features across quartiles",
        fontsize=13,
        fontweight="bold",
        pad=20
    )
    
    # Annotate cells
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            text_color = "white" if val > 60 else GREY_DARK
            ax.text(j, i, f"{val:.0f}", ha="center", va="center", color=text_color, fontsize=8, fontweight="bold")
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Adoption Rate (%)", fontsize=10, fontweight="bold")
    
    plt.tight_layout()
    plt.show()

# =============================================================================
# VISUALIZATION 4: TOP PERFORMER FEATURE PROFILE (Radial/Spider Chart)
# Shows the adoption profile of top-performing suppliers
# =============================================================================
for segment in segments[:3]:
    seg_adoption = adoption_by_quartile[adoption_by_quartile["segment"] == segment]
    
    q1_data = seg_adoption[seg_adoption["quartile"] == "Q1 (Bottom 25%)"]
    q4_data = seg_adoption[seg_adoption["quartile"] == "Q4 (Top 25%)"]
    
    if len(q1_data) == 0 or len(q4_data) == 0:
        continue
    
    # Select key features for radar chart (not all 15)
    key_features = [
        "Individual pricing configured",
        "Group pricing configured",
        "Add-ons configured",
        "Any pricing scales (native or over API)",
        "Prices provided over API",
        "Live pricing and availability",
        "Tour options configured",
        "Individual pricing has multiple price tiers"
    ]
    
    q1_values = [float(q1_data[f].iloc[0]) for f in key_features]
    q4_values = [float(q4_data[f].iloc[0]) for f in key_features]
    
    # Radar chart
    angles = np.linspace(0, 2 * np.pi, len(key_features), endpoint=False).tolist()
    q1_values += q1_values[:1]
    q4_values += q4_values[:1]
    angles += angles[:1]
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection="polar"))
    
    ax.plot(angles, q4_values, "o-", linewidth=2.5, color=ORANGE_DARK, label="Q4 (Top 25%)")
    ax.fill(angles, q4_values, alpha=0.15, color=ORANGE_DARK)
    
    ax.plot(angles, q1_values, "o-", linewidth=2.5, color=GREY, label="Q1 (Bottom 25%)")
    ax.fill(angles, q1_values, alpha=0.1, color=GREY)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(key_features, fontsize=9)
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(["20%", "40%", "60%", "80%", "100%"], fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.5)
    
    ax.set_title(
        f"Q1: Top Performer Feature Profile\n{segment} — Adoption comparison Q4 vs Q1",
        fontsize=13,
        fontweight="bold",
        pad=30
    )
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=10)
    
    plt.tight_layout()
    plt.show()

# =============================================================================
# SUMMARY TABLE: Top Correlation Features by Segment
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Features with Strongest Correlation to Supplier Success (Top 5 per segment)")
print("="*100)

for segment in segments:
    seg_adoption = adoption_by_quartile[adoption_by_quartile["segment"] == segment]
    correlation_df = calculate_correlation_strength(seg_adoption)
    
    if correlation_df is None:
        continue
    
    top_5 = correlation_df.nlargest(5, "gap")
    
    print(f"\n{segment}:")
    print(f"{'Feature':<50} {'Q1 Adoption':<15} {'Q4 Adoption':<15} {'Gap (pp)':<10}")
    print("-" * 95)
    for _, row in top_5.iterrows():
        print(f"{row['feature']:<50} {row['Q1_adoption']:>12.1f}% {row['Q4_adoption']:>12.1f}% {row['gap']:>8.1f}pp")

print("\n✓ Analysis complete.")


In [0]:
# =============================================================================
# ANALYSIS 1: Feature Adoption by Supplier Performance Quartile
# Question: Which pricing features are most adopted by high-performing suppliers?
# Approach: Show ALL adoption rates across ALL quartiles for ALL features within each segment
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

CMAP = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()
print(f"Loaded {len(df):,} tours from {df['supplier_id'].nunique():,} suppliers")

# =============================================================================
# TYPE FIXES
# =============================================================================
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# FEATURES
# =============================================================================
df["Tour options configured"] = (df.get("total_options", 0) > 0).astype(int)
df["Pricing scales NOT provided over API"] = (df.get("has_api_scales", 0) == 0).astype(int)
df["Prices NOT provided over API"] = (df.get("uses_api_pricing", 0) == 0).astype(int)

FEATURES = [
    ("Individual pricing configured", "has_individual_pricing"),
    ("Group pricing configured", "has_group_pricing"),
    ("Add-ons configured", "has_addons"),
    ("Tour options configured", "Tour options configured"),
    ("Individual pricing has multiple price tiers", "has_individual_scales"),
    ("Group pricing has multiple price tiers", "has_group_scales"),
    ("Add-ons have multiple price tiers", "has_addon_scales"),
    ("Any pricing scales (native or over API)", "has_scale_pricing"),
    ("Pricing scales provided over API", "has_api_scales"),
    ("Pricing scales not provided over API (native)", "has_native_scales"),
    ("Pricing scales NOT provided over API", "Pricing scales NOT provided over API"),
    ("Prices provided over API", "uses_api_pricing"),
    ("Prices NOT provided over API", "Prices NOT provided over API"),
    ("Static API pricing (not live)", "has_static_api_pricing"),
    ("Live pricing and availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

FEATURE_NAMES = [n for n, _ in FEATURES]

# =============================================================================
# SUPPLIER GMV (aggregate from tours)
# =============================================================================
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

# =============================================================================
# QUARTILES WITHIN EACH SEGMENT
# =============================================================================
def assign_quartile_within_segment(group):
    active = group[group["supplier_gmv_l12mo"] > 0].copy()
    if len(active) == 0:
        group["quartile"] = "No GMV"
        return group
    
    q25 = active["supplier_gmv_l12mo"].quantile(0.25)
    q50 = active["supplier_gmv_l12mo"].quantile(0.50)
    q75 = active["supplier_gmv_l12mo"].quantile(0.75)
    
    def get_q(x):
        if x <= 0: return "No GMV"
        if x <= q25: return "Q1 (Bottom 25%)"
        if x <= q50: return "Q2 (25-50%)"
        if x <= q75: return "Q3 (50-75%)"
        return "Q4 (Top 25%)"
    
    group["quartile"] = group["supplier_gmv_l12mo"].apply(get_q)
    return group

supplier_gmv = supplier_gmv.groupby("supplier_segment", group_keys=False).apply(assign_quartile_within_segment)

df = df.merge(supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "quartile"]], on="supplier_id", how="left")
df["quartile"] = df["quartile"].fillna("No GMV")

QUARTILE_ORDER = ["Q1 (Bottom 25%)", "Q2 (25-50%)", "Q3 (50-75%)", "Q4 (Top 25%)"]

print("Quartiles created within each segment")

# =============================================================================
# CALCULATE ADOPTION BY QUARTILE FOR EACH SEGMENT
# =============================================================================
segments = sorted(df["supplier_segment"].dropna().unique())

adoption_data = []
for segment in segments:
    seg_df = df[df["supplier_segment"] == segment]
    
    for q in QUARTILE_ORDER:
        q_df = seg_df[seg_df["quartile"] == q]
        if len(q_df) == 0:
            continue
        
        row = {
            "segment": segment,
            "quartile": q,
            "tours": len(q_df),
            "suppliers": q_df["supplier_id"].nunique(),
            "total_gmv": q_df["gmv_l12mo"].sum(),
        }
        
        for feat_name, col in FEATURES:
            row[feat_name] = 100.0 * q_df[col].mean()
        
        adoption_data.append(row)

adoption_df = pd.DataFrame(adoption_data)

# =============================================================================
# CHART 1: COMPREHENSIVE HEATMAP (Features x Quartiles) PER SEGMENT
# Shows ALL data: every feature, every quartile, every segment
# This is the exploratory, descriptive view you asked for
# =============================================================================

for segment in segments:
    seg_data = adoption_df[adoption_df["segment"] == segment].copy()
    
    if len(seg_data) == 0:
        continue
    
    # Build matrix: features (rows) x quartiles (columns)
    matrix = []
    tour_counts = []
    
    for feat_name in FEATURE_NAMES:
        row = []
        for q in QUARTILE_ORDER:
            q_data = seg_data[seg_data["quartile"] == q]
            if len(q_data) > 0:
                row.append(q_data[feat_name].iloc[0])
            else:
                row.append(0)
        matrix.append(row)
    
    # Get tour counts for each quartile for the header
    for q in QUARTILE_ORDER:
        q_data = seg_data[seg_data["quartile"] == q]
        if len(q_data) > 0:
            tour_counts.append(int(q_data["tours"].iloc[0]))
        else:
            tour_counts.append(0)
    
    matrix = np.array(matrix)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 12))
    
    # Heatmap
    im = ax.imshow(matrix, aspect="auto", cmap=CMAP, vmin=0, vmax=100)
    
    # Axis labels
    quartile_labels = [f"{q}\n({tour_counts[i]:,} tours)" for i, q in enumerate(QUARTILE_ORDER)]
    ax.set_xticks(np.arange(len(QUARTILE_ORDER)))
    ax.set_xticklabels(quartile_labels, fontsize=11, fontweight="bold")
    
    ax.set_yticks(np.arange(len(FEATURE_NAMES)))
    ax.set_yticklabels(FEATURE_NAMES, fontsize=10)
    
    # Title
    total_suppliers = seg_data["suppliers"].iloc[0] if len(seg_data) > 0 else 0
    total_gmv = seg_data["total_gmv"].sum()
    
    ax.set_title(
        f"Feature Adoption by Supplier Performance Quartile\n"
        f"Segment: {segment} | {len(seg_data)} quartiles | {total_suppliers:,} suppliers | €{total_gmv:,.0f} GMV L12M\n"
        f"Question: Which features are most adopted by high-performing suppliers?",
        fontsize=14,
        fontweight="bold",
        pad=20
    )
    
    # Annotate each cell with adoption percentage
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            text_color = "white" if val > 55 else GREY_DARK
            ax.text(j, i, f"{val:.0f}%", 
                   ha="center", va="center", 
                   color=text_color, 
                   fontsize=9, 
                   fontweight="bold")
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Adoption Rate (%)", fontsize=11, fontweight="bold")
    cbar.ax.tick_params(labelsize=10)
    
    # Add gridlines for readability
    ax.set_xticks(np.arange(len(QUARTILE_ORDER)) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(FEATURE_NAMES)) - 0.5, minor=True)
    ax.grid(which="minor", color=GREY_PALE, linestyle='-', linewidth=1)
    ax.tick_params(which="minor", size=0)
    
    plt.tight_layout()
    plt.show()

# =============================================================================
# CHART 2: GROUPED BAR CHART (Key Features Only, All Quartiles)
# For each key feature, show bars for Q1, Q2, Q3, Q4 side-by-side
# This makes quartile comparison easier than heatmap
# =============================================================================

# Select key features (the most important ones for decision-making)
KEY_FEATURES = [
    "Individual pricing configured",
    "Group pricing configured",
    "Add-ons configured",
    "Tour options configured",
    "Any pricing scales (native or over API)",
    "Prices provided over API",
    "Live pricing and availability",
    "Individual pricing has multiple price tiers",
    "Group pricing has multiple price tiers",
]

for segment in segments:
    seg_data = adoption_df[adoption_df["segment"] == segment].copy()
    
    if len(seg_data) == 0:
        continue
    
    # Prepare data for grouped bars
    n_features = len(KEY_FEATURES)
    n_quartiles = len(QUARTILE_ORDER)
    
    fig, ax = plt.subplots(figsize=(16, 10))
    
    x = np.arange(n_features)
    width = 0.2
    
    colors = [GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE_DARK]
    
    for i, q in enumerate(QUARTILE_ORDER):
        values = []
        for feat in KEY_FEATURES:
            q_data = seg_data[seg_data["quartile"] == q]
            if len(q_data) > 0:
                values.append(q_data[feat].iloc[0])
            else:
                values.append(0)
        
        offset = (i - n_quartiles/2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=q, color=colors[i], edgecolor=GREY_DARK, linewidth=0.5)
        
        # Add value labels on bars
        for j, bar in enumerate(bars):
            height = bar.get_height()
            if height > 5:  # Only label if bar is visible
                ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                       f'{height:.0f}%',
                       ha='center', va='bottom', fontsize=7, color=GREY_DARK)
    
    ax.set_ylabel("Adoption Rate (%)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Pricing Feature", fontsize=12, fontweight="bold")
    ax.set_title(
        f"Feature Adoption Across Supplier Performance Quartiles\n"
        f"Segment: {segment}\n"
        f"Question: Which features show the strongest correlation with supplier success?",
        fontsize=14,
        fontweight="bold",
        pad=20
    )
    ax.set_xticks(x)
    ax.set_xticklabels(KEY_FEATURES, rotation=45, ha='right', fontsize=10)
    ax.legend(title="Supplier Performance\n(by GMV within segment)", 
             fontsize=10, 
             title_fontsize=11,
             loc='upper left',
             framealpha=0.95)
    ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.show()

print("\n✓ Analysis complete: 2 charts per segment showing ALL adoption data across quartiles")


In [0]:
# =============================================================================
# BETTER ANALYSIS: What Do High-Performing Suppliers Actually Use?
# Focus: Top Quartile (Q4) Adoption Rates Across Segments
# Question: Which pricing features are most adopted by high-performing suppliers?
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Create derived features
df["Tour options configured"] = (df.get("total_options", 0) > 0).astype(int)
df["Pricing scales NOT provided over API"] = (df.get("has_api_scales", 0) == 0).astype(int)
df["Prices NOT provided over API"] = (df.get("uses_api_pricing", 0) == 0).astype(int)

# =============================================================================
# FEATURES - Focus on Most Important Only
# =============================================================================
FEATURES = [
    ("Individual pricing", "has_individual_pricing"),
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Tour options", "Tour options configured"),
    ("Individual price tiers", "has_individual_scales"),
    ("Group price tiers", "has_group_scales"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# CALCULATE SUPPLIER GMV AND QUARTILES
# =============================================================================
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

def assign_quartile(group):
    active = group[group["supplier_gmv_l12mo"] > 0].copy()
    if len(active) == 0:
        group["quartile"] = "No GMV"
        return group
    
    q25 = active["supplier_gmv_l12mo"].quantile(0.25)
    q50 = active["supplier_gmv_l12mo"].quantile(0.50)
    q75 = active["supplier_gmv_l12mo"].quantile(0.75)
    
    def get_q(x):
        if x <= 0: return "No GMV"
        if x <= q25: return "Q1"
        if x <= q50: return "Q2"
        if x <= q75: return "Q3"
        return "Q4"
    
    group["quartile"] = group["supplier_gmv_l12mo"].apply(get_q)
    return group

supplier_gmv = supplier_gmv.groupby("supplier_segment", group_keys=False).apply(assign_quartile)
df = df.merge(supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "quartile"]], on="supplier_id", how="left")
df["quartile"] = df["quartile"].fillna("No GMV")

# =============================================================================
# FOCUS: Q4 (TOP PERFORMERS) ADOPTION ONLY
# =============================================================================
q4_data = df[df["quartile"] == "Q4"].copy()

segments = sorted([s for s in df["supplier_segment"].dropna().unique() 
                   if df[(df["supplier_segment"] == s) & (df["quartile"] == "Q4")].shape[0] > 0])

q4_adoption = []
for segment in segments:
    seg_q4 = q4_data[q4_data["supplier_segment"] == segment]
    
    if len(seg_q4) == 0:
        continue
    
    row = {
        "segment": segment,
        "suppliers": seg_q4["supplier_id"].nunique(),
        "tours": len(seg_q4),
        "total_gmv": seg_q4["gmv_l12mo"].sum(),
    }
    
    for feat_name, col in FEATURES:
        row[feat_name] = 100.0 * seg_q4[col].mean()
    
    q4_adoption.append(row)

q4_df = pd.DataFrame(q4_adoption)

# Sort segments by GMV (largest first)
q4_df = q4_df.sort_values("total_gmv", ascending=False)

print("\n" + "="*100)
print("HIGH-PERFORMING SUPPLIER FEATURE ADOPTION (Top 25% by GMV within each segment)")
print("="*100)
print(q4_df[["segment", "suppliers", "tours", "total_gmv"]].to_string(index=False))
print("="*100)

# =============================================================================
# CHART 1: WHAT DO TOP PERFORMERS USE?
# Heatmap showing Q4 adoption rates across all segments
# =============================================================================

fig, ax = plt.subplots(figsize=(16, 10))

# Build matrix: segments (rows) x features (columns)
feature_names = [f[0] for f in FEATURES]
segment_list = q4_df["segment"].tolist()

matrix = []
for segment in segment_list:
    row = []
    seg_data = q4_df[q4_df["segment"] == segment]
    for feat_name, _ in FEATURES:
        row.append(seg_data[feat_name].iloc[0])
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Features
ax.set_xticks(np.arange(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=11, fontweight="bold")

# Y-axis: Segments with context
segment_labels = []
for segment in segment_list:
    seg_data = q4_df[q4_df["segment"] == segment]
    suppliers = int(seg_data["suppliers"].iloc[0])
    gmv_m = seg_data["total_gmv"].iloc[0] / 1_000_000
    segment_labels.append(f"{segment}\n({suppliers} suppliers | €{gmv_m:.1f}M)")

ax.set_yticks(np.arange(len(segment_list)))
ax.set_yticklabels(segment_labels, fontsize=11)

# Title
ax.set_title(
    "What Do High-Performing Suppliers Actually Use?\n"
    "Feature Adoption Rates for Top 25% GMV Suppliers (Q4) by Segment\n"
    "Question: Which pricing features are most adopted by successful suppliers?",
    fontsize=15,
    fontweight="bold",
    pad=25
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=10, 
               fontweight="bold")

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate Among Top Performers (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segment_list)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.savefig("/mnt/data/q4_adoption_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n✓ Chart 1: Q4 Adoption Heatmap saved")

# =============================================================================
# CHART 2: FEATURE RANKING BY AVERAGE Q4 ADOPTION
# Which features are most commonly adopted by top performers overall?
# =============================================================================

# Calculate average adoption across all Q4 suppliers (weighted by GMV)
total_q4_gmv = q4_df["total_gmv"].sum()
feature_rankings = []

for feat_name, _ in FEATURES:
    weighted_avg = 0
    for _, row in q4_df.iterrows():
        weight = row["total_gmv"] / total_q4_gmv
        weighted_avg += row[feat_name] * weight
    
    # Also get range (min to max across segments)
    min_val = q4_df[feat_name].min()
    max_val = q4_df[feat_name].max()
    
    feature_rankings.append({
        "feature": feat_name,
        "avg_adoption": weighted_avg,
        "min": min_val,
        "max": max_val,
        "range": max_val - min_val
    })

ranking_df = pd.DataFrame(feature_rankings).sort_values("avg_adoption", ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(len(ranking_df))

# Main bars (average adoption)
bars = ax.barh(y_pos, ranking_df["avg_adoption"], 
               color=ORANGE, edgecolor=GREY_DARK, linewidth=1.5, alpha=0.9)

# Add range indicators (min-max across segments)
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], 
            color=GREY_DARK, linewidth=3, alpha=0.6, zorder=1)
    ax.scatter([row["min"], row["max"]], [i, i], 
               color=GREY_DARK, s=50, zorder=2, alpha=0.8)

# Value labels
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.text(row["avg_adoption"] + 2, i, f"{row['avg_adoption']:.0f}%", 
           va="center", ha="left", fontsize=10, fontweight="bold", color=GREY_DARK)

ax.set_yticks(y_pos)
ax.set_yticklabels(ranking_df["feature"], fontsize=11)
ax.set_xlabel("Adoption Rate Among Top Performers (%)", fontsize=12, fontweight="bold")
ax.set_title(
    "Feature Adoption Ranking: What Top Performers Use Most\n"
    "GMV-weighted average across all segments (Q4 suppliers only)\n"
    "Bar = weighted average | Line = range across segments",
    fontsize=14,
    fontweight="bold",
    pad=20
)
ax.set_xlim(0, 110)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Add summary box
top_3 = ranking_df.nlargest(3, "avg_adoption")["feature"].tolist()
summary = "Most Adopted by Top Performers:\n" + "\n".join([f"{i+1}. {f}" for i, f in enumerate(top_3)])
ax.text(
    0.98, 0.02, summary,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.4, edgecolor=ORANGE_DARK, linewidth=2)
)

plt.tight_layout()
plt.savefig("/mnt/data/q4_feature_ranking.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n✓ Chart 2: Q4 Feature Ranking saved")

# =============================================================================
# SUMMARY TABLE: TOP 5 FEATURES BY SEGMENT
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Top 5 Features Adopted by High-Performing Suppliers (Q4) in Each Segment")
print("="*100)

for segment in segment_list:
    seg_data = q4_df[q4_df["segment"] == segment].iloc[0]
    
    feature_adoptions = []
    for feat_name, _ in FEATURES:
        feature_adoptions.append((feat_name, seg_data[feat_name]))
    
    feature_adoptions = sorted(feature_adoptions, key=lambda x: x[1], reverse=True)
    
    print(f"\n{segment} ({int(seg_data['suppliers'])} suppliers | €{seg_data['total_gmv']/1e6:.1f}M GMV):")
    print(f"{'Feature':<40} {'Adoption':<15}")
    print("-" * 55)
    for feat, adoption in feature_adoptions[:5]:
        print(f"{feat:<40} {adoption:>12.0f}%")

# =============================================================================
# ACTIONABLE INSIGHTS SUMMARY
# =============================================================================
print("\n" + "="*100)
print("ACTIONABLE INSIGHTS")
print("="*100)

# Find universal features (>80% across all top performers)
universal_features = []
for feat_name, _ in FEATURES:
    if q4_df[feat_name].min() > 80:
        avg = q4_df[feat_name].mean()
        universal_features.append((feat_name, avg))

if universal_features:
    print("\n1. UNIVERSAL FEATURES (>80% adoption across ALL top-performing segments):")
    for feat, avg in sorted(universal_features, key=lambda x: x[1], reverse=True):
        print(f"   • {feat}: {avg:.0f}% average")

# Find differentiating features (high variance across segments)
differentiating = []
for feat_name, _ in FEATURES:
    variance = q4_df[feat_name].std()
    if variance > 20:  # More than 20pp standard deviation
        differentiating.append((feat_name, variance, q4_df[feat_name].min(), q4_df[feat_name].max()))

if differentiating:
    print("\n2. SEGMENT-SPECIFIC FEATURES (high variance = different value by segment):")
    for feat, std, min_v, max_v in sorted(differentiating, key=lambda x: x[1], reverse=True):
        print(f"   • {feat}: {min_v:.0f}% to {max_v:.0f}% (σ={std:.0f}pp)")

# Find emerging opportunities (features with high max but low min)
opportunities = []
for feat_name, _ in FEATURES:
    min_val = q4_df[feat_name].min()
    max_val = q4_df[feat_name].max()
    if max_val > 60 and min_val < 30 and (max_val - min_val) > 40:
        opportunities.append((feat_name, min_val, max_val))

if opportunities:
    print("\n3. GROWTH OPPORTUNITIES (high adoption in some Q4 segments, low in others):")
    for feat, min_v, max_v in sorted(opportunities, key=lambda x: x[2] - x[1], reverse=True):
        print(f"   • {feat}: ranges from {min_v:.0f}% to {max_v:.0f}% adoption")
        print(f"     → Opportunity to expand to low-adoption segments")

print("\n" + "="*100)
print("✓ Analysis complete: Focus on what top performers actually use")
print("="*100)


In [0]:
# =============================================================================
# BETTER ANALYSIS: What Do High-Performing Suppliers Actually Use?
# Focus: Top Quartile (Q4) Adoption Rates Across Segments
# Question: Which pricing features are most adopted by high-performing suppliers?
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Create derived features
df["Tour options configured"] = (df.get("total_options", 0) > 0).astype(int)
df["Pricing scales NOT provided over API"] = (df.get("has_api_scales", 0) == 0).astype(int)
df["Prices NOT provided over API"] = (df.get("uses_api_pricing", 0) == 0).astype(int)

# =============================================================================
# FEATURES - Focus on Most Important Only
# =============================================================================
FEATURES = [
    ("Individual pricing", "has_individual_pricing"),
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Tour options", "Tour options configured"),
    ("Individual price tiers", "has_individual_scales"),
    ("Group price tiers", "has_group_scales"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# CALCULATE SUPPLIER GMV AND QUARTILES
# =============================================================================
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

def assign_quartile(group):
    active = group[group["supplier_gmv_l12mo"] > 0].copy()
    if len(active) == 0:
        group["quartile"] = "No GMV"
        return group
    
    q25 = active["supplier_gmv_l12mo"].quantile(0.25)
    q50 = active["supplier_gmv_l12mo"].quantile(0.50)
    q75 = active["supplier_gmv_l12mo"].quantile(0.75)
    
    def get_q(x):
        if x <= 0: return "No GMV"
        if x <= q25: return "Q1"
        if x <= q50: return "Q2"
        if x <= q75: return "Q3"
        return "Q4"
    
    group["quartile"] = group["supplier_gmv_l12mo"].apply(get_q)
    return group

supplier_gmv = supplier_gmv.groupby("supplier_segment", group_keys=False).apply(assign_quartile)
df = df.merge(supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "quartile"]], on="supplier_id", how="left")
df["quartile"] = df["quartile"].fillna("No GMV")

# =============================================================================
# FOCUS: Q4 (TOP PERFORMERS) ADOPTION ONLY
# =============================================================================
q4_data = df[df["quartile"] == "Q4"].copy()

segments = sorted([s for s in df["supplier_segment"].dropna().unique() 
                   if df[(df["supplier_segment"] == s) & (df["quartile"] == "Q4")].shape[0] > 0])

q4_adoption = []
for segment in segments:
    seg_q4 = q4_data[q4_data["supplier_segment"] == segment]
    
    if len(seg_q4) == 0:
        continue
    
    row = {
        "segment": segment,
        "suppliers": seg_q4["supplier_id"].nunique(),
        "tours": len(seg_q4),
        "total_gmv": seg_q4["gmv_l12mo"].sum(),
    }
    
    for feat_name, col in FEATURES:
        row[feat_name] = 100.0 * seg_q4[col].mean()
    
    q4_adoption.append(row)

q4_df = pd.DataFrame(q4_adoption)

# Sort segments by GMV (largest first)
q4_df = q4_df.sort_values("total_gmv", ascending=False)

print("\n" + "="*100)
print("HIGH-PERFORMING SUPPLIER FEATURE ADOPTION (Top 25% by GMV within each segment)")
print("="*100)
print(q4_df[["segment", "suppliers", "tours", "total_gmv"]].to_string(index=False))
print("="*100)

# =============================================================================
# CHART 1: WHAT DO TOP PERFORMERS USE?
# Heatmap showing Q4 adoption rates across all segments
# =============================================================================

fig, ax = plt.subplots(figsize=(16, 10))

# Build matrix: segments (rows) x features (columns)
feature_names = [f[0] for f in FEATURES]
segment_list = q4_df["segment"].tolist()

matrix = []
for segment in segment_list:
    row = []
    seg_data = q4_df[q4_df["segment"] == segment]
    for feat_name, _ in FEATURES:
        row.append(seg_data[feat_name].iloc[0])
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Features
ax.set_xticks(np.arange(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=11, fontweight="bold")

# Y-axis: Segments with context
segment_labels = []
for segment in segment_list:
    seg_data = q4_df[q4_df["segment"] == segment]
    suppliers = int(seg_data["suppliers"].iloc[0])
    gmv_m = seg_data["total_gmv"].iloc[0] / 1_000_000
    segment_labels.append(f"{segment}\n({suppliers} suppliers | €{gmv_m:.1f}M)")

ax.set_yticks(np.arange(len(segment_list)))
ax.set_yticklabels(segment_labels, fontsize=11)

# Title
ax.set_title(
    "What Do High-Performing Suppliers Actually Use?\n"
    "Feature Adoption Rates for Top 25% GMV Suppliers (Q4) by Segment\n"
    "Question: Which pricing features are most adopted by successful suppliers?",
    fontsize=15,
    fontweight="bold",
    pad=25
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=10, 
               fontweight="bold")

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate Among Top Performers (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segment_list)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Q4 Adoption Heatmap")

# =============================================================================
# CHART 2: FEATURE RANKING BY AVERAGE Q4 ADOPTION
# Which features are most commonly adopted by top performers overall?
# =============================================================================

# Calculate average adoption across all Q4 suppliers (weighted by GMV)
total_q4_gmv = q4_df["total_gmv"].sum()
feature_rankings = []

for feat_name, _ in FEATURES:
    weighted_avg = 0
    for _, row in q4_df.iterrows():
        weight = row["total_gmv"] / total_q4_gmv
        weighted_avg += row[feat_name] * weight
    
    # Also get range (min to max across segments)
    min_val = q4_df[feat_name].min()
    max_val = q4_df[feat_name].max()
    
    feature_rankings.append({
        "feature": feat_name,
        "avg_adoption": weighted_avg,
        "min": min_val,
        "max": max_val,
        "range": max_val - min_val
    })

ranking_df = pd.DataFrame(feature_rankings).sort_values("avg_adoption", ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(len(ranking_df))

# Main bars (average adoption)
bars = ax.barh(y_pos, ranking_df["avg_adoption"], 
               color=ORANGE, edgecolor=GREY_DARK, linewidth=1.5, alpha=0.9)

# Add range indicators (min-max across segments)
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], 
            color=GREY_DARK, linewidth=3, alpha=0.6, zorder=1)
    ax.scatter([row["min"], row["max"]], [i, i], 
               color=GREY_DARK, s=50, zorder=2, alpha=0.8)

# Value labels
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.text(row["avg_adoption"] + 2, i, f"{row['avg_adoption']:.0f}%", 
           va="center", ha="left", fontsize=10, fontweight="bold", color=GREY_DARK)

ax.set_yticks(y_pos)
ax.set_yticklabels(ranking_df["feature"], fontsize=11)
ax.set_xlabel("Adoption Rate Among Top Performers (%)", fontsize=12, fontweight="bold")
ax.set_title(
    "Feature Adoption Ranking: What Top Performers Use Most\n"
    "GMV-weighted average across all segments (Q4 suppliers only)\n"
    "Bar = weighted average | Line = range across segments",
    fontsize=14,
    fontweight="bold",
    pad=20
)
ax.set_xlim(0, 110)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Add summary box
top_3 = ranking_df.nlargest(3, "avg_adoption")["feature"].tolist()
summary = "Most Adopted by Top Performers:\n" + "\n".join([f"{i+1}. {f}" for i, f in enumerate(top_3)])
ax.text(
    0.98, 0.02, summary,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.4, edgecolor=ORANGE_DARK, linewidth=2)
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Q4 Feature Ranking")

# =============================================================================
# SUMMARY TABLE: TOP 5 FEATURES BY SEGMENT
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Top 5 Features Adopted by High-Performing Suppliers (Q4) in Each Segment")
print("="*100)

for segment in segment_list:
    seg_data = q4_df[q4_df["segment"] == segment].iloc[0]
    
    feature_adoptions = []
    for feat_name, _ in FEATURES:
        feature_adoptions.append((feat_name, seg_data[feat_name]))
    
    feature_adoptions = sorted(feature_adoptions, key=lambda x: x[1], reverse=True)
    
    print(f"\n{segment} ({int(seg_data['suppliers'])} suppliers | €{seg_data['total_gmv']/1e6:.1f}M GMV):")
    print(f"{'Feature':<40} {'Adoption':<15}")
    print("-" * 55)
    for feat, adoption in feature_adoptions[:5]:
        print(f"{feat:<40} {adoption:>12.0f}%")

# =============================================================================
# ACTIONABLE INSIGHTS SUMMARY
# =============================================================================
print("\n" + "="*100)
print("ACTIONABLE INSIGHTS")
print("="*100)

# Find universal features (>80% across all top performers)
universal_features = []
for feat_name, _ in FEATURES:
    if q4_df[feat_name].min() > 80:
        avg = q4_df[feat_name].mean()
        universal_features.append((feat_name, avg))

if universal_features:
    print("\n1. UNIVERSAL FEATURES (>80% adoption across ALL top-performing segments):")
    for feat, avg in sorted(universal_features, key=lambda x: x[1], reverse=True):
        print(f"   • {feat}: {avg:.0f}% average")

# Find differentiating features (high variance across segments)
differentiating = []
for feat_name, _ in FEATURES:
    variance = q4_df[feat_name].std()
    if variance > 20:  # More than 20pp standard deviation
        differentiating.append((feat_name, variance, q4_df[feat_name].min(), q4_df[feat_name].max()))

if differentiating:
    print("\n2. SEGMENT-SPECIFIC FEATURES (high variance = different value by segment):")
    for feat, std, min_v, max_v in sorted(differentiating, key=lambda x: x[1], reverse=True):
        print(f"   • {feat}: {min_v:.0f}% to {max_v:.0f}% (σ={std:.0f}pp)")

# Find emerging opportunities (features with high max but low min)
opportunities = []
for feat_name, _ in FEATURES:
    min_val = q4_df[feat_name].min()
    max_val = q4_df[feat_name].max()
    if max_val > 60 and min_val < 30 and (max_val - min_val) > 40:
        opportunities.append((feat_name, min_val, max_val))

if opportunities:
    print("\n3. GROWTH OPPORTUNITIES (high adoption in some Q4 segments, low in others):")
    for feat, min_v, max_v in sorted(opportunities, key=lambda x: x[2] - x[1], reverse=True):
        print(f"   • {feat}: ranges from {min_v:.0f}% to {max_v:.0f}% adoption")
        print(f"     → Opportunity to expand to low-adoption segments")

print("\n" + "="*100)
print("✓ Analysis complete: Focus on what top performers actually use")
print("="*100)


In [0]:
# =============================================================================
# IMPROVED ANALYSIS with all three enhancements
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Create derived features
df["Tour options configured"] = (df.get("total_options", 0) > 0).astype(int)
df["Pricing scales NOT provided over API"] = (df.get("has_api_scales", 0) == 0).astype(int)
df["Prices NOT provided over API"] = (df.get("uses_api_pricing", 0) == 0).astype(int)

# =============================================================================
# FEATURES
# =============================================================================
FEATURES = [
    ("Individual pricing", "has_individual_pricing"),
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Tour options", "Tour options configured"),
    ("Individual price tiers", "has_individual_scales"),
    ("Group price tiers", "has_group_scales"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# CALCULATE SUPPLIER GMV AND QUARTILES
# =============================================================================
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

def assign_quartile(group):
    active = group[group["supplier_gmv_l12mo"] > 0].copy()
    if len(active) == 0:
        group["quartile"] = "No GMV"
        return group
    
    q25 = active["supplier_gmv_l12mo"].quantile(0.25)
    q50 = active["supplier_gmv_l12mo"].quantile(0.50)
    q75 = active["supplier_gmv_l12mo"].quantile(0.75)
    
    def get_q(x):
        if x <= 0: return "No GMV"
        if x <= q25: return "Q1"
        if x <= q50: return "Q2"
        if x <= q75: return "Q3"
        return "Q4"
    
    group["quartile"] = group["supplier_gmv_l12mo"].apply(get_q)
    return group

supplier_gmv = supplier_gmv.groupby("supplier_segment", group_keys=False).apply(assign_quartile)
df = df.merge(supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "quartile"]], on="supplier_id", how="left")
df["quartile"] = df["quartile"].fillna("No GMV")

# =============================================================================
# FOCUS: Q4 (TOP PERFORMERS) ADOPTION ONLY
# =============================================================================
q4_data = df[df["quartile"] == "Q4"].copy()

segments = sorted([s for s in df["supplier_segment"].dropna().unique() 
                   if df[(df["supplier_segment"] == s) & (df["quartile"] == "Q4")].shape[0] > 0])

q4_adoption = []
for segment in segments:
    seg_q4 = q4_data[q4_data["supplier_segment"] == segment]
    
    if len(seg_q4) == 0:
        continue
    
    row = {
        "segment": segment,
        "suppliers": seg_q4["supplier_id"].nunique(),
        "tours": len(seg_q4),
        "total_gmv": seg_q4["gmv_l12mo"].sum(),
    }
    
    for feat_name, col in FEATURES:
        row[feat_name] = 100.0 * seg_q4[col].mean()
    
    q4_adoption.append(row)

q4_df = pd.DataFrame(q4_adoption)

# =============================================================================
# IMPROVEMENT 1: Sort segments by total GMV (descending) for Chart 1
# =============================================================================
q4_df = q4_df.sort_values("total_gmv", ascending=False)

# =============================================================================
# CHART 1: WHAT DO TOP PERFORMERS USE? (IMPROVED)
# =============================================================================

fig, ax = plt.subplots(figsize=(16, 10))

# Build matrix: segments (rows) x features (columns)
feature_names = [f[0] for f in FEATURES]
segment_list = q4_df["segment"].tolist()

matrix = []
for segment in segment_list:
    row = []
    seg_data = q4_df[q4_df["segment"] == segment]
    for feat_name, _ in FEATURES:
        row.append(seg_data[feat_name].iloc[0])
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Features
ax.set_xticks(np.arange(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=11, fontweight="bold")

# Y-axis: Segments with context (now sorted by GMV)
segment_labels = []
for segment in segment_list:
    seg_data = q4_df[q4_df["segment"] == segment]
    suppliers = int(seg_data["suppliers"].iloc[0])
    gmv_m = seg_data["total_gmv"].iloc[0] / 1_000_000
    segment_labels.append(f"{segment}\n({suppliers} suppliers | €{gmv_m:.1f}M)")

ax.set_yticks(np.arange(len(segment_list)))
ax.set_yticklabels(segment_labels, fontsize=11)

# Title
ax.set_title(
    "What Do High-Performing Suppliers Actually Use?\n"
    "Feature Adoption Rates for Top 25% GMV Suppliers (Q4) by Segment\n"
    "Question: Which pricing features are most adopted by successful suppliers?",
    fontsize=15,
    fontweight="bold",
    pad=25
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=10, 
               fontweight="bold")

# =============================================================================
# IMPROVEMENT 2: Add annotation highlighting Leisure Brand API pricing
# =============================================================================
# Find the position of "Leisure Brand" row and "API pricing" column
api_pricing_idx = [i for i, (name, _) in enumerate(FEATURES) if name == "API pricing"][0]
leisure_brand_idx = segment_list.index("Leisure Brand") if "Leisure Brand" in segment_list else None

if leisure_brand_idx is not None:
    # Draw a rectangle around the cell
    rect = patches.Rectangle(
        (api_pricing_idx - 0.48, leisure_brand_idx - 0.48),
        0.96, 0.96,
        linewidth=3,
        edgecolor=ORANGE_DARK,
        facecolor='none',
        linestyle='--'
    )
    ax.add_patch(rect)
    
    # Add annotation with arrow
    ax.annotate(
        'Leisure Brand is the only segment\nwhere API pricing shows strong\nadoption (75%)',
        xy=(api_pricing_idx, leisure_brand_idx),
        xytext=(api_pricing_idx + 2.5, leisure_brand_idx - 1.5),
        fontsize=10,
        fontweight='bold',
        color=ORANGE_DARK,
        bbox=dict(boxstyle="round,pad=0.5", facecolor=ORANGE_PALE, edgecolor=ORANGE_DARK, linewidth=2, alpha=0.9),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', color=ORANGE_DARK, lw=2.5)
    )

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate Among Top Performers (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segment_list)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Q4 Adoption Heatmap (sorted by GMV, with API pricing annotation)")

# =============================================================================
# CHART 2: FEATURE RANKING (IMPROVED WITH COLOR CODING)
# =============================================================================

# Calculate average adoption across all Q4 suppliers (weighted by GMV)
total_q4_gmv = q4_df["total_gmv"].sum()
feature_rankings = []

for feat_name, _ in FEATURES:
    weighted_avg = 0
    for _, row in q4_df.iterrows():
        weight = row["total_gmv"] / total_q4_gmv
        weighted_avg += row[feat_name] * weight
    
    # Also get range (min to max across segments)
    min_val = q4_df[feat_name].min()
    max_val = q4_df[feat_name].max()
    
    feature_rankings.append({
        "feature": feat_name,
        "avg_adoption": weighted_avg,
        "min": min_val,
        "max": max_val,
        "range": max_val - min_val
    })

ranking_df = pd.DataFrame(feature_rankings).sort_values("avg_adoption", ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(len(ranking_df))

# =============================================================================
# IMPROVEMENT 3: Color-code bars based on adoption thresholds
# =============================================================================
def get_bar_color(adoption_rate):
    """Assign color based on adoption threshold"""
    if adoption_rate >= 80:
        return ORANGE_DARK  # High adoption (>80%)
    elif adoption_rate >= 50:
        return ORANGE       # Medium-high adoption (50-80%)
    elif adoption_rate >= 20:
        return ORANGE_LIGHT # Medium adoption (20-50%)
    else:
        return ORANGE_PALE  # Low adoption (<20%)

# Create bars with different colors
bar_colors = [get_bar_color(row["avg_adoption"]) for _, row in ranking_df.iterrows()]
bars = ax.barh(y_pos, ranking_df["avg_adoption"], 
               color=bar_colors, edgecolor=GREY_DARK, linewidth=1.5, alpha=0.9)

# Add range indicators (min-max across segments)
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], 
            color=GREY_DARK, linewidth=3, alpha=0.6, zorder=1)
    ax.scatter([row["min"], row["max"]], [i, i], 
               color=GREY_DARK, s=50, zorder=2, alpha=0.8)

# Value labels
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.text(row["avg_adoption"] + 2, i, f"{row['avg_adoption']:.0f}%", 
           va="center", ha="left", fontsize=10, fontweight="bold", color=GREY_DARK)

ax.set_yticks(y_pos)
ax.set_yticklabels(ranking_df["feature"], fontsize=11)
ax.set_xlabel("Adoption Rate Among Top Performers (%)", fontsize=12, fontweight="bold")
ax.set_title(
    "Feature Adoption Ranking: What Top Performers Use Most\n"
    "GMV-weighted average across all segments (Q4 suppliers only)\n"
    "Bar = weighted average | Line = range across segments | Color = adoption level",
    fontsize=14,
    fontweight="bold",
    pad=20
)
ax.set_xlim(0, 110)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Add legend for color coding
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=ORANGE_DARK, edgecolor=GREY_DARK, label='High adoption (≥80%)'),
    Patch(facecolor=ORANGE, edgecolor=GREY_DARK, label='Medium-high (50-79%)'),
    Patch(facecolor=ORANGE_LIGHT, edgecolor=GREY_DARK, label='Medium (20-49%)'),
    Patch(facecolor=ORANGE_PALE, edgecolor=GREY_DARK, label='Low (<20%)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10, framealpha=0.95)

# Update summary box
top_3 = ranking_df.nlargest(3, "avg_adoption")["feature"].tolist()
summary = "Most Adopted by Top Performers:\n" + "\n".join([f"{i+1}. {f}" for i, f in enumerate(top_3)])
ax.text(
    0.98, 0.28, summary,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.4, edgecolor=ORANGE_DARK, linewidth=2)
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Q4 Feature Ranking (color-coded by adoption level)")

# =============================================================================
# SUMMARY OUTPUT
# =============================================================================
print("\n" + "="*100)
print("HIGH-PERFORMING SUPPLIER FEATURE ADOPTION (Top 25% by GMV within each segment)")
print("Segments sorted by total GMV (descending)")
print("="*100)
print(q4_df[["segment", "suppliers", "tours", "total_gmv"]].to_string(index=False))
print("="*100)

print("\n" + "="*100)
print("SUMMARY: Top 5 Features Adopted by High-Performing Suppliers (Q4) in Each Segment")
print("="*100)

for segment in segment_list:
    seg_data = q4_df[q4_df["segment"] == segment].iloc[0]
    
    feature_adoptions = []
    for feat_name, _ in FEATURES:
        feature_adoptions.append((feat_name, seg_data[feat_name]))
    
    feature_adoptions = sorted(feature_adoptions, key=lambda x: x[1], reverse=True)
    
    print(f"\n{segment} ({int(seg_data['suppliers'])} suppliers | €{seg_data['total_gmv']/1e6:.1f}M GMV):")
    print(f"{'Feature':<40} {'Adoption':<15}")
    print("-" * 55)
    for feat, adoption in feature_adoptions[:5]:
        print(f"{feat:<40} {adoption:>12.0f}%")

print("\n" + "="*100)
print("✓ Analysis complete with all improvements")
print("="*100)


In [0]:
# =============================================================================
# CORRECTED ANALYSIS - Meaningful Metrics Only
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# CORRECTED FEATURES - More Meaningful Metrics
# =============================================================================

# FIX 1: Multiple tour options (>1, not just >0)
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)

# FIX 2: For individual pricing, focus on sophisticated usage
# We'll keep basic individual pricing but add context
df["Individual pricing with tiers"] = df.get("has_individual_scales", 0)
df["Individual pricing with addons"] = ((df.get("has_individual_pricing", 0) == 1) & (df.get("has_addons", 0) == 1)).astype(int)

# Other meaningful features
FEATURES = [
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Individual pricing with add-ons", "Individual pricing with addons"),
    ("Group pricing with multiple tiers", "has_group_scales"),
    ("Add-ons with multiple tiers", "has_addon_scales"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# CALCULATE SUPPLIER GMV AND QUARTILES
# =============================================================================
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

def assign_quartile(group):
    active = group[group["supplier_gmv_l12mo"] > 0].copy()
    if len(active) == 0:
        group["quartile"] = "No GMV"
        return group
    
    q25 = active["supplier_gmv_l12mo"].quantile(0.25)
    q50 = active["supplier_gmv_l12mo"].quantile(0.50)
    q75 = active["supplier_gmv_l12mo"].quantile(0.75)
    
    def get_q(x):
        if x <= 0: return "No GMV"
        if x <= q25: return "Q1"
        if x <= q50: return "Q2"
        if x <= q75: return "Q3"
        return "Q4"
    
    group["quartile"] = group["supplier_gmv_l12mo"].apply(get_q)
    return group

supplier_gmv = supplier_gmv.groupby("supplier_segment", group_keys=False).apply(assign_quartile)
df = df.merge(supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "quartile"]], on="supplier_id", how="left")
df["quartile"] = df["quartile"].fillna("No GMV")

# =============================================================================
# FOCUS: Q4 (TOP PERFORMERS) ADOPTION ONLY
# =============================================================================
q4_data = df[df["quartile"] == "Q4"].copy()

segments = sorted([s for s in df["supplier_segment"].dropna().unique() 
                   if df[(df["supplier_segment"] == s) & (df["quartile"] == "Q4")].shape[0] > 0])

q4_adoption = []
for segment in segments:
    seg_q4 = q4_data[q4_data["supplier_segment"] == segment]
    
    if len(seg_q4) == 0:
        continue
    
    row = {
        "segment": segment,
        "suppliers": seg_q4["supplier_id"].nunique(),
        "tours": len(seg_q4),
        "total_gmv": seg_q4["gmv_l12mo"].sum(),
    }
    
    for feat_name, col in FEATURES:
        row[feat_name] = 100.0 * seg_q4[col].mean()
    
    q4_adoption.append(row)

q4_df = pd.DataFrame(q4_adoption)
q4_df = q4_df.sort_values("total_gmv", ascending=False)

print("\n" + "="*100)
print("CORRECTED METRICS:")
print("="*100)
print("1. Replaced 'Tour options' with 'Multiple tour options (>1)' - shows product variety")
print("2. Focusing on sophisticated individual pricing:")
print("   - Individual pricing WITH multiple tiers (adult/child/senior)")
print("   - Individual pricing WITH add-ons")
print("="*100)

# =============================================================================
# CHART 1: CORRECTED HEATMAP
# =============================================================================

fig, ax = plt.subplots(figsize=(16, 10))

# Build matrix
feature_names = [f[0] for f in FEATURES]
segment_list = q4_df["segment"].tolist()

matrix = []
for segment in segment_list:
    row = []
    seg_data = q4_df[q4_df["segment"] == segment]
    for feat_name, _ in FEATURES:
        row.append(seg_data[feat_name].iloc[0])
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Features
ax.set_xticks(np.arange(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=11, fontweight="bold")

# Y-axis: Segments
segment_labels = []
for segment in segment_list:
    seg_data = q4_df[q4_df["segment"] == segment]
    suppliers = int(seg_data["suppliers"].iloc[0])
    gmv_m = seg_data["total_gmv"].iloc[0] / 1_000_000
    segment_labels.append(f"{segment}\n({suppliers} suppliers | €{gmv_m:.1f}M)")

ax.set_yticks(np.arange(len(segment_list)))
ax.set_yticklabels(segment_labels, fontsize=11)

# Title
ax.set_title(
    "What Do High-Performing Suppliers Actually Use?\n"
    "Feature Adoption Rates for Top 25% GMV Suppliers (Q4) by Segment\n"
    "Corrected: Multiple options (>1) & sophisticated individual pricing only",
    fontsize=15,
    fontweight="bold",
    pad=25
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=10, 
               fontweight="bold")

# Add annotation for API pricing
api_pricing_idx = [i for i, (name, _) in enumerate(FEATURES) if name == "API pricing"][0]
leisure_brand_idx = segment_list.index("Leisure Brand") if "Leisure Brand" in segment_list else None

if leisure_brand_idx is not None:
    rect = patches.Rectangle(
        (api_pricing_idx - 0.48, leisure_brand_idx - 0.48),
        0.96, 0.96,
        linewidth=3,
        edgecolor=ORANGE_DARK,
        facecolor='none',
        linestyle='--'
    )
    ax.add_patch(rect)
    
    ax.annotate(
        'Leisure Brand: 75% API pricing\n(highest across all segments)',
        xy=(api_pricing_idx, leisure_brand_idx),
        xytext=(api_pricing_idx + 2.5, leisure_brand_idx - 1.5),
        fontsize=10,
        fontweight='bold',
        color=ORANGE_DARK,
        bbox=dict(boxstyle="round,pad=0.5", facecolor=ORANGE_PALE, edgecolor=ORANGE_DARK, linewidth=2, alpha=0.9),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', color=ORANGE_DARK, lw=2.5)
    )

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate Among Top Performers (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segment_list)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Corrected Q4 Adoption Heatmap")

# =============================================================================
# CHART 2: CORRECTED RANKING
# =============================================================================

total_q4_gmv = q4_df["total_gmv"].sum()
feature_rankings = []

for feat_name, _ in FEATURES:
    weighted_avg = 0
    for _, row in q4_df.iterrows():
        weight = row["total_gmv"] / total_q4_gmv
        weighted_avg += row[feat_name] * weight
    
    min_val = q4_df[feat_name].min()
    max_val = q4_df[feat_name].max()
    
    feature_rankings.append({
        "feature": feat_name,
        "avg_adoption": weighted_avg,
        "min": min_val,
        "max": max_val,
        "range": max_val - min_val
    })

ranking_df = pd.DataFrame(feature_rankings).sort_values("avg_adoption", ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(len(ranking_df))

def get_bar_color(adoption_rate):
    if adoption_rate >= 80:
        return ORANGE_DARK
    elif adoption_rate >= 50:
        return ORANGE
    elif adoption_rate >= 20:
        return ORANGE_LIGHT
    else:
        return ORANGE_PALE

bar_colors = [get_bar_color(row["avg_adoption"]) for _, row in ranking_df.iterrows()]
bars = ax.barh(y_pos, ranking_df["avg_adoption"], 
               color=bar_colors, edgecolor=GREY_DARK, linewidth=1.5, alpha=0.9)

# Add range indicators
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], 
            color=GREY_DARK, linewidth=3, alpha=0.6, zorder=1)
    ax.scatter([row["min"], row["max"]], [i, i], 
               color=GREY_DARK, s=50, zorder=2, alpha=0.8)

# Value labels
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.text(row["avg_adoption"] + 2, i, f"{row['avg_adoption']:.0f}%", 
           va="center", ha="left", fontsize=10, fontweight="bold", color=GREY_DARK)

ax.set_yticks(y_pos)
ax.set_yticklabels(ranking_df["feature"], fontsize=11)
ax.set_xlabel("Adoption Rate Among Top Performers (%)", fontsize=12, fontweight="bold")
ax.set_title(
    "Feature Adoption Ranking: What Top Performers Use Most\n"
    "GMV-weighted average across all segments (Q4 suppliers only)\n"
    "Corrected metrics: Multiple options (>1) & sophisticated individual pricing",
    fontsize=14,
    fontweight="bold",
    pad=20
)
ax.set_xlim(0, 110)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=ORANGE_DARK, edgecolor=GREY_DARK, label='High (≥80%)'),
    Patch(facecolor=ORANGE, edgecolor=GREY_DARK, label='Medium-high (50-79%)'),
    Patch(facecolor=ORANGE_LIGHT, edgecolor=GREY_DARK, label='Medium (20-49%)'),
    Patch(facecolor=ORANGE_PALE, edgecolor=GREY_DARK, label='Low (<20%)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10, framealpha=0.95)

# Summary box
top_3 = ranking_df.nlargest(3, "avg_adoption")["feature"].tolist()
summary = "Most Adopted by Top Performers:\n" + "\n".join([f"{i+1}. {f}" for i, f in enumerate(top_3)])
ax.text(
    0.98, 0.28, summary,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.4, edgecolor=ORANGE_DARK, linewidth=2)
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Corrected Q4 Feature Ranking")

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Top 5 Features by Segment (Corrected Metrics)")
print("="*100)

for segment in segment_list:
    seg_data = q4_df[q4_df["segment"] == segment].iloc[0]
    
    feature_adoptions = []
    for feat_name, _ in FEATURES:
        feature_adoptions.append((feat_name, seg_data[feat_name]))
    
    feature_adoptions = sorted(feature_adoptions, key=lambda x: x[1], reverse=True)
    
    print(f"\n{segment} ({int(seg_data['suppliers'])} suppliers | €{seg_data['total_gmv']/1e6:.1f}M GMV):")
    print(f"{'Feature':<50} {'Adoption':<15}")
    print("-" * 65)
    for feat, adoption in feature_adoptions[:5]:
        print(f"{feat:<50} {adoption:>12.0f}%")

print("\n" + "="*100)
print("✓ Analysis complete with corrected, meaningful metrics")
print("="*100)


In [0]:
# =============================================================================
# TOP 80% GMV GENERATING SUPPLIERS ANALYSIS
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# MEANINGFUL FEATURES
# =============================================================================

# Multiple tour options (>1, not just >0)
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)

# Sophisticated individual pricing
df["Individual pricing with tiers"] = df.get("has_individual_scales", 0)
df["Individual pricing with addons"] = ((df.get("has_individual_pricing", 0) == 1) & (df.get("has_addons", 0) == 1)).astype(int)

FEATURES = [
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Individual pricing with add-ons", "Individual pricing with addons"),
    ("Group pricing with multiple tiers", "has_group_scales"),
    ("Add-ons with multiple tiers", "has_addon_scales"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# CALCULATE SUPPLIER GMV AND TOP 80% FLAG
# =============================================================================
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

def assign_top_80_percent(group):
    """
    Flag suppliers who collectively generate the top 80% of GMV within their segment
    """
    # Sort by GMV descending
    group = group.sort_values("supplier_gmv_l12mo", ascending=False)
    
    # Calculate cumulative GMV
    total_gmv = group["supplier_gmv_l12mo"].sum()
    group["cumulative_gmv"] = group["supplier_gmv_l12mo"].cumsum()
    group["cumulative_gmv_pct"] = (group["cumulative_gmv"] / total_gmv) * 100
    
    # Flag top 80%
    group["is_top_80_pct"] = (group["cumulative_gmv_pct"] <= 80).astype(int)
    
    return group

supplier_gmv = supplier_gmv.groupby("supplier_segment", group_keys=False).apply(assign_top_80_percent)

# Merge back to tour-level data
df = df.merge(
    supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "is_top_80_pct", "cumulative_gmv_pct"]], 
    on="supplier_id", 
    how="left"
)
df["is_top_80_pct"] = df["is_top_80_pct"].fillna(0)

print("\n" + "="*100)
print("TOP 80% GMV GENERATING SUPPLIERS")
print("="*100)
print("Method: Within each segment, identified suppliers who collectively generate top 80% of GMV")
print("="*100)

# Show statistics
for segment in sorted(df["supplier_segment"].dropna().unique()):
    seg_suppliers = supplier_gmv[supplier_gmv["supplier_segment"] == segment]
    total_suppliers = len(seg_suppliers)
    top_80_suppliers = seg_suppliers["is_top_80_pct"].sum()
    total_gmv = seg_suppliers["supplier_gmv_l12mo"].sum()
    top_80_gmv = seg_suppliers[seg_suppliers["is_top_80_pct"] == 1]["supplier_gmv_l12mo"].sum()
    
    print(f"\n{segment}:")
    print(f"  Total suppliers: {total_suppliers:,}")
    print(f"  Top 80% GMV suppliers: {top_80_suppliers:,} ({100*top_80_suppliers/total_suppliers:.1f}% of suppliers)")
    print(f"  Their GMV: €{top_80_gmv:,.0f} ({100*top_80_gmv/total_gmv:.1f}% of total)")

print("\n" + "="*100)

# =============================================================================
# FOCUS: TOP 80% GMV SUPPLIERS ONLY
# =============================================================================
top_80_data = df[df["is_top_80_pct"] == 1].copy()

segments = sorted([s for s in df["supplier_segment"].dropna().unique() 
                   if df[(df["supplier_segment"] == s) & (df["is_top_80_pct"] == 1)].shape[0] > 0])

top_80_adoption = []
for segment in segments:
    seg_top = top_80_data[top_80_data["supplier_segment"] == segment]
    
    if len(seg_top) == 0:
        continue
    
    row = {
        "segment": segment,
        "suppliers": seg_top["supplier_id"].nunique(),
        "tours": len(seg_top),
        "total_gmv": seg_top["gmv_l12mo"].sum(),
    }
    
    for feat_name, col in FEATURES:
        row[feat_name] = 100.0 * seg_top[col].mean()
    
    top_80_adoption.append(row)

top_80_df = pd.DataFrame(top_80_adoption)
top_80_df = top_80_df.sort_values("total_gmv", ascending=False)

# =============================================================================
# CHART 1: TOP 80% GMV SUPPLIERS - HEATMAP
# =============================================================================

fig, ax = plt.subplots(figsize=(16, 10))

# Build matrix
feature_names = [f[0] for f in FEATURES]
segment_list = top_80_df["segment"].tolist()

matrix = []
for segment in segment_list:
    row = []
    seg_data = top_80_df[top_80_df["segment"] == segment]
    for feat_name, _ in FEATURES:
        row.append(seg_data[feat_name].iloc[0])
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Features
ax.set_xticks(np.arange(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=11, fontweight="bold")

# Y-axis: Segments
segment_labels = []
for segment in segment_list:
    seg_data = top_80_df[top_80_df["segment"] == segment]
    suppliers = int(seg_data["suppliers"].iloc[0])
    gmv_m = seg_data["total_gmv"].iloc[0] / 1_000_000
    
    # Calculate what % of suppliers this represents
    total_in_segment = supplier_gmv[supplier_gmv["supplier_segment"] == segment].shape[0]
    pct_suppliers = 100 * suppliers / total_in_segment if total_in_segment > 0 else 0
    
    segment_labels.append(f"{segment}\n({suppliers} suppliers = {pct_suppliers:.0f}% | €{gmv_m:.1f}M = 80% GMV)")

ax.set_yticks(np.arange(len(segment_list)))
ax.set_yticklabels(segment_labels, fontsize=10)

# Title
ax.set_title(
    "What Do Revenue-Driving Suppliers Actually Use?\n"
    "Feature Adoption Rates for Top 80% GMV-Generating Suppliers by Segment\n"
    "Question: Which pricing features are most adopted by successful suppliers?",
    fontsize=15,
    fontweight="bold",
    pad=25
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=10, 
               fontweight="bold")

# Add annotation for API pricing
api_pricing_idx = [i for i, (name, _) in enumerate(FEATURES) if name == "API pricing"][0]
leisure_brand_idx = segment_list.index("Leisure Brand") if "Leisure Brand" in segment_list else None

if leisure_brand_idx is not None:
    rect = patches.Rectangle(
        (api_pricing_idx - 0.48, leisure_brand_idx - 0.48),
        0.96, 0.96,
        linewidth=3,
        edgecolor=ORANGE_DARK,
        facecolor='none',
        linestyle='--'
    )
    ax.add_patch(rect)
    
    leisure_api_adoption = matrix[leisure_brand_idx, api_pricing_idx]
    ax.annotate(
        f'Leisure Brand: {leisure_api_adoption:.0f}% API pricing\n(highest across all segments)',
        xy=(api_pricing_idx, leisure_brand_idx),
        xytext=(api_pricing_idx + 2.5, leisure_brand_idx - 1.5),
        fontsize=10,
        fontweight='bold',
        color=ORANGE_DARK,
        bbox=dict(boxstyle="round,pad=0.5", facecolor=ORANGE_PALE, edgecolor=ORANGE_DARK, linewidth=2, alpha=0.9),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', color=ORANGE_DARK, lw=2.5)
    )

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate Among Revenue Drivers (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segment_list)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Top 80% GMV Suppliers - Heatmap")

# =============================================================================
# CHART 2: FEATURE RANKING (TOP 80% GMV SUPPLIERS)
# =============================================================================

total_gmv = top_80_df["total_gmv"].sum()
feature_rankings = []

for feat_name, _ in FEATURES:
    weighted_avg = 0
    for _, row in top_80_df.iterrows():
        weight = row["total_gmv"] / total_gmv
        weighted_avg += row[feat_name] * weight
    
    min_val = top_80_df[feat_name].min()
    max_val = top_80_df[feat_name].max()
    
    feature_rankings.append({
        "feature": feat_name,
        "avg_adoption": weighted_avg,
        "min": min_val,
        "max": max_val,
        "range": max_val - min_val
    })

ranking_df = pd.DataFrame(feature_rankings).sort_values("avg_adoption", ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(len(ranking_df))

def get_bar_color(adoption_rate):
    if adoption_rate >= 80:
        return ORANGE_DARK
    elif adoption_rate >= 50:
        return ORANGE
    elif adoption_rate >= 20:
        return ORANGE_LIGHT
    else:
        return ORANGE_PALE

bar_colors = [get_bar_color(row["avg_adoption"]) for _, row in ranking_df.iterrows()]
bars = ax.barh(y_pos, ranking_df["avg_adoption"], 
               color=bar_colors, edgecolor=GREY_DARK, linewidth=1.5, alpha=0.9)

# Add range indicators
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], 
            color=GREY_DARK, linewidth=3, alpha=0.6, zorder=1)
    ax.scatter([row["min"], row["max"]], [i, i], 
               color=GREY_DARK, s=50, zorder=2, alpha=0.8)

# Value labels
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.text(row["avg_adoption"] + 2, i, f"{row['avg_adoption']:.0f}%", 
           va="center", ha="left", fontsize=10, fontweight="bold", color=GREY_DARK)

ax.set_yticks(y_pos)
ax.set_yticklabels(ranking_df["feature"], fontsize=11)
ax.set_xlabel("Adoption Rate Among Revenue Drivers (%)", fontsize=12, fontweight="bold")
ax.set_title(
    "Feature Adoption Ranking: What Revenue-Driving Suppliers Use Most\n"
    "GMV-weighted average across all segments (Top 80% GMV suppliers only)\n"
    "Bar = weighted average | Line = range across segments | Color = adoption level",
    fontsize=14,
    fontweight="bold",
    pad=20
)
ax.set_xlim(0, 110)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=ORANGE_DARK, edgecolor=GREY_DARK, label='High (≥80%)'),
    Patch(facecolor=ORANGE, edgecolor=GREY_DARK, label='Medium-high (50-79%)'),
    Patch(facecolor=ORANGE_LIGHT, edgecolor=GREY_DARK, label='Medium (20-49%)'),
    Patch(facecolor=ORANGE_PALE, edgecolor=GREY_DARK, label='Low (<20%)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10, framealpha=0.95)

# Summary box
top_3 = ranking_df.nlargest(3, "avg_adoption")["feature"].tolist()
summary = "Most Adopted by Revenue Drivers:\n" + "\n".join([f"{i+1}. {f}" for i, f in enumerate(top_3)])
ax.text(
    0.98, 0.28, summary,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.4, edgecolor=ORANGE_DARK, linewidth=2)
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Top 80% GMV Suppliers - Feature Ranking")

# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Top 5 Features by Segment (Top 80% GMV Suppliers)")
print("="*100)

for segment in segment_list:
    seg_data = top_80_df[top_80_df["segment"] == segment].iloc[0]
    
    feature_adoptions = []
    for feat_name, _ in FEATURES:
        feature_adoptions.append((feat_name, seg_data[feat_name]))
    
    feature_adoptions = sorted(feature_adoptions, key=lambda x: x[1], reverse=True)
    
    print(f"\n{segment} ({int(seg_data['suppliers'])} suppliers | €{seg_data['total_gmv']/1e6:.1f}M GMV):")
    print(f"{'Feature':<50} {'Adoption':<15}")
    print("-" * 65)
    for feat, adoption in feature_adoptions[:5]:
        print(f"{feat:<50} {adoption:>12.0f}%")

print("\n" + "="*100)
print("✓ Analysis complete: Top 80% GMV-generating suppliers")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 2: Managed vs Not Managed Feature Adoption
# FIX: is_managed is "yes"/"no" not 1/0
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# FIX: is_managed is string "yes"/"no", not 1/0
if "is_managed" in df.columns:
    df["is_managed"] = df["is_managed"].astype(str).str.lower().fillna("no")
    df["is_managed"] = df["is_managed"].replace({"yes": "yes", "no": "no"})

# =============================================================================
# MEANINGFUL FEATURES
# =============================================================================
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)
df["Individual pricing with tiers"] = df.get("has_individual_scales", 0)
df["Individual pricing with addons"] = ((df.get("has_individual_pricing", 0) == 1) & (df.get("has_addons", 0) == 1)).astype(int)

FEATURES = [
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Individual pricing with add-ons", "Individual pricing with addons"),
    ("Group pricing with multiple tiers", "has_group_scales"),
    ("Add-ons with multiple tiers", "has_addon_scales"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# CALCULATE SUPPLIER GMV AND TOP 80%
# =============================================================================
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({"gmv_l12mo": "sum", "is_managed": "first"})  # Use first instead of max for string
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

def assign_top_80_percent(group):
    group = group.sort_values("supplier_gmv_l12mo", ascending=False)
    total_gmv = group["supplier_gmv_l12mo"].sum()
    group["cumulative_gmv"] = group["supplier_gmv_l12mo"].cumsum()
    group["cumulative_gmv_pct"] = (group["cumulative_gmv"] / total_gmv) * 100
    group["is_top_80_pct"] = (group["cumulative_gmv_pct"] <= 80).astype(int)
    return group

supplier_gmv = supplier_gmv.groupby("supplier_segment", group_keys=False).apply(assign_top_80_percent)

df = df.merge(
    supplier_gmv[["supplier_id", "supplier_gmv_l12mo", "is_top_80_pct", "is_managed"]], 
    on="supplier_id", 
    how="left",
    suffixes=('', '_supplier')
)
df["is_top_80_pct"] = df["is_top_80_pct"].fillna(0)
df["is_managed"] = df["is_managed_supplier"].fillna("no")

# Filter to top 80% GMV suppliers only
top_80_data = df[df["is_top_80_pct"] == 1].copy()

print("\n" + "="*100)
print("QUESTION 2: MANAGED VS NOT MANAGED FEATURE ADOPTION")
print("="*100)
print("Question: What features do Managed vs Not Managed suppliers adopt?")
print("Sample: Top 80% GMV-generating suppliers only")
print("="*100)

# Check managed values
print(f"\nManaged values in data: {top_80_data['is_managed'].unique()}")

# =============================================================================
# CALCULATE ADOPTION BY MANAGED STATUS
# =============================================================================
segments = sorted(top_80_data["supplier_segment"].dropna().unique())

adoption_data = []
for segment in segments:
    seg_data = top_80_data[top_80_data["supplier_segment"] == segment]
    
    # FIX: Use "yes" and "no" strings
    managed = seg_data[seg_data["is_managed"] == "yes"]
    not_managed = seg_data[seg_data["is_managed"] == "no"]
    
    if len(managed) == 0 and len(not_managed) == 0:
        continue
    
    for status_data, status_label in [
        (managed, "Managed"),
        (not_managed, "Not Managed")
    ]:
        if len(status_data) == 0:
            continue
            
        row = {
            "segment": segment,
            "management_status": status_label,
            "suppliers": status_data["supplier_id"].nunique(),
            "tours": len(status_data),
            "gmv": status_data["gmv_l12mo"].sum(),
        }
        
        for feat_name, col in FEATURES:
            row[feat_name] = 100.0 * status_data[col].mean()
        
        adoption_data.append(row)

adoption_df = pd.DataFrame(adoption_data)

# Show sample distribution
print("\nSample Distribution:")
for segment in segments:
    seg_managed = adoption_df[(adoption_df["segment"] == segment) & (adoption_df["management_status"] == "Managed")]
    seg_not_managed = adoption_df[(adoption_df["segment"] == segment) & (adoption_df["management_status"] == "Not Managed")]
    
    m_count = int(seg_managed["suppliers"].sum()) if len(seg_managed) > 0 else 0
    nm_count = int(seg_not_managed["suppliers"].sum()) if len(seg_not_managed) > 0 else 0
    total = m_count + nm_count
    
    if total > 0:
        print(f"  {segment}: {m_count} managed ({100*m_count/total:.0f}%), {nm_count} not managed ({100*nm_count/total:.0f}%)")

# =============================================================================
# CHART 1: GROUPED BAR CHART BY SEGMENT
# For each segment, show Managed vs Not Managed side-by-side for key features
# =============================================================================

# Select features with meaningful adoption to keep chart readable
KEY_FEATURES = [
    "Multiple tour options (>1)",
    "Any pricing scales",
    "API pricing",
    "Individual pricing with multiple tiers",
    "Group pricing",
    "Add-ons",
    "Live pricing & availability"
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

segment_order = sorted(segments, key=lambda s: adoption_df[adoption_df["segment"]==s]["gmv"].sum(), reverse=True)

for idx, segment in enumerate(segment_order[:6]):  # Top 6 segments by GMV
    ax = axes[idx]
    
    seg_data = adoption_df[adoption_df["segment"] == segment]
    managed_data = seg_data[seg_data["management_status"] == "Managed"]
    not_managed_data = seg_data[seg_data["management_status"] == "Not Managed"]
    
    x = np.arange(len(KEY_FEATURES))
    width = 0.35
    
    managed_values = [float(managed_data[feat].iloc[0]) if len(managed_data) > 0 else 0 for feat in KEY_FEATURES]
    not_managed_values = [float(not_managed_data[feat].iloc[0]) if len(not_managed_data) > 0 else 0 for feat in KEY_FEATURES]
    
    bars1 = ax.bar(x - width/2, managed_values, width, label='Managed', 
                   color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1)
    bars2 = ax.bar(x + width/2, not_managed_values, width, label='Not Managed', 
                   color=GREY_LIGHT, edgecolor=GREY_DARK, linewidth=1)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 5:
                ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                       f'{height:.0f}%',
                       ha='center', va='bottom', fontsize=7, fontweight='bold', color=GREY_DARK)
    
    # Get supplier counts
    m_suppliers = int(managed_data["suppliers"].iloc[0]) if len(managed_data) > 0 else 0
    nm_suppliers = int(not_managed_data["suppliers"].iloc[0]) if len(not_managed_data) > 0 else 0
    
    ax.set_ylabel('Adoption Rate (%)', fontsize=10, fontweight='bold')
    ax.set_title(f'{segment}\n{m_suppliers} managed | {nm_suppliers} not managed', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(KEY_FEATURES, rotation=45, ha='right', fontsize=8)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

# Hide extra subplots if less than 6 segments
for idx in range(len(segment_order), 6):
    axes[idx].axis('off')

plt.suptitle(
    'Managed vs Not Managed: Feature Adoption by Segment\n'
    'Question: Do managed suppliers adopt different features than not managed? (Top 80% GMV suppliers)',
    fontsize=15,
    fontweight='bold',
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

print("\n✓ Chart 1: Grouped bar chart by segment")

# =============================================================================
# CHART 2: LOLLIPOP/DOT PLOT - Overall Adoption Comparison
# Shows all features, Managed vs Not Managed as connected dots
# =============================================================================

# Calculate overall weighted averages
overall_adoption = []
for feat_name, _ in FEATURES:
    managed_avg = 0
    not_managed_avg = 0
    
    managed_data = adoption_df[adoption_df["management_status"] == "Managed"]
    not_managed_data = adoption_df[adoption_df["management_status"] == "Not Managed"]
    
    if len(managed_data) > 0:
        managed_gmv = managed_data["gmv"].sum()
        for _, row in managed_data.iterrows():
            weight = row["gmv"] / managed_gmv
            managed_avg += row[feat_name] * weight
    
    if len(not_managed_data) > 0:
        not_managed_gmv = not_managed_data["gmv"].sum()
        for _, row in not_managed_data.iterrows():
            weight = row["gmv"] / not_managed_gmv
            not_managed_avg += row[feat_name] * weight
    
    overall_adoption.append({
        "feature": feat_name,
        "managed": managed_avg,
        "not_managed": not_managed_avg
    })

overall_df = pd.DataFrame(overall_adoption)
overall_df = overall_df.sort_values("managed", ascending=True)

fig, ax = plt.subplots(figsize=(14, 9))

y_pos = np.arange(len(overall_df))

# Draw connecting lines
for i, (idx, row) in enumerate(overall_df.iterrows()):
    ax.plot([row["not_managed"], row["managed"]], [i, i], 
            color=GREY, linewidth=2, alpha=0.6, zorder=1)

# Draw dots
ax.scatter(overall_df["managed"], y_pos, 
          s=200, color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=2, 
          label='Managed', zorder=3, alpha=0.9)
ax.scatter(overall_df["not_managed"], y_pos, 
          s=200, color=GREY_LIGHT, edgecolor=GREY_DARK, linewidth=2, 
          label='Not Managed', zorder=3, alpha=0.9)

# Add value labels
for i, (idx, row) in enumerate(overall_df.iterrows()):
    # Managed value
    ax.text(row["managed"] + 1.5, i, f'{row["managed"]:.0f}%', 
           va='center', ha='left', fontsize=9, fontweight='bold', color=ORANGE_DARK)
    # Not Managed value
    ax.text(row["not_managed"] - 1.5, i, f'{row["not_managed"]:.0f}%', 
           va='center', ha='right', fontsize=9, fontweight='bold', color=GREY)

ax.set_yticks(y_pos)
ax.set_yticklabels(overall_df["feature"], fontsize=11)
ax.set_xlabel('Adoption Rate (%)', fontsize=12, fontweight='bold')
ax.set_title(
    'Managed vs Not Managed: Overall Feature Adoption Comparison\n'
    'GMV-weighted average across all segments (Top 80% GMV suppliers)\n'
    'Line connects the two groups for each feature',
    fontsize=14,
    fontweight='bold',
    pad=20
)
ax.set_xlim(0, 110)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.legend(fontsize=11, loc='lower right', framealpha=0.95)

# Add summary counts
managed_total = adoption_df[adoption_df["management_status"] == "Managed"]["suppliers"].sum()
not_managed_total = adoption_df[adoption_df["management_status"] == "Not Managed"]["suppliers"].sum()
summary_text = f"Sample: {int(managed_total)} managed suppliers | {int(not_managed_total)} not managed suppliers"
ax.text(0.98, 0.02, summary_text,
       transform=ax.transAxes,
       fontsize=10,
       verticalalignment='bottom',
       horizontalalignment='right',
       bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.3, edgecolor=GREY))

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Lollipop chart showing overall adoption comparison")

# =============================================================================
# SUMMARY TABLE
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Managed vs Not Managed Adoption Rates")
print("="*100)

for _, row in overall_df.sort_values("managed", ascending=False).head(10).iterrows():
    diff = row["managed"] - row["not_managed"]
    print(f"\n{row['feature']}:")
    print(f"  Managed: {row['managed']:.0f}%")
    print(f"  Not Managed: {row['not_managed']:.0f}%")
    if abs(diff) > 5:
        direction = "higher" if diff > 0 else "lower"
        print(f"  → Managed is {abs(diff):.0f}pp {direction}")

print("\n" + "="*100)
print("✓ Analysis complete: Managed vs Not Managed adoption comparison")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 2 REVISED: Managed vs Not Managed - CONTROLLING FOR SUPPLIER SIZE
# Compare at same supplier size and segment
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# is_managed is "yes"/"no"
if "is_managed" in df.columns:
    df["is_managed"] = df["is_managed"].astype(str).str.lower().fillna("no")

# =============================================================================
# MEANINGFUL FEATURES
# =============================================================================
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)

FEATURES = [
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# CALCULATE SUPPLIER-LEVEL GMV AND MANAGED STATUS
# =============================================================================
supplier_data = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "gmv_l12mo": "sum",
          "is_managed": "first"
      })
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

# =============================================================================
# CREATE SUPPLIER SIZE BUCKETS
# =============================================================================
def assign_size_bucket(gmv):
    """Assign supplier to size bucket based on GMV"""
    if gmv == 0:
        return "No GMV"
    elif gmv < 10000:
        return "Micro (€0-10K)"
    elif gmv < 50000:
        return "Small (€10-50K)"
    elif gmv < 200000:
        return "Medium (€50-200K)"
    elif gmv < 500000:
        return "Large (€200-500K)"
    else:
        return "Enterprise (€500K+)"

supplier_data["size_bucket"] = supplier_data["supplier_gmv_l12mo"].apply(assign_size_bucket)

SIZE_BUCKET_ORDER = [
    "Micro (€0-10K)",
    "Small (€10-50K)", 
    "Medium (€50-200K)",
    "Large (€200-500K)",
    "Enterprise (€500K+)"
]

# Merge back to tour level
df = df.merge(
    supplier_data[["supplier_id", "supplier_gmv_l12mo", "size_bucket", "is_managed"]],
    on="supplier_id",
    how="left",
    suffixes=('', '_supplier')
)
df["is_managed"] = df["is_managed_supplier"].fillna("no")

print("\n" + "="*100)
print("QUESTION 2 REVISED: MANAGED VS NOT MANAGED - CONTROLLING FOR SUPPLIER SIZE")
print("="*100)
print("Question: At the same supplier size, do managed suppliers adopt different features?")
print("="*100)

# Show distribution by size and managed status
print("\nSupplier Distribution by Size Bucket and Managed Status:")
print(f"{'Size Bucket':<25} {'Managed':<15} {'Not Managed':<15} {'% Managed':<15}")
print("-" * 70)

for bucket in SIZE_BUCKET_ORDER:
    bucket_suppliers = supplier_data[supplier_data["size_bucket"] == bucket]
    managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "yes"])
    not_managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "no"])
    total = managed + not_managed
    pct_managed = 100 * managed / total if total > 0 else 0
    
    print(f"{bucket:<25} {managed:<15,} {not_managed:<15,} {pct_managed:<15.1f}%")

# =============================================================================
# CALCULATE ADOPTION BY SIZE BUCKET AND MANAGED STATUS
# =============================================================================
adoption_data = []

for bucket in SIZE_BUCKET_ORDER:
    bucket_data = df[df["size_bucket"] == bucket]
    
    managed = bucket_data[bucket_data["is_managed"] == "yes"]
    not_managed = bucket_data[bucket_data["is_managed"] == "no"]
    
    for status_data, status_label in [
        (managed, "Managed"),
        (not_managed, "Not Managed")
    ]:
        if len(status_data) == 0:
            continue
            
        row = {
            "size_bucket": bucket,
            "management_status": status_label,
            "suppliers": status_data["supplier_id"].nunique(),
            "tours": len(status_data),
            "gmv": status_data["gmv_l12mo"].sum(),
        }
        
        for feat_name, col in FEATURES:
            row[feat_name] = 100.0 * status_data[col].mean()
        
        adoption_data.append(row)

adoption_df = pd.DataFrame(adoption_data)

# =============================================================================
# CHART 1: FACETED BAR CHART - Feature Adoption by Size Bucket
# Shows Managed vs Not Managed side-by-side for each size bucket
# =============================================================================

fig, axes = plt.subplots(1, 5, figsize=(20, 6), sharey=True)

feature_names = [f[0] for f in FEATURES]

for idx, bucket in enumerate(SIZE_BUCKET_ORDER):
    ax = axes[idx]
    
    bucket_data = adoption_df[adoption_df["size_bucket"] == bucket]
    managed_data = bucket_data[bucket_data["management_status"] == "Managed"]
    not_managed_data = bucket_data[bucket_data["management_status"] == "Not Managed"]
    
    x = np.arange(len(feature_names))
    width = 0.35
    
    managed_values = [float(managed_data[feat].iloc[0]) if len(managed_data) > 0 else 0 for feat in feature_names]
    not_managed_values = [float(not_managed_data[feat].iloc[0]) if len(not_managed_data) > 0 else 0 for feat in feature_names]
    
    bars1 = ax.bar(x - width/2, managed_values, width, label='Managed', 
                   color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    bars2 = ax.bar(x + width/2, not_managed_values, width, label='Not Managed', 
                   color=GREY_LIGHT, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    
    # Add value labels on bars (only if >5%)
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 5:
                ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                       f'{height:.0f}%',
                       ha='center', va='bottom', fontsize=7, fontweight='bold', color=GREY_DARK)
    
    # Get supplier counts
    m_suppliers = int(managed_data["suppliers"].iloc[0]) if len(managed_data) > 0 else 0
    nm_suppliers = int(not_managed_data["suppliers"].iloc[0]) if len(not_managed_data) > 0 else 0
    
    ax.set_title(f'{bucket}\n{m_suppliers} M | {nm_suppliers} NM', fontsize=10, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if idx == 0:
        ax.set_ylabel('Adoption Rate (%)', fontsize=11, fontweight='bold')
        ax.legend(fontsize=9, loc='upper left')

plt.suptitle(
    'Managed vs Not Managed: Feature Adoption Across Supplier Size Buckets\n'
    'Question: At the same supplier size, does managed support drive different adoption patterns?',
    fontsize=14,
    fontweight='bold',
    y=1.02
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Feature adoption by size bucket (controlling for size)")

# =============================================================================
# CHART 2: HEATMAP - Adoption Rates by Size x Management Status
# Rows = Features, Columns = Size buckets split by Managed/Not Managed
# =============================================================================

fig, ax = plt.subplots(figsize=(18, 10))

# Build matrix: features (rows) x (size_bucket × management_status) (columns)
column_labels = []
matrix = []

for feat_name, _ in FEATURES:
    row = []
    for bucket in SIZE_BUCKET_ORDER:
        for status in ["Managed", "Not Managed"]:
            data = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                              (adoption_df["management_status"] == status)]
            
            if len(data) > 0:
                row.append(data[feat_name].iloc[0])
            else:
                row.append(0)
            
            # Build column labels (only once)
            if len(matrix) == 0:
                supplier_count = int(data["suppliers"].iloc[0]) if len(data) > 0 else 0
                column_labels.append(f"{bucket}\n{status}\n({supplier_count} suppliers)")
    
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Size × Management combinations
ax.set_xticks(np.arange(len(column_labels)))
ax.set_xticklabels(column_labels, rotation=45, ha='right', fontsize=9)

# Y-axis: Features
ax.set_yticks(np.arange(len(feature_names)))
ax.set_yticklabels(feature_names, fontsize=11, fontweight='bold')

# Title
ax.set_title(
    'Feature Adoption: Managed vs Not Managed Across Supplier Sizes\n'
    'Question: Does managed support drive adoption when controlling for supplier size?\n'
    'Each size bucket split into Managed (left) and Not Managed (right)',
    fontsize=14,
    fontweight='bold',
    pad=20
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=9, 
               fontweight="bold")

# Add vertical lines to separate size buckets
for i in range(1, len(SIZE_BUCKET_ORDER)):
    ax.axvline(i * 2 - 0.5, color=GREY_DARK, linewidth=3, alpha=0.7)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(column_labels)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Heatmap showing adoption across size buckets and management status")

# =============================================================================
# SUMMARY: WHERE DOES MANAGED SUPPORT MATTER?
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Features Where Managed Support Shows Different Adoption (by size bucket)")
print("="*100)

for bucket in SIZE_BUCKET_ORDER:
    bucket_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                  (adoption_df["management_status"] == "Managed")]
    bucket_not_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                      (adoption_df["management_status"] == "Not Managed")]
    
    if len(bucket_managed) == 0 or len(bucket_not_managed) == 0:
        continue
    
    print(f"\n{bucket}:")
    
    differences = []
    for feat_name, _ in FEATURES:
        m_val = float(bucket_managed[feat_name].iloc[0])
        nm_val = float(bucket_not_managed[feat_name].iloc[0])
        diff = m_val - nm_val
        
        if abs(diff) > 10:  # Only show meaningful differences
            differences.append((feat_name, m_val, nm_val, diff))
    
    if differences:
        differences = sorted(differences, key=lambda x: abs(x[3]), reverse=True)
        for feat, m_val, nm_val, diff in differences:
            direction = "higher" if diff > 0 else "lower"
            print(f"  {feat}: Managed {m_val:.0f}% vs Not Managed {nm_val:.0f}% ({diff:+.0f}pp {direction})")
    else:
        print(f"  No meaningful differences (all features <10pp gap)")

print("\n" + "="*100)
print("✓ Analysis complete: Managed vs Not Managed controlling for supplier size")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 2 REVISED: Managed vs Not Managed - Size Buckets Based on Tertiles
# Small/Medium/Large based on 33rd and 67th percentiles
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# is_managed is "yes"/"no"
if "is_managed" in df.columns:
    df["is_managed"] = df["is_managed"].astype(str).str.lower().fillna("no")

# =============================================================================
# MEANINGFUL FEATURES
# =============================================================================
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)

FEATURES = [
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# CALCULATE SUPPLIER-LEVEL GMV AND MANAGED STATUS
# =============================================================================
supplier_data = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "gmv_l12mo": "sum",
          "is_managed": "first"
      })
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

# =============================================================================
# CREATE SIZE BUCKETS BASED ON TERTILES (33rd and 67th percentiles)
# =============================================================================

# Filter to suppliers with GMV > 0
active_suppliers = supplier_data[supplier_data["supplier_gmv_l12mo"] > 0].copy()

# Calculate tertiles
p33 = active_suppliers["supplier_gmv_l12mo"].quantile(0.33)
p67 = active_suppliers["supplier_gmv_l12mo"].quantile(0.67)

print("\n" + "="*100)
print("SIZE BUCKET DEFINITIONS (Based on GMV Tertiles)")
print("="*100)
print(f"Small:  €0 - €{p33:,.0f} (bottom 33%)")
print(f"Medium: €{p33:,.0f} - €{p67:,.0f} (middle 33%)")
print(f"Large:  €{p67:,.0f}+ (top 33%)")
print("="*100)

def assign_size_bucket_tertile(gmv):
    """Assign supplier to size bucket based on tertiles"""
    if gmv <= 0:
        return "No GMV"
    elif gmv <= p33:
        return "Small"
    elif gmv <= p67:
        return "Medium"
    else:
        return "Large"

supplier_data["size_bucket"] = supplier_data["supplier_gmv_l12mo"].apply(assign_size_bucket_tertile)

SIZE_BUCKET_ORDER = ["Small", "Medium", "Large"]

# Merge back to tour level
df = df.merge(
    supplier_data[["supplier_id", "supplier_gmv_l12mo", "size_bucket", "is_managed"]],
    on="supplier_id",
    how="left",
    suffixes=('', '_supplier')
)
df["is_managed"] = df["is_managed_supplier"].fillna("no")

print("\nQUESTION 2: MANAGED VS NOT MANAGED - CONTROLLING FOR SUPPLIER SIZE")
print("Question: At the same supplier size, do managed suppliers adopt different features?")
print("="*100)

# Show distribution by size and managed status
print("\nSupplier Distribution by Size Bucket and Managed Status:")
print(f"{'Size Bucket':<15} {'Managed':<15} {'Not Managed':<15} {'Total':<15} {'% Managed':<15}")
print("-" * 75)

for bucket in SIZE_BUCKET_ORDER:
    bucket_suppliers = supplier_data[supplier_data["size_bucket"] == bucket]
    managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "yes"])
    not_managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "no"])
    total = managed + not_managed
    pct_managed = 100 * managed / total if total > 0 else 0
    
    print(f"{bucket:<15} {managed:<15,} {not_managed:<15,} {total:<15,} {pct_managed:<15.1f}%")

# =============================================================================
# CALCULATE ADOPTION BY SIZE BUCKET AND MANAGED STATUS
# =============================================================================
adoption_data = []

for bucket in SIZE_BUCKET_ORDER:
    bucket_data = df[df["size_bucket"] == bucket]
    
    managed = bucket_data[bucket_data["is_managed"] == "yes"]
    not_managed = bucket_data[bucket_data["is_managed"] == "no"]
    
    for status_data, status_label in [
        (managed, "Managed"),
        (not_managed, "Not Managed")
    ]:
        if len(status_data) == 0:
            continue
            
        row = {
            "size_bucket": bucket,
            "management_status": status_label,
            "suppliers": status_data["supplier_id"].nunique(),
            "tours": len(status_data),
            "gmv": status_data["gmv_l12mo"].sum(),
        }
        
        for feat_name, col in FEATURES:
            row[feat_name] = 100.0 * status_data[col].mean()
        
        adoption_data.append(row)

adoption_df = pd.DataFrame(adoption_data)

# =============================================================================
# CHART 1: GROUPED BAR CHART - Feature Adoption by Size Bucket
# Shows Managed vs Not Managed side-by-side for each size bucket
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

feature_names = [f[0] for f in FEATURES]

for idx, bucket in enumerate(SIZE_BUCKET_ORDER):
    ax = axes[idx]
    
    bucket_data = adoption_df[adoption_df["size_bucket"] == bucket]
    managed_data = bucket_data[bucket_data["management_status"] == "Managed"]
    not_managed_data = bucket_data[bucket_data["management_status"] == "Not Managed"]
    
    x = np.arange(len(feature_names))
    width = 0.35
    
    managed_values = [float(managed_data[feat].iloc[0]) if len(managed_data) > 0 else 0 for feat in feature_names]
    not_managed_values = [float(not_managed_data[feat].iloc[0]) if len(not_managed_data) > 0 else 0 for feat in feature_names]
    
    bars1 = ax.bar(x - width/2, managed_values, width, label='Managed', 
                   color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    bars2 = ax.bar(x + width/2, not_managed_values, width, label='Not Managed', 
                   color=GREY_LIGHT, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    
    # Add value labels on bars (only if >5%)
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 5:
                ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                       f'{height:.0f}%',
                       ha='center', va='bottom', fontsize=8, fontweight='bold', color=GREY_DARK)
    
    # Get supplier counts
    m_suppliers = int(managed_data["suppliers"].iloc[0]) if len(managed_data) > 0 else 0
    nm_suppliers = int(not_managed_data["suppliers"].iloc[0]) if len(not_managed_data) > 0 else 0
    
    # Get GMV range for this bucket
    bucket_suppliers = supplier_data[supplier_data["size_bucket"] == bucket]
    if len(bucket_suppliers) > 0:
        min_gmv = bucket_suppliers["supplier_gmv_l12mo"].min()
        max_gmv = bucket_suppliers["supplier_gmv_l12mo"].max()
        gmv_range = f"€{min_gmv:,.0f}-€{max_gmv:,.0f}"
    else:
        gmv_range = "N/A"
    
    ax.set_title(f'{bucket}\n{gmv_range}\n{m_suppliers} M | {nm_suppliers} NM', 
                fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if idx == 0:
        ax.set_ylabel('Adoption Rate (%)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10, loc='upper left')

plt.suptitle(
    'Managed vs Not Managed: Feature Adoption Across Supplier Size Buckets\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV\n'
    'Question: At the same supplier size, does managed support drive different adoption patterns?',
    fontsize=14,
    fontweight='bold',
    y=1.00
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Feature adoption by size bucket (tertile-based)")

# =============================================================================
# CHART 2: HEATMAP - Adoption Rates by Size x Management Status
# Rows = Features, Columns = Size buckets split by Managed/Not Managed
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 9))

# Build matrix: features (rows) x (size_bucket × management_status) (columns)
column_labels = []
matrix = []

for feat_name, _ in FEATURES:
    row = []
    for bucket in SIZE_BUCKET_ORDER:
        for status in ["Managed", "Not Managed"]:
            data = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                              (adoption_df["management_status"] == status)]
            
            if len(data) > 0:
                row.append(data[feat_name].iloc[0])
            else:
                row.append(0)
            
            # Build column labels (only once)
            if len(matrix) == 0:
                supplier_count = int(data["suppliers"].iloc[0]) if len(data) > 0 else 0
                column_labels.append(f"{bucket}\n{status}\n({supplier_count} suppliers)")
    
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Size × Management combinations
ax.set_xticks(np.arange(len(column_labels)))
ax.set_xticklabels(column_labels, rotation=45, ha='right', fontsize=10)

# Y-axis: Features
ax.set_yticks(np.arange(len(feature_names)))
ax.set_yticklabels(feature_names, fontsize=11, fontweight='bold')

# Title
ax.set_title(
    'Feature Adoption: Managed vs Not Managed Across Supplier Sizes\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV\n'
    'Question: Does managed support drive adoption when controlling for supplier size?',
    fontsize=13,
    fontweight='bold',
    pad=20
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=10, 
               fontweight="bold")

# Add vertical lines to separate size buckets
for i in range(1, len(SIZE_BUCKET_ORDER)):
    ax.axvline(i * 2 - 0.5, color=GREY_DARK, linewidth=3, alpha=0.7)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(column_labels)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Heatmap showing adoption across size buckets (tertiles) and management status")

# =============================================================================
# SUMMARY: WHERE DOES MANAGED SUPPORT MATTER?
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Features Where Managed Support Shows Different Adoption (by size bucket)")
print("="*100)

for bucket in SIZE_BUCKET_ORDER:
    bucket_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                  (adoption_df["management_status"] == "Managed")]
    bucket_not_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                      (adoption_df["management_status"] == "Not Managed")]
    
    if len(bucket_managed) == 0 or len(bucket_not_managed) == 0:
        continue
    
    print(f"\n{bucket}:")
    
    differences = []
    for feat_name, _ in FEATURES:
        m_val = float(bucket_managed[feat_name].iloc[0])
        nm_val = float(bucket_not_managed[feat_name].iloc[0])
        diff = m_val - nm_val
        
        if abs(diff) > 10:  # Only show meaningful differences
            differences.append((feat_name, m_val, nm_val, diff))
    
    if differences:
        differences = sorted(differences, key=lambda x: abs(x[3]), reverse=True)
        for feat, m_val, nm_val, diff in differences:
            direction = "higher" if diff > 0 else "lower"
            print(f"  {feat}: Managed {m_val:.0f}% vs Not Managed {nm_val:.0f}% ({diff:+.0f}pp {direction})")
    else:
        print(f"  No meaningful differences (all features <10pp gap)")

print("\n" + "="*100)
print("✓ Analysis complete: Managed vs Not Managed with tertile-based size buckets")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 2 REVISED: Managed vs Not Managed - ALL 11 FEATURES
# Small/Medium/Large based on 33rd and 67th percentiles
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# is_managed is "yes"/"no"
if "is_managed" in df.columns:
    df["is_managed"] = df["is_managed"].astype(str).str.lower().fillna("no")

# =============================================================================
# ALL 11 FEATURES (CONSISTENT WITH QUESTION 1)
# =============================================================================
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)
df["Individual pricing with add-ons"] = (df.get("has_individual_pricing", 0) & df.get("has_addons", 0)).astype(int)
df["Group pricing with multiple tiers"] = df.get("has_group_scales", 0)
df["Add-ons with multiple tiers"] = df.get("has_addon_scales", 0)
df["Any API pricing scales"] = df.get("has_api_scales", 0)

FEATURES = [
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Individual pricing with add-ons", "Individual pricing with add-ons"),
    ("Group pricing with multiple tiers", "Group pricing with multiple tiers"),
    ("Add-ons with multiple tiers", "Add-ons with multiple tiers"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("Any API pricing scales", "Any API pricing scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

print(f"\n✓ Using all {len(FEATURES)} features for analysis")

# =============================================================================
# CALCULATE SUPPLIER-LEVEL GMV AND MANAGED STATUS
# =============================================================================
supplier_data = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "gmv_l12mo": "sum",
          "is_managed": "first"
      })
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

# =============================================================================
# CREATE SIZE BUCKETS BASED ON TERTILES (33rd and 67th percentiles)
# =============================================================================

# Filter to suppliers with GMV > 0
active_suppliers = supplier_data[supplier_data["supplier_gmv_l12mo"] > 0].copy()

# Calculate tertiles
p33 = active_suppliers["supplier_gmv_l12mo"].quantile(0.33)
p67 = active_suppliers["supplier_gmv_l12mo"].quantile(0.67)

print("\n" + "="*100)
print("SIZE BUCKET DEFINITIONS (Based on GMV Tertiles)")
print("="*100)
print(f"Small:  €0 - €{p33:,.0f} (bottom 33%)")
print(f"Medium: €{p33:,.0f} - €{p67:,.0f} (middle 33%)")
print(f"Large:  €{p67:,.0f}+ (top 33%)")
print("="*100)

def assign_size_bucket_tertile(gmv):
    """Assign supplier to size bucket based on tertiles"""
    if gmv <= 0:
        return "No GMV"
    elif gmv <= p33:
        return "Small"
    elif gmv <= p67:
        return "Medium"
    else:
        return "Large"

supplier_data["size_bucket"] = supplier_data["supplier_gmv_l12mo"].apply(assign_size_bucket_tertile)

SIZE_BUCKET_ORDER = ["Small", "Medium", "Large"]

# Merge back to tour level
df = df.merge(
    supplier_data[["supplier_id", "supplier_gmv_l12mo", "size_bucket", "is_managed"]],
    on="supplier_id",
    how="left",
    suffixes=('', '_supplier')
)
df["is_managed"] = df["is_managed_supplier"].fillna("no")

print("\nQUESTION 2: MANAGED VS NOT MANAGED - CONTROLLING FOR SUPPLIER SIZE")
print("Question: At the same supplier size, do managed suppliers adopt different features?")
print("="*100)

# Show distribution by size and managed status
print("\nSupplier Distribution by Size Bucket and Managed Status:")
print(f"{'Size Bucket':<15} {'Managed':<15} {'Not Managed':<15} {'Total':<15} {'% Managed':<15}")
print("-" * 75)

for bucket in SIZE_BUCKET_ORDER:
    bucket_suppliers = supplier_data[supplier_data["size_bucket"] == bucket]
    managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "yes"])
    not_managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "no"])
    total = managed + not_managed
    pct_managed = 100 * managed / total if total > 0 else 0
    
    print(f"{bucket:<15} {managed:<15,} {not_managed:<15,} {total:<15,} {pct_managed:<15.1f}%")

# =============================================================================
# CALCULATE ADOPTION BY SIZE BUCKET AND MANAGED STATUS
# =============================================================================
adoption_data = []

for bucket in SIZE_BUCKET_ORDER:
    bucket_data = df[df["size_bucket"] == bucket]
    
    managed = bucket_data[bucket_data["is_managed"] == "yes"]
    not_managed = bucket_data[bucket_data["is_managed"] == "no"]
    
    for status_data, status_label in [
        (managed, "Managed"),
        (not_managed, "Not Managed")
    ]:
        if len(status_data) == 0:
            continue
            
        row = {
            "size_bucket": bucket,
            "management_status": status_label,
            "suppliers": status_data["supplier_id"].nunique(),
            "tours": len(status_data),
            "gmv": status_data["gmv_l12mo"].sum(),
        }
        
        for feat_name, col in FEATURES:
            row[feat_name] = 100.0 * status_data[col].mean()
        
        adoption_data.append(row)

adoption_df = pd.DataFrame(adoption_data)

# =============================================================================
# CHART 1: GROUPED BAR CHART - Feature Adoption by Size Bucket
# Shows Managed vs Not Managed side-by-side for each size bucket
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=True)

feature_names = [f[0] for f in FEATURES]

for idx, bucket in enumerate(SIZE_BUCKET_ORDER):
    ax = axes[idx]
    
    bucket_data = adoption_df[adoption_df["size_bucket"] == bucket]
    managed_data = bucket_data[bucket_data["management_status"] == "Managed"]
    not_managed_data = bucket_data[bucket_data["management_status"] == "Not Managed"]
    
    x = np.arange(len(feature_names))
    width = 0.35
    
    managed_values = [float(managed_data[feat].iloc[0]) if len(managed_data) > 0 else 0 for feat in feature_names]
    not_managed_values = [float(not_managed_data[feat].iloc[0]) if len(not_managed_data) > 0 else 0 for feat in feature_names]
    
    bars1 = ax.bar(x - width/2, managed_values, width, label='Managed', 
                   color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    bars2 = ax.bar(x + width/2, not_managed_values, width, label='Not Managed', 
                   color=GREY_LIGHT, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    
    # Add value labels on bars (only if >5%)
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 5:
                ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                       f'{height:.0f}%',
                       ha='center', va='bottom', fontsize=7, fontweight='bold', color=GREY_DARK)
    
    # Get supplier counts
    m_suppliers = int(managed_data["suppliers"].iloc[0]) if len(managed_data) > 0 else 0
    nm_suppliers = int(not_managed_data["suppliers"].iloc[0]) if len(not_managed_data) > 0 else 0
    
    # Get GMV range for this bucket
    bucket_suppliers = supplier_data[supplier_data["size_bucket"] == bucket]
    if len(bucket_suppliers) > 0:
        min_gmv = bucket_suppliers["supplier_gmv_l12mo"].min()
        max_gmv = bucket_suppliers["supplier_gmv_l12mo"].max()
        gmv_range = f"€{min_gmv:,.0f}-€{max_gmv:,.0f}"
    else:
        gmv_range = "N/A"
    
    ax.set_title(f'{bucket}\n{gmv_range}\n{m_suppliers} M | {nm_suppliers} NM', 
                fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if idx == 0:
        ax.set_ylabel('Adoption Rate (%)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10, loc='upper left')

plt.suptitle(
    'Managed vs Not Managed: Feature Adoption Across Supplier Size Buckets\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV\n'
    'Question: At the same supplier size, does managed support drive different adoption patterns?',
    fontsize=14,
    fontweight='bold',
    y=1.00
)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 1: Feature adoption by size bucket (all {len(FEATURES)} features)")

# =============================================================================
# CHART 2: HEATMAP - Adoption Rates by Size x Management Status
# Rows = Features, Columns = Size buckets split by Managed/Not Managed
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 12))

# Build matrix: features (rows) x (size_bucket × management_status) (columns)
column_labels = []
matrix = []

for feat_name, _ in FEATURES:
    row = []
    for bucket in SIZE_BUCKET_ORDER:
        for status in ["Managed", "Not Managed"]:
            data = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                              (adoption_df["management_status"] == status)]
            
            if len(data) > 0:
                row.append(data[feat_name].iloc[0])
            else:
                row.append(0)
            
            # Build column labels (only once)
            if len(matrix) == 0:
                supplier_count = int(data["suppliers"].iloc[0]) if len(data) > 0 else 0
                column_labels.append(f"{bucket}\n{status}\n({supplier_count} suppliers)")
    
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Size × Management combinations
ax.set_xticks(np.arange(len(column_labels)))
ax.set_xticklabels(column_labels, rotation=45, ha='right', fontsize=10)

# Y-axis: Features
ax.set_yticks(np.arange(len(feature_names)))
ax.set_yticklabels(feature_names, fontsize=10, fontweight='bold')

# Title
ax.set_title(
    'Feature Adoption: Managed vs Not Managed Across Supplier Sizes\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV\n'
    'Question: Does managed support drive adoption when controlling for supplier size?',
    fontsize=13,
    fontweight='bold',
    pad=20
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=9, 
               fontweight="bold")

# Add vertical lines to separate size buckets
for i in range(1, len(SIZE_BUCKET_ORDER)):
    ax.axvline(i * 2 - 0.5, color=GREY_DARK, linewidth=3, alpha=0.7)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(column_labels)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 2: Heatmap showing adoption across size buckets (all {len(FEATURES)} features)")

# =============================================================================
# SUMMARY: WHERE DOES MANAGED SUPPORT MATTER?
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Features Where Managed Support Shows Different Adoption (by size bucket)")
print("="*100)

for bucket in SIZE_BUCKET_ORDER:
    bucket_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                  (adoption_df["management_status"] == "Managed")]
    bucket_not_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                      (adoption_df["management_status"] == "Not Managed")]
    
    if len(bucket_managed) == 0 or len(bucket_not_managed) == 0:
        continue
    
    print(f"\n{bucket}:")
    
    differences = []
    for feat_name, _ in FEATURES:
        m_val = float(bucket_managed[feat_name].iloc[0])
        nm_val = float(bucket_not_managed[feat_name].iloc[0])
        diff = m_val - nm_val
        
        if abs(diff) > 10:  # Only show meaningful differences
            differences.append((feat_name, m_val, nm_val, diff))
    
    if differences:
        differences = sorted(differences, key=lambda x: abs(x[3]), reverse=True)
        for feat, m_val, nm_val, diff in differences:
            direction = "higher" if diff > 0 else "lower"
            print(f"  {feat}: Managed {m_val:.0f}% vs Not Managed {nm_val:.0f}% ({diff:+.0f}pp {direction})")
    else:
        print(f"  No meaningful differences (all features <10pp gap)")

print("\n" + "="*100)
print(f"✓ Analysis complete: Managed vs Not Managed with all {len(FEATURES)} features")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 2 REVISED: Managed vs Not Managed - ALL 11 FEATURES
# Small/Medium/Large based on 33rd and 67th percentiles
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# is_managed is "yes"/"no"
if "is_managed" in df.columns:
    df["is_managed"] = df["is_managed"].astype(str).str.lower().fillna("no")

# =============================================================================
# ALL 11 FEATURES (CONSISTENT WITH QUESTION 1)
# =============================================================================
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)
df["Individual pricing with add-ons"] = (df.get("has_individual_pricing", 0) & df.get("has_addons", 0)).astype(int)
df["Group pricing with multiple tiers"] = df.get("has_group_scales", 0)
df["Add-ons with multiple tiers"] = df.get("has_addon_scales", 0)

FEATURES = [
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Individual pricing with add-ons", "Individual pricing with add-ons"),
    ("Group pricing with multiple tiers", "Group pricing with multiple tiers"),
    ("Add-ons with multiple tiers", "Add-ons with multiple tiers"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("Any pricing scales", "has_scale_pricing"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

print(f"\n✓ Using all {len(FEATURES)} features for analysis")

# =============================================================================
# CALCULATE SUPPLIER-LEVEL GMV AND MANAGED STATUS
# =============================================================================
supplier_data = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "gmv_l12mo": "sum",
          "is_managed": "first"
      })
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

# =============================================================================
# CREATE SIZE BUCKETS BASED ON TERTILES (33rd and 67th percentiles)
# =============================================================================

# Filter to suppliers with GMV > 0
active_suppliers = supplier_data[supplier_data["supplier_gmv_l12mo"] > 0].copy()

# Calculate tertiles
p33 = active_suppliers["supplier_gmv_l12mo"].quantile(0.33)
p67 = active_suppliers["supplier_gmv_l12mo"].quantile(0.67)

print("\n" + "="*100)
print("SIZE BUCKET DEFINITIONS (Based on GMV Tertiles)")
print("="*100)
print(f"Small:  €0 - €{p33:,.0f} (bottom 33%)")
print(f"Medium: €{p33:,.0f} - €{p67:,.0f} (middle 33%)")
print(f"Large:  €{p67:,.0f}+ (top 33%)")
print("="*100)

def assign_size_bucket_tertile(gmv):
    """Assign supplier to size bucket based on tertiles"""
    if gmv <= 0:
        return "No GMV"
    elif gmv <= p33:
        return "Small"
    elif gmv <= p67:
        return "Medium"
    else:
        return "Large"

supplier_data["size_bucket"] = supplier_data["supplier_gmv_l12mo"].apply(assign_size_bucket_tertile)

SIZE_BUCKET_ORDER = ["Small", "Medium", "Large"]

# Merge back to tour level
df = df.merge(
    supplier_data[["supplier_id", "supplier_gmv_l12mo", "size_bucket", "is_managed"]],
    on="supplier_id",
    how="left",
    suffixes=('', '_supplier')
)
df["is_managed"] = df["is_managed_supplier"].fillna("no")

print("\nQUESTION 2: MANAGED VS NOT MANAGED - CONTROLLING FOR SUPPLIER SIZE")
print("Question: At the same supplier size, do managed suppliers adopt different features?")
print("="*100)

# Show distribution by size and managed status
print("\nSupplier Distribution by Size Bucket and Managed Status:")
print(f"{'Size Bucket':<15} {'Managed':<15} {'Not Managed':<15} {'Total':<15} {'% Managed':<15}")
print("-" * 75)

for bucket in SIZE_BUCKET_ORDER:
    bucket_suppliers = supplier_data[supplier_data["size_bucket"] == bucket]
    managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "yes"])
    not_managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "no"])
    total = managed + not_managed
    pct_managed = 100 * managed / total if total > 0 else 0
    
    print(f"{bucket:<15} {managed:<15,} {not_managed:<15,} {total:<15,} {pct_managed:<15.1f}%")

# =============================================================================
# CALCULATE ADOPTION BY SIZE BUCKET AND MANAGED STATUS
# =============================================================================
adoption_data = []

for bucket in SIZE_BUCKET_ORDER:
    bucket_data = df[df["size_bucket"] == bucket]
    
    managed = bucket_data[bucket_data["is_managed"] == "yes"]
    not_managed = bucket_data[bucket_data["is_managed"] == "no"]
    
    for status_data, status_label in [
        (managed, "Managed"),
        (not_managed, "Not Managed")
    ]:
        if len(status_data) == 0:
            continue
            
        row = {
            "size_bucket": bucket,
            "management_status": status_label,
            "suppliers": status_data["supplier_id"].nunique(),
            "tours": len(status_data),
            "gmv": status_data["gmv_l12mo"].sum(),
        }
        
        for feat_name, col in FEATURES:
            row[feat_name] = 100.0 * status_data[col].mean()
        
        adoption_data.append(row)

adoption_df = pd.DataFrame(adoption_data)

# =============================================================================
# CHART 1: GROUPED BAR CHART - Feature Adoption by Size Bucket
# Shows Managed vs Not Managed side-by-side for each size bucket
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=True)

feature_names = [f[0] for f in FEATURES]

for idx, bucket in enumerate(SIZE_BUCKET_ORDER):
    ax = axes[idx]
    
    bucket_data = adoption_df[adoption_df["size_bucket"] == bucket]
    managed_data = bucket_data[bucket_data["management_status"] == "Managed"]
    not_managed_data = bucket_data[bucket_data["management_status"] == "Not Managed"]
    
    x = np.arange(len(feature_names))
    width = 0.35
    
    managed_values = [float(managed_data[feat].iloc[0]) if len(managed_data) > 0 else 0 for feat in feature_names]
    not_managed_values = [float(not_managed_data[feat].iloc[0]) if len(not_managed_data) > 0 else 0 for feat in feature_names]
    
    bars1 = ax.bar(x - width/2, managed_values, width, label='Managed', 
                   color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    bars2 = ax.bar(x + width/2, not_managed_values, width, label='Not Managed', 
                   color=GREY_LIGHT, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    
    # Add value labels on bars (only if >5%)
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 5:
                ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                       f'{height:.0f}%',
                       ha='center', va='bottom', fontsize=7, fontweight='bold', color=GREY_DARK)
    
    # Get supplier counts
    m_suppliers = int(managed_data["suppliers"].iloc[0]) if len(managed_data) > 0 else 0
    nm_suppliers = int(not_managed_data["suppliers"].iloc[0]) if len(not_managed_data) > 0 else 0
    
    # Get GMV range for this bucket
    bucket_suppliers = supplier_data[supplier_data["size_bucket"] == bucket]
    if len(bucket_suppliers) > 0:
        min_gmv = bucket_suppliers["supplier_gmv_l12mo"].min()
        max_gmv = bucket_suppliers["supplier_gmv_l12mo"].max()
        gmv_range = f"€{min_gmv:,.0f}-€{max_gmv:,.0f}"
    else:
        gmv_range = "N/A"
    
    ax.set_title(f'{bucket}\n{gmv_range}\n{m_suppliers} M | {nm_suppliers} NM', 
                fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if idx == 0:
        ax.set_ylabel('Adoption Rate (%)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10, loc='upper left')

plt.suptitle(
    'Managed vs Not Managed: Feature Adoption Across Supplier Size Buckets\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV\n'
    'Question: At the same supplier size, does managed support drive different adoption patterns?',
    fontsize=14,
    fontweight='bold',
    y=1.00
)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 1: Feature adoption by size bucket (all {len(FEATURES)} features)")

# =============================================================================
# CHART 2: HEATMAP - Adoption Rates by Size x Management Status
# Rows = Features, Columns = Size buckets split by Managed/Not Managed
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 12))

# Build matrix: features (rows) x (size_bucket × management_status) (columns)
column_labels = []
matrix = []

for feat_name, _ in FEATURES:
    row = []
    for bucket in SIZE_BUCKET_ORDER:
        for status in ["Managed", "Not Managed"]:
            data = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                              (adoption_df["management_status"] == status)]
            
            if len(data) > 0:
                row.append(data[feat_name].iloc[0])
            else:
                row.append(0)
            
            # Build column labels (only once)
            if len(matrix) == 0:
                supplier_count = int(data["suppliers"].iloc[0]) if len(data) > 0 else 0
                column_labels.append(f"{bucket}\n{status}\n({supplier_count} suppliers)")
    
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Size × Management combinations
ax.set_xticks(np.arange(len(column_labels)))
ax.set_xticklabels(column_labels, rotation=45, ha='right', fontsize=10)

# Y-axis: Features
ax.set_yticks(np.arange(len(feature_names)))
ax.set_yticklabels(feature_names, fontsize=10, fontweight='bold')

# Title
ax.set_title(
    'Feature Adoption: Managed vs Not Managed Across Supplier Sizes\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV\n'
    'Question: Does managed support drive adoption when controlling for supplier size?',
    fontsize=13,
    fontweight='bold',
    pad=20
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=9, 
               fontweight="bold")

# Add vertical lines to separate size buckets
for i in range(1, len(SIZE_BUCKET_ORDER)):
    ax.axvline(i * 2 - 0.5, color=GREY_DARK, linewidth=3, alpha=0.7)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(column_labels)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 2: Heatmap showing adoption across size buckets (all {len(FEATURES)} features)")

# =============================================================================
# SUMMARY: WHERE DOES MANAGED SUPPORT MATTER?
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Features Where Managed Support Shows Different Adoption (by size bucket)")
print("="*100)

for bucket in SIZE_BUCKET_ORDER:
    bucket_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                  (adoption_df["management_status"] == "Managed")]
    bucket_not_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                      (adoption_df["management_status"] == "Not Managed")]
    
    if len(bucket_managed) == 0 or len(bucket_not_managed) == 0:
        continue
    
    print(f"\n{bucket}:")
    
    differences = []
    for feat_name, _ in FEATURES:
        m_val = float(bucket_managed[feat_name].iloc[0])
        nm_val = float(bucket_not_managed[feat_name].iloc[0])
        diff = m_val - nm_val
        
        if abs(diff) > 10:  # Only show meaningful differences
            differences.append((feat_name, m_val, nm_val, diff))
    
    if differences:
        differences = sorted(differences, key=lambda x: abs(x[3]), reverse=True)
        for feat, m_val, nm_val, diff in differences:
            direction = "higher" if diff > 0 else "lower"
            print(f"  {feat}: Managed {m_val:.0f}% vs Not Managed {nm_val:.0f}% ({diff:+.0f}pp {direction})")
    else:
        print(f"  No meaningful differences (all features <10pp gap)")

print("\n" + "="*100)
print(f"✓ Analysis complete: Managed vs Not Managed with all {len(FEATURES)} features")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 3: Feature Combinations Among Top 80% GMV Suppliers
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# THE 11 FEATURES
# =============================================================================
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)
df["Individual pricing with add-ons"] = (df.get("has_individual_pricing", 0) & df.get("has_addons", 0)).astype(int)
df["Group pricing with multiple tiers"] = df.get("has_group_scales", 0)
df["Add-ons with multiple tiers"] = df.get("has_addon_scales", 0)

FEATURES = [
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Multiple tour options", "Multiple tour options"),
    ("Individual tiers", "has_individual_scales"),
    ("Individual + Add-ons", "Individual pricing with add-ons"),
    ("Group tiers", "Group pricing with multiple tiers"),
    ("Add-on tiers", "Add-ons with multiple tiers"),
    ("Any scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API scales", "has_api_scales"),  # ← ADD THIS
    ("Live pricing", "has_live_dynamic_pricing"),
]

# Aggregate to supplier level
feature_cols = [col for _, col in FEATURES]

supplier_data = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg(dict([(col, "max") for col in feature_cols] + [("gmv_l12mo", "sum")]))
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

print("\n" + "="*100)
print("QUESTION 3: FEATURE COMBINATIONS AMONG TOP 80% GMV SUPPLIERS")
print("="*100)

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS (CUMULATIVE)
# =============================================================================

active_suppliers = supplier_data[supplier_data["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            # Sort by GMV descending
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            
            # Calculate cumulative GMV percentage
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            
            # Mark top 80%
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)

print("\nTop 80% GMV Suppliers by Segment:")
print(f"{'Segment':<20} {'Total Suppliers':<20} {'Top 80% GMV':<20} {'%':<15}")
print("-" * 75)

for segment in sorted(active_suppliers["supplier_segment"].dropna().unique()):
    seg_data = active_suppliers[active_suppliers["supplier_segment"] == segment]
    total = len(seg_data)
    top_80 = seg_data["is_top_80"].sum()
    pct = 100 * top_80 / total
    
    print(f"{segment:<20} {total:<20,} {top_80:<20,} {pct:<15.1f}%")

# =============================================================================
# CREATE COMBINATION STRINGS
# =============================================================================

def create_combo_string(row):
    active = []
    for feat_name, feat_col in FEATURES:
        if row[feat_col] == 1:
            active.append(feat_name)
    
    if len(active) == 0:
        return "No features"
    
    return " + ".join(active)

active_suppliers["combo"] = active_suppliers.apply(create_combo_string, axis=1)

# =============================================================================
# TOP 10 COMBINATIONS PER SEGMENT (TOP 80% ONLY)
# =============================================================================

print("\n" + "="*100)
print("TOP 10 FEATURE COMBINATIONS - TOP 80% GMV SUPPLIERS PER SEGMENT")
print("="*100)

for segment in sorted(active_suppliers["supplier_segment"].dropna().unique()):
    top_80 = active_suppliers[(active_suppliers["supplier_segment"] == segment) & 
                               (active_suppliers["is_top_80"] == 1)]
    
    print(f"\n{'='*100}")
    print(f"{segment} (n={len(top_80):,} suppliers generating 80% of GMV)")
    print(f"{'='*100}")
    
    top_combos = top_80["combo"].value_counts().head(10)
    
    print(f"{'Rank':<6} {'Count':<10} {'%':<10} {'Combination'}")
    print("-" * 120)
    
    for rank, (combo, count) in enumerate(top_combos.items(), 1):
        pct = 100 * count / len(top_80)
        print(f"{rank:<6} {count:<10} {pct:<10.1f} {combo}")

# =============================================================================
# CHART: TOP 10 COMBINATIONS PER SEGMENT
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(28, 16))
axes = axes.flatten()

segments = sorted(active_suppliers["supplier_segment"].dropna().unique())

for idx, segment in enumerate(segments):
    if idx >= len(axes):
        break
    
    ax = axes[idx]
    
    top_80 = active_suppliers[(active_suppliers["supplier_segment"] == segment) & 
                               (active_suppliers["is_top_80"] == 1)]
    
    # Get top 10 combos
    top_combos = top_80["combo"].value_counts().head(10)
    
    values = []
    labels = []
    
    for combo, count in top_combos.items():
        pct = 100 * count / len(top_80)
        values.append(pct)
        labels.append(combo)
    
    y = np.arange(len(labels))
    
    bars = ax.barh(y, values, color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.5, bar.get_y() + bar.get_height()/2.,
               f'{width:.1f}%',
               ha='left', va='center', fontsize=9, fontweight='bold', color=GREY_DARK)
    
    ax.set_title(f'{segment}\n{len(top_80):,} suppliers (top 80% GMV)', 
                fontsize=12, fontweight='bold')
    ax.set_xlabel('% of Top 80% Suppliers', fontsize=11)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.set_xlim(0, max(values) * 1.15)

for idx in range(len(segments), len(axes)):
    axes[idx].axis('off')

plt.suptitle(
    'Most Common Feature Combinations Among Top Performers\n'
    'Top 80% GMV-generating suppliers per segment',
    fontsize=16,
    fontweight='bold',
    y=0.995
)

plt.tight_layout()
plt.show()

print("\n✓ Analysis complete")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 1: TOP 80% GMV GENERATING SUPPLIERS - SUPPLIER-LEVEL ADOPTION
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# THE 11 FEATURES
# =============================================================================
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)
df["Individual pricing with add-ons"] = (df.get("has_individual_pricing", 0) & df.get("has_addons", 0)).astype(int)
df["Group pricing with multiple tiers"] = df.get("has_group_scales", 0)
df["Add-ons with multiple tiers"] = df.get("has_addon_scales", 0)

FEATURES = [
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Individual pricing with add-ons", "Individual pricing with add-ons"),
    ("Group pricing with multiple tiers", "Group pricing with multiple tiers"),
    ("Add-ons with multiple tiers", "Add-ons with multiple tiers"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

# =============================================================================
# AGGREGATE TO SUPPLIER LEVEL FIRST
# =============================================================================
feature_cols = [col for _, col in FEATURES]

supplier_data = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg(dict([(col, "max") for col in feature_cols] + [("gmv_l12mo", "sum")]))
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

print("\n" + "="*100)
print("QUESTION 1: FEATURE ADOPTION AMONG TOP 80% GMV SUPPLIERS")
print("="*100)

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS (CUMULATIVE)
# =============================================================================

active_suppliers = supplier_data[supplier_data["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            # Sort by GMV descending
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            
            # Calculate cumulative GMV percentage
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            
            # Mark top 80%
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)

print("\nTop 80% GMV Suppliers by Segment:")
print(f"{'Segment':<20} {'Total Suppliers':<20} {'Top 80% GMV':<20} {'% of Suppliers':<20} {'GMV':<20}")
print("-" * 100)

for segment in sorted(active_suppliers["supplier_segment"].dropna().unique()):
    seg_data = active_suppliers[active_suppliers["supplier_segment"] == segment]
    total = len(seg_data)
    top_80 = seg_data["is_top_80"].sum()
    pct = 100 * top_80 / total
    gmv = seg_data[seg_data["is_top_80"] == 1]["supplier_gmv_l12mo"].sum() / 1_000_000
    
    print(f"{segment:<20} {total:<20,} {top_80:<20,} {pct:<20.0f}% €{gmv:>18,.1f}M")

# =============================================================================
# CALCULATE ADOPTION RATES (SUPPLIER-LEVEL)
# =============================================================================

segments = sorted(active_suppliers["supplier_segment"].dropna().unique())

top_80_adoption = []
for segment in segments:
    seg_top = active_suppliers[(active_suppliers["supplier_segment"] == segment) & 
                                (active_suppliers["is_top_80"] == 1)]
    
    if len(seg_top) == 0:
        continue
    
    row = {
        "segment": segment,
        "suppliers": len(seg_top),
        "total_gmv": seg_top["supplier_gmv_l12mo"].sum(),
    }
    
    # Calculate % of SUPPLIERS (not tours) that have each feature
    for feat_name, col in FEATURES:
        row[feat_name] = 100.0 * seg_top[col].mean()
    
    top_80_adoption.append(row)

top_80_df = pd.DataFrame(top_80_adoption)
top_80_df = top_80_df.sort_values("total_gmv", ascending=False)

print("\n" + "="*100)
print("FEATURE ADOPTION RATES (% OF SUPPLIERS)")
print("="*100)

for _, row in top_80_df.iterrows():
    print(f"\n{row['segment']} ({int(row['suppliers'])} suppliers | €{row['total_gmv']/1e6:.1f}M):")
    print(f"{'Feature':<50} {'Adoption %':<15}")
    print("-" * 65)
    
    feature_adoptions = [(feat_name, row[feat_name]) for feat_name, _ in FEATURES]
    feature_adoptions = sorted(feature_adoptions, key=lambda x: x[1], reverse=True)
    
    for feat, adoption in feature_adoptions:
        print(f"{feat:<50} {adoption:>12.1f}%")

# =============================================================================
# CHART 1: HEATMAP BY SEGMENT
# =============================================================================

fig, ax = plt.subplots(figsize=(16, 10))

feature_names = [f[0] for f in FEATURES]
segment_list = top_80_df["segment"].tolist()

matrix = []
for segment in segment_list:
    row = []
    seg_data = top_80_df[top_80_df["segment"] == segment]
    for feat_name, _ in FEATURES:
        row.append(seg_data[feat_name].iloc[0])
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Features
ax.set_xticks(np.arange(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=11, fontweight="bold")

# Y-axis: Segments
segment_labels = []
for segment in segment_list:
    seg_data = top_80_df[top_80_df["segment"] == segment]
    suppliers = int(seg_data["suppliers"].iloc[0])
    gmv_m = seg_data["total_gmv"].iloc[0] / 1_000_000
    
    # Calculate what % of suppliers this represents
    total_in_segment = active_suppliers[active_suppliers["supplier_segment"] == segment].shape[0]
    pct_suppliers = 100 * suppliers / total_in_segment if total_in_segment > 0 else 0
    
    segment_labels.append(f"{segment}\n({suppliers} suppliers = {pct_suppliers:.0f}% | €{gmv_m:.1f}M = 80% GMV)")

ax.set_yticks(np.arange(len(segment_list)))
ax.set_yticklabels(segment_labels, fontsize=10)

# Title
ax.set_title(
    "What Do Revenue-Driving Suppliers Actually Use?\n"
    "Feature Adoption Rates for Top 80% GMV-Generating Suppliers by Segment\n"
    "Question: Which pricing features are most adopted by successful suppliers?",
    fontsize=15,
    fontweight="bold",
    pad=25
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=10, 
               fontweight="bold")

# Highlight Leisure Brand API pricing
api_pricing_idx = [i for i, (name, _) in enumerate(FEATURES) if name == "API pricing"][0]
leisure_brand_idx = segment_list.index("Leisure Brand") if "Leisure Brand" in segment_list else None

if leisure_brand_idx is not None:
    rect = patches.Rectangle(
        (api_pricing_idx - 0.48, leisure_brand_idx - 0.48),
        0.96, 0.96,
        linewidth=3,
        edgecolor=ORANGE_DARK,
        facecolor='none',
        linestyle='--'
    )
    ax.add_patch(rect)
    
    leisure_api_adoption = matrix[leisure_brand_idx, api_pricing_idx]
    ax.annotate(
        f'Leisure Brand: {leisure_api_adoption:.0f}% API pricing\n(highest across all segments)',
        xy=(api_pricing_idx, leisure_brand_idx),
        xytext=(api_pricing_idx + 2.5, leisure_brand_idx - 1.5),
        fontsize=10,
        fontweight='bold',
        color=ORANGE_DARK,
        bbox=dict(boxstyle="round,pad=0.5", facecolor=ORANGE_PALE, edgecolor=ORANGE_DARK, linewidth=2, alpha=0.9),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', color=ORANGE_DARK, lw=2.5)
    )

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Adoption Rate Among Revenue Drivers (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segment_list)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Feature adoption by segment (supplier-level)")

# =============================================================================
# CHART 2: OVERALL FEATURE RANKING
# =============================================================================

total_gmv = top_80_df["total_gmv"].sum()
feature_rankings = []

for feat_name, _ in FEATURES:
    weighted_avg = 0
    for _, row in top_80_df.iterrows():
        weight = row["total_gmv"] / total_gmv
        weighted_avg += row[feat_name] * weight
    
    min_val = top_80_df[feat_name].min()
    max_val = top_80_df[feat_name].max()
    
    feature_rankings.append({
        "feature": feat_name,
        "avg_adoption": weighted_avg,
        "min": min_val,
        "max": max_val,
        "range": max_val - min_val
    })

ranking_df = pd.DataFrame(feature_rankings).sort_values("avg_adoption", ascending=True)

fig, ax = plt.subplots(figsize=(14, 8))

y_pos = np.arange(len(ranking_df))

def get_bar_color(adoption_rate):
    if adoption_rate >= 80:
        return ORANGE_DARK
    elif adoption_rate >= 50:
        return ORANGE
    elif adoption_rate >= 20:
        return ORANGE_LIGHT
    else:
        return ORANGE_PALE

bar_colors = [get_bar_color(row["avg_adoption"]) for _, row in ranking_df.iterrows()]
bars = ax.barh(y_pos, ranking_df["avg_adoption"], 
               color=bar_colors, edgecolor=GREY_DARK, linewidth=1.5, alpha=0.9)

# Add range indicators
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], 
            color=GREY_DARK, linewidth=3, alpha=0.6, zorder=1)
    ax.scatter([row["min"], row["max"]], [i, i], 
               color=GREY_DARK, s=50, zorder=2, alpha=0.8)

# Value labels
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.text(row["avg_adoption"] + 2, i, f"{row['avg_adoption']:.0f}%", 
           va="center", ha="left", fontsize=10, fontweight="bold", color=GREY_DARK)

ax.set_yticks(y_pos)
ax.set_yticklabels(ranking_df["feature"], fontsize=11)
ax.set_xlabel("Adoption Rate Among Revenue Drivers (%)", fontsize=12, fontweight="bold")
ax.set_title(
    "Feature Adoption Ranking: What Revenue-Driving Suppliers Use Most\n"
    "GMV-weighted average across all segments (Top 80% GMV suppliers only)\n"
    "Bar = weighted average | Line = range across segments | Color = adoption level",
    fontsize=14,
    fontweight="bold",
    pad=20
)
ax.set_xlim(0, 110)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=ORANGE_DARK, edgecolor=GREY_DARK, label='High (≥80%)'),
    Patch(facecolor=ORANGE, edgecolor=GREY_DARK, label='Medium-high (50-79%)'),
    Patch(facecolor=ORANGE_LIGHT, edgecolor=GREY_DARK, label='Medium (20-49%)'),
    Patch(facecolor=ORANGE_PALE, edgecolor=GREY_DARK, label='Low (<20%)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10, framealpha=0.95)

# Summary box
top_3 = ranking_df.nlargest(3, "avg_adoption")["feature"].tolist()
summary = "Most Adopted by Revenue Drivers:\n" + "\n".join([f"{i+1}. {f}" for i, f in enumerate(top_3)])
ax.text(
    0.98, 0.28, summary,
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.4, edgecolor=ORANGE_DARK, linewidth=2)
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Overall feature ranking (supplier-level)")
print("\n" + "="*100)
print("✓ Analysis complete: Question 1 (supplier-level adoption)")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 6: CONFIGURATION DEPTH - AVERAGE PER TOUR PER SUPPLIER
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons", "num_individual_tiers", "num_group_tiers"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

print("\n" + "="*100)
print("QUESTION 6: CONFIGURATION DEPTH - AVERAGE PER TOUR")
print("="*100)
print("How deeply do suppliers configure features on their tours?")
print("="*100)

# =============================================================================
# CALCULATE SUPPLIER-LEVEL AVERAGES (MEAN ACROSS TOURS)
# =============================================================================

supplier_stats = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "gmv_l12mo": "sum",
          "total_options": "mean",      # Average options per tour
          "num_addons": "mean",          # Average add-ons per tour
          "num_individual_tiers": "mean", # Average individual tiers per tour
          "num_group_tiers": "mean",     # Average group tiers per tour
          "tour_id": "count"             # Number of tours
      })
      .reset_index()
      .rename(columns={
          "gmv_l12mo": "supplier_gmv_l12mo",
          "total_options": "avg_options_per_tour",
          "num_addons": "avg_addons_per_tour",
          "num_individual_tiers": "avg_individual_tiers_per_tour",
          "num_group_tiers": "avg_group_tiers_per_tour",
          "tour_id": "num_tours"
      })
)

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS
# =============================================================================

active_suppliers = supplier_stats[supplier_stats["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)
top_80_suppliers = active_suppliers[active_suppliers["is_top_80"] == 1].copy()

print(f"\nAnalyzing {len(top_80_suppliers):,} top 80% GMV suppliers")

# =============================================================================
# DEFINE SUPPLIER SIZE BUCKETS
# =============================================================================

def classify_size(gmv):
    if gmv < 50000:
        return "Small"
    elif gmv < 500000:
        return "Medium"
    else:
        return "Large"

top_80_suppliers["size_bucket"] = top_80_suppliers["supplier_gmv_l12mo"].apply(classify_size)

print("\nSupplier Size Distribution:")
size_dist = top_80_suppliers.groupby("size_bucket").agg({
    "supplier_id": "count",
    "num_tours": "sum"
}).rename(columns={"supplier_id": "suppliers"})
print(size_dist)

# =============================================================================
# DEPTH CLASSIFICATION BY AVERAGE PER TOUR
# =============================================================================

def classify_options_depth(avg):
    if avg <= 3:
        return "Shallow (≤3)"
    elif avg <= 7:
        return "Medium (4-7)"
    else:
        return "Deep (8+)"

def classify_addon_depth(avg):
    if avg <= 2:
        return "Shallow (≤2)"
    elif avg <= 5:
        return "Medium (3-5)"
    else:
        return "Deep (6+)"

def classify_tier_depth(avg):
    if avg <= 3:
        return "Shallow (≤3)"
    elif avg <= 5:
        return "Medium (4-5)"
    else:
        return "Deep (6+)"

# Apply classifications
top_80_suppliers["options_depth"] = top_80_suppliers["avg_options_per_tour"].apply(classify_options_depth)
top_80_suppliers["addons_depth"] = top_80_suppliers["avg_addons_per_tour"].apply(classify_addon_depth)
top_80_suppliers["ind_tiers_depth"] = top_80_suppliers["avg_individual_tiers_per_tour"].apply(classify_tier_depth)
top_80_suppliers["grp_tiers_depth"] = top_80_suppliers["avg_group_tiers_per_tour"].apply(classify_tier_depth)

# =============================================================================
# SUMMARY STATISTICS
# =============================================================================

print("\n" + "="*100)
print("AVERAGE CONFIGURATION DEPTH PER TOUR (BY SUPPLIER SIZE)")
print("="*100)

metrics = [
    ("Tour Options", "avg_options_per_tour", "options_depth"),
    ("Add-ons", "avg_addons_per_tour", "addons_depth"),
    ("Individual Tiers", "avg_individual_tiers_per_tour", "ind_tiers_depth"),
    ("Group Tiers", "avg_group_tiers_per_tour", "grp_tiers_depth")
]

for metric_name, col, depth_col in metrics:
    print(f"\n{metric_name}:")
    for size in ["Small", "Medium", "Large"]:
        size_data = top_80_suppliers[top_80_suppliers["size_bucket"] == size]
        avg = size_data[col].mean()
        median = size_data[col].median()
        shallow_pct = 100 * (size_data[depth_col].str.contains("Shallow").sum() / len(size_data))
        print(f"  {size}: Mean={avg:.2f}, Median={median:.2f}, {shallow_pct:.0f}% shallow")

# =============================================================================
# CHART 1: DISTRIBUTION BY SIZE
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (metric_name, col, depth_col) in enumerate(metrics):
    ax = axes[idx]
    
    sizes = ["Small", "Medium", "Large"]
    shallow_vals = []
    medium_vals = []
    deep_vals = []
    
    for size in sizes:
        size_data = top_80_suppliers[top_80_suppliers["size_bucket"] == size]
        total = len(size_data)
        
        shallow = 100 * (size_data[depth_col].str.contains("Shallow").sum() / total)
        medium = 100 * (size_data[depth_col].str.contains("Medium").sum() / total)
        deep = 100 * (size_data[depth_col].str.contains("Deep").sum() / total)
        
        shallow_vals.append(shallow)
        medium_vals.append(medium)
        deep_vals.append(deep)
    
    x = np.arange(len(sizes))
    width = 0.5
    
    p1 = ax.bar(x, shallow_vals, width, label='Shallow', color=ORANGE_PALE, edgecolor=GREY_DARK, linewidth=1.5)
    p2 = ax.bar(x, medium_vals, width, bottom=shallow_vals, label='Medium', color=ORANGE, edgecolor=GREY_DARK, linewidth=1.5)
    p3 = ax.bar(x, deep_vals, width, bottom=np.array(shallow_vals)+np.array(medium_vals), 
                label='Deep', color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.5)
    
    # Add average values as text
    for i, size in enumerate(sizes):
        size_data = top_80_suppliers[top_80_suppliers["size_bucket"] == size]
        avg = size_data[col].mean()
        ax.text(i, 105, f'Avg: {avg:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.set_title(f'{metric_name} per Tour', fontsize=13, fontweight='bold')
    ax.set_ylabel('% of Suppliers', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(sizes, fontsize=11)
    ax.set_ylim(0, 120)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.legend(loc='upper right', fontsize=9)

plt.suptitle(
    'Configuration Depth: Average Items Per Tour\n'
    'Among top 80% GMV suppliers by size',
    fontsize=16,
    fontweight='bold',
    y=0.995
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Depth distribution")

# =============================================================================
# CHART 2: AVERAGE VALUES HEATMAP
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 6))

sizes = ["Small", "Medium", "Large"]
metric_names = [m[0] for m in metrics]
metric_cols = [m[1] for m in metrics]

avg_matrix = []
for size in sizes:
    row = []
    for col in metric_cols:
        size_data = top_80_suppliers[top_80_suppliers["size_bucket"] == size]
        avg = size_data[col].mean()
        row.append(avg)
    avg_matrix.append(row)

avg_matrix = np.array(avg_matrix)

from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

im = ax.imshow(avg_matrix, aspect="auto", cmap=cmap)

ax.set_xticks(np.arange(len(metric_names)))
ax.set_xticklabels(metric_names, fontsize=12, fontweight='bold')
ax.set_yticks(np.arange(len(sizes)))
ax.set_yticklabels(sizes, fontsize=12)

ax.set_title(
    'Average Configuration Depth per Tour by Supplier Size\n'
    '(mean across all tours per supplier, then averaged by size)',
    fontsize=14,
    fontweight='bold',
    pad=20
)

for i in range(len(sizes)):
    for j in range(len(metric_names)):
        val = avg_matrix[i, j]
        text_color = "white" if val > avg_matrix.max() * 0.5 else GREY_DARK
        ax.text(j, i, f"{val:.1f}", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=14, 
               fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Average per Tour", fontsize=12, fontweight="bold")

ax.set_xticks(np.arange(len(metric_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(sizes)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=3)
ax.tick_params()


In [0]:
# =============================================================================
# QUESTION 6: CONFIGURATION DEPTH - AVERAGE PER TOUR
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons", "num_individual_tiers", "num_group_tiers"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

print("\n" + "="*100)
print("QUESTION 6: CONFIGURATION DEPTH - HOW DEEPLY DO SUPPLIERS CONFIGURE THEIR TOURS?")
print("="*100)
print("Metric: Average items per tour (calculated per supplier, then averaged by segment)")
print("="*100)

# =============================================================================
# CALCULATE SUPPLIER-LEVEL AVERAGES (MEAN ACROSS TOURS)
# =============================================================================

supplier_stats = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "gmv_l12mo": "sum",
          "total_options": "mean",
          "num_addons": "mean",
          "num_individual_tiers": "mean",
          "num_group_tiers": "mean",
          "tour_id": "count"
      })
      .reset_index()
      .rename(columns={
          "gmv_l12mo": "supplier_gmv_l12mo",
          "total_options": "avg_options_per_tour",
          "num_addons": "avg_addons_per_tour",
          "num_individual_tiers": "avg_individual_tiers_per_tour",
          "num_group_tiers": "avg_group_tiers_per_tour",
          "tour_id": "num_tours"
      })
)

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS
# =============================================================================

active_suppliers = supplier_stats[supplier_stats["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)
top_80_suppliers = active_suppliers[active_suppliers["is_top_80"] == 1].copy()

print(f"\nAnalyzing {len(top_80_suppliers):,} top 80% GMV suppliers")

# =============================================================================
# SUMMARY STATISTICS BY SEGMENT
# =============================================================================

print("\n" + "="*100)
print("AVERAGE CONFIGURATION DEPTH PER TOUR (BY SEGMENT)")
print("="*100)

segments = sorted(top_80_suppliers["supplier_segment"].dropna().unique())

metrics = [
    ("Tour Options", "avg_options_per_tour", "Pricing/booking options per tour"),
    ("Add-ons", "avg_addons_per_tour", "Upsell items per tour"),
    ("Individual Price Tiers", "avg_individual_tiers_per_tour", "Individual price points per tour"),
    ("Group Price Tiers", "avg_group_tiers_per_tour", "Group price points per tour")
]

depth_summary = []

for metric_name, col, description in metrics:
    print(f"\n{metric_name} ({description}):")
    for segment in segments:
        seg_data = top_80_suppliers[top_80_suppliers["supplier_segment"] == segment]
        if len(seg_data) > 0:
            avg = seg_data[col].mean()
            median = seg_data[col].median()
            p25 = seg_data[col].quantile(0.25)
            p75 = seg_data[col].quantile(0.75)
            print(f"  {segment}: Mean={avg:.2f}, Median={median:.2f} (P25={p25:.2f}, P75={p75:.2f})")
            
            depth_summary.append({
                "metric": metric_name,
                "segment": segment,
                "mean": avg,
                "median": median,
                "p25": p25,
                "p75": p75
            })

depth_df = pd.DataFrame(depth_summary)

# =============================================================================
# CHART 1: HEATMAP - AVERAGE DEPTH BY SEGMENT
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 8))

metric_names = [m[0] for m in metrics]
metric_cols = [m[1] for m in metrics]

avg_matrix = []
for segment in segments:
    row = []
    for col in metric_cols:
        seg_data = top_80_suppliers[top_80_suppliers["supplier_segment"] == segment]
        if len(seg_data) > 0:
            avg = seg_data[col].mean()
            row.append(avg)
        else:
            row.append(0)
    avg_matrix.append(row)

avg_matrix = np.array(avg_matrix)

cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

im = ax.imshow(avg_matrix, aspect="auto", cmap=cmap)

ax.set_xticks(np.arange(len(metric_names)))
ax.set_xticklabels(metric_names, fontsize=12, fontweight='bold')
ax.set_yticks(np.arange(len(segments)))
ax.set_yticklabels(segments, fontsize=12)

ax.set_title(
    'Configuration Depth: How Many Items Per Tour?\n'
    'Average across top 80% GMV suppliers by segment',
    fontsize=15,
    fontweight='bold',
    pad=20
)

# Annotate cells with values
for i in range(len(segments)):
    for j in range(len(metric_names)):
        val = avg_matrix[i, j]
        if val > 0:
            text_color = "white" if val > avg_matrix.max() * 0.5 else GREY_DARK
            ax.text(j, i, f"{val:.1f}", 
                   ha="center", va="center", 
                   color=text_color, 
                   fontsize=13, 
                   fontweight="bold")

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Average per Tour", fontsize=12, fontweight="bold")

ax.set_xticks(np.arange(len(metric_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segments)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=3)
ax.tick_params(which="minor", size=0)

# Add explanations
explanation_text = (
    "Reading: Scale Seeker suppliers use an average of X.X tour options per tour\n"
    "Calculation: For each supplier, average items per tour; then average across suppliers in segment"
)
ax.text(0.5, -0.15, explanation_text, 
        transform=ax.transAxes,
        fontsize=10,
        ha='center',
        bbox=dict(boxstyle="round,pad=0.8", facecolor=GREY_PALE, alpha=0.8))

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Configuration depth heatmap")

# =============================================================================
# CHART 2: DISTRIBUTION CURVES BY SEGMENT (TOUR OPTIONS ONLY)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (metric_name, col, description) in enumerate(metrics):
    ax = axes[idx]
    
    for segment in segments:
        seg_data = top_80_suppliers[top_80_suppliers["supplier_segment"] == segment]
        if len(seg_data) > 0:
            values = seg_data[col].values
            # Create histogram/distribution
            counts, bins = np.histogram(values, bins=20)
            # Normalize to percentage
            counts_pct = 100 * counts / counts.sum()
            bin_centers = (bins[:-1] + bins[1:]) / 2
            
            ax.plot(bin_centers, counts_pct, marker='o', linewidth=2, label=segment, alpha=0.7)
    
    ax.set_title(f'{metric_name} Distribution\n{description}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Average per Tour', fontsize=11)
    ax.set_ylabel('% of Suppliers', fontsize=11)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3, linestyle='--')

plt.suptitle(
    'Configuration Depth Distributions by Segment\n'
    'How many items do suppliers average per tour?',
    fontsize=15,
    fontweight='bold',
    y=0.995
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Distribution curves")

# =============================================================================
# KEY INSIGHTS
# =============================================================================

print("\n" + "="*100)
print("KEY INSIGHTS")
print("="*100)

for metric_name, col, description in metrics:
    print(f"\n{metric_name}:")
    
    # Find highest and lowest segments
    segment_avgs = []
    for segment in segments:
        seg_data = top_80_suppliers[top_80_suppliers["supplier_segment"] == segment]
        if len(seg_data) > 0:
            avg = seg_data[col].mean()
            segment_avgs.append((segment, avg))
    
    segment_avgs = sorted(segment_avgs, key=lambda x: x[1], reverse=True)
    
    if len(segment_avgs) >= 2:
        highest = segment_avgs[0]
        lowest = segment_avgs[-1]
        print(f"  Highest: {highest[0]} ({highest[1]:.2f} per tour)")
        print(f"  Lowest: {lowest[0]} ({lowest[1]:.2f} per tour)")
        print(f"  Range: {highest[1] - lowest[1]:.2f} difference")

print("\n" + "="*100)
print("✓ Analysis complete: Configuration depth")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 6: CONFIGURATION DEPTH - ARE ADOPTERS USING FEATURES DEEPLY?
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons", "num_individual_tiers", "num_group_tiers"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

print("\n" + "="*100)
print("QUESTION 6: AMONG ADOPTERS, HOW DEEPLY ARE FEATURES CONFIGURED?")
print("="*100)
print("Analysis: Calculate average items per tour for suppliers who actually use each feature")
print("="*100)

# =============================================================================
# AGGREGATE TO SUPPLIER LEVEL
# =============================================================================

supplier_stats = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "gmv_l12mo": "sum",
          "total_options": "mean",
          "num_addons": "mean",
          "num_individual_tiers": "mean",
          "num_group_tiers": "mean",
          "tour_id": "count"
      })
      .reset_index()
      .rename(columns={
          "gmv_l12mo": "supplier_gmv_l12mo",
          "total_options": "avg_options_per_tour",
          "num_addons": "avg_addons_per_tour",
          "num_individual_tiers": "avg_ind_tiers_per_tour",
          "num_group_tiers": "avg_grp_tiers_per_tour",
          "tour_id": "num_tours"
      })
)

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS
# =============================================================================

active_suppliers = supplier_stats[supplier_stats["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)
top_80 = active_suppliers[active_suppliers["is_top_80"] == 1].copy()

print(f"\nAnalyzing {len(top_80):,} top 80% GMV suppliers")

# =============================================================================
# CALCULATE DEPTH AMONG ADOPTERS ONLY
# =============================================================================

segments = sorted(top_80["supplier_segment"].dropna().unique())

metrics = [
    ("Tour Options", "avg_options_per_tour", ">1 option"),
    ("Add-ons", "avg_addons_per_tour", "≥1 add-on"),
    ("Individual Tiers", "avg_ind_tiers_per_tour", "≥1 tier"),
    ("Group Tiers", "avg_grp_tiers_per_tour", "≥1 tier")
]

results = []

for metric_name, col, threshold_desc in metrics:
    print(f"\n{metric_name} ({threshold_desc}):")
    
    # Define adopters based on metric
    if "options" in col:
        adopters = top_80[top_80[col] > 1].copy()  # More than 1 option
    else:
        adopters = top_80[top_80[col] > 0].copy()  # At least 1
    
    print(f"  Total adopters: {len(adopters):,} ({100*len(adopters)/len(top_80):.0f}% of top 80%)")
    
    for segment in segments:
        seg_adopters = adopters[adopters["supplier_segment"] == segment]
        
        if len(seg_adopters) > 0:
            mean_val = seg_adopters[col].mean()
            median_val = seg_adopters[col].median()
            p75_val = seg_adopters[col].quantile(0.75)
            
            adopters_pct = 100 * len(seg_adopters) / len(top_80[top_80["supplier_segment"] == segment])
            
            print(f"    {segment}: {len(seg_adopters):,} adopters ({adopters_pct:.0f}%) | Mean={mean_val:.1f}, Median={median_val:.1f}, P75={p75_val:.1f}")
            
            results.append({
                "metric": metric_name,
                "segment": segment,
                "adopters": len(seg_adopters),
                "adopters_pct": adopters_pct,
                "mean": mean_val,
                "median": median_val,
                "p75": p75_val
            })
        else:
            results.append({
                "metric": metric_name,
                "segment": segment,
                "adopters": 0,
                "adopters_pct": 0,
                "mean": 0,
                "median": 0,
                "p75": 0
            })

results_df = pd.DataFrame(results)

# =============================================================================
# CHART: DEPTH HEATMAP WITH ADOPTION CONTEXT
# =============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [1, 1.2]})

# LEFT: Adoption Rate (% of suppliers in segment who use this)
adoption_matrix = []
for segment in segments:
    row = []
    for metric_name, _, _ in metrics:
        metric_data = results_df[(results_df["segment"] == segment) & 
                                  (results_df["metric"] == metric_name)]
        if len(metric_data) > 0:
            row.append(metric_data["adopters_pct"].iloc[0])
        else:
            row.append(0)
    adoption_matrix.append(row)

adoption_matrix = np.array(adoption_matrix)

cmap1 = LinearSegmentedColormap.from_list("grey_orange", [GREY_PALE, ORANGE_PALE, ORANGE, ORANGE_DARK], N=256)
im1 = ax1.imshow(adoption_matrix, aspect="auto", cmap=cmap1, vmin=0, vmax=100)

metric_names = [m[0] for m in metrics]
ax1.set_xticks(np.arange(len(metric_names)))
ax1.set_xticklabels(metric_names, rotation=30, ha='right', fontsize=11, fontweight='bold')
ax1.set_yticks(np.arange(len(segments)))
ax1.set_yticklabels(segments, fontsize=11)
ax1.set_title("Adoption Rate\n(% of suppliers using feature)", fontsize=13, fontweight='bold', pad=15)

for i in range(len(segments)):
    for j in range(len(metric_names)):
        val = adoption_matrix[i, j]
        text_color = "white" if val > 50 else GREY_DARK
        ax1.text(j, i, f"{val:.0f}%", ha="center", va="center", 
                color=text_color, fontsize=11, fontweight="bold")

ax1.set_xticks(np.arange(len(metric_names)) - 0.5, minor=True)
ax1.set_yticks(np.arange(len(segments)) - 0.5, minor=True)
ax1.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax1.tick_params(which="minor", size=0)

# RIGHT: Average Depth (among adopters)
depth_matrix = []
for segment in segments:
    row = []
    for metric_name, _, _ in metrics:
        metric_data = results_df[(results_df["segment"] == segment) & 
                                  (results_df["metric"] == metric_name)]
        if len(metric_data) > 0 and metric_data["adopters"].iloc[0] > 0:
            row.append(metric_data["mean"].iloc[0])
        else:
            row.append(np.nan)
    depth_matrix.append(row)

depth_matrix = np.array(depth_matrix)

# Mask NaN values
masked_depth = np.ma.masked_invalid(depth_matrix)

cmap2 = LinearSegmentedColormap.from_list("grey_orange", [GREY_PALE, ORANGE_PALE, ORANGE, ORANGE_DARK], N=256)
cmap2.set_bad(color='white')
im2 = ax2.imshow(masked_depth, aspect="auto", cmap=cmap2)

ax2.set_xticks(np.arange(len(metric_names)))
ax2.set_xticklabels(metric_names, rotation=30, ha='right', fontsize=11, fontweight='bold')
ax2.set_yticks(np.arange(len(segments)))
ax2.set_yticklabels(segments, fontsize=11)
ax2.set_title("Configuration Depth\n(average per tour, among adopters)", fontsize=13, fontweight='bold', pad=15)

for i in range(len(segments)):
    for j in range(len(metric_names)):
        val = depth_matrix[i, j]
        if not np.isnan(val):
            text_color = "white" if val > np.nanmax(depth_matrix) * 0.5 else GREY_DARK
            ax2.text(j, i, f"{val:.1f}", ha="center", va="center", 
                    color=text_color, fontsize=11, fontweight="bold")

ax2.set_xticks(np.arange(len(metric_names)) - 0.5, minor=True)
ax2.set_yticks(np.arange(len(segments)) - 0.5, minor=True)
ax2.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax2.tick_params(which="minor", size=0)

# Colorbars
cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label("% Adopters", fontsize=11, fontweight='bold')

cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label("Avg per Tour", fontsize=11, fontweight='bold')

plt.suptitle(
    'Question 6: Among Adopters, How Deeply Are Features Configured?\n'
    'Left = Who uses it | Right = How much they use',
    fontsize=16,
    fontweight='bold',
    y=0.98
)

plt.tight_layout()
plt.show()

print("\n✓ Chart: Adoption vs Depth")

# =============================================================================
# KEY INSIGHTS
# =============================================================================

print("\n" + "="*100)
print("KEY INSIGHTS: SHALLOW VS DEEP USAGE")
print("="*100)

for metric_name, col, threshold_desc in metrics:
    metric_results = results_df[results_df["metric"] == metric_name]
    
    # Filter segments with actual adopters
    with_adopters = metric_results[metric_results["adopters"] > 0]
    
    if len(with_adopters) > 0:
        highest = with_adopters.loc[with_adopters["mean"].idxmax()]
        lowest = with_adopters.loc[with_adopters["mean"].idxmin()]
        
        print(f"\n{metric_name}:")
        print(f"  Deepest usage: {highest['segment']} ({highest['mean']:.1f} per tour, {highest['adopters']:,} adopters)")
        print(f"  Shallowest usage: {lowest['segment']} ({lowest['mean']:.1f} per tour, {lowest['adopters']:,} adopters)")
        
        # Classify shallow vs deep
        overall_mean = with_adopters["mean"].mean()
        if overall_mean < 2:
            usage_level = "SHALLOW - Most suppliers checking a box"
        elif overall_mean < 4:
            usage_level = "MODERATE - Some real configuration"
        else:
            usage_level = "DEEP - Suppliers extracting value"
        
        print(f"  Overall pattern: {usage_level} (platform avg: {overall_mean:.1f} per tour)")

print("\n" + "="*100)
print("✓ Analysis complete")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 6: CONFIGURATION DEPTH AMONG ADOPTERS
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons", "num_individual_tiers", "num_group_tiers"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

print("\n" + "="*100)
print("QUESTION 6: HOW DEEPLY DO ADOPTERS CONFIGURE FEATURES?")
print("="*100)
print("Among suppliers who use a feature, how many items do they average per tour?")
print("="*100)

# =============================================================================
# AGGREGATE TO SUPPLIER LEVEL
# =============================================================================

supplier_stats = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "gmv_l12mo": "sum",
          "total_options": "mean",
          "num_addons": "mean",
          "num_individual_tiers": "mean",
          "num_group_tiers": "mean",
          "tour_id": "count"
      })
      .reset_index()
      .rename(columns={
          "gmv_l12mo": "supplier_gmv_l12mo",
          "total_options": "avg_options_per_tour",
          "num_addons": "avg_addons_per_tour",
          "num_individual_tiers": "avg_ind_tiers_per_tour",
          "num_group_tiers": "avg_grp_tiers_per_tour",
          "tour_id": "num_tours"
      })
)

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS
# =============================================================================

active_suppliers = supplier_stats[supplier_stats["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)
top_80 = active_suppliers[active_suppliers["is_top_80"] == 1].copy()

print(f"\nAnalyzing {len(top_80):,} top 80% GMV suppliers")

# =============================================================================
# CALCULATE DEPTH AMONG ADOPTERS ONLY
# =============================================================================

segments = sorted(top_80["supplier_segment"].dropna().unique())

metrics = [
    ("Tour Options", "avg_options_per_tour", ">1"),
    ("Add-ons", "avg_addons_per_tour", "≥1"),
    ("Individual Tiers", "avg_ind_tiers_per_tour", "≥1"),
    ("Group Tiers", "avg_grp_tiers_per_tour", "≥1")
]

results = []

print("\n" + "="*100)
print("CONFIGURATION DEPTH BY SEGMENT (ADOPTERS ONLY)")
print("="*100)

for metric_name, col, threshold_desc in metrics:
    print(f"\n{metric_name} (among suppliers with {threshold_desc} items):")
    
    # Define adopters
    if "options" in col:
        adopters = top_80[top_80[col] > 1].copy()
    else:
        adopters = top_80[top_80[col] > 0].copy()
    
    total_adopters = len(adopters)
    adoption_rate = 100 * total_adopters / len(top_80)
    
    print(f"  Platform-wide: {total_adopters:,} adopters ({adoption_rate:.0f}% of top 80%)")
    
    for segment in segments:
        seg_top_80 = top_80[top_80["supplier_segment"] == segment]
        seg_adopters = adopters[adopters["supplier_segment"] == segment]
        
        if len(seg_adopters) > 0:
            mean_val = seg_adopters[col].mean()
            median_val = seg_adopters[col].median()
            
            seg_adoption_rate = 100 * len(seg_adopters) / len(seg_top_80)
            
            print(f"    {segment}: {len(seg_adopters):,} adopters ({seg_adoption_rate:.0f}%) | "
                  f"Mean={mean_val:.1f}, Median={median_val:.1f} per tour")
            
            results.append({
                "metric": metric_name,
                "segment": segment,
                "adopters": len(seg_adopters),
                "total_segment": len(seg_top_80),
                "adoption_pct": seg_adoption_rate,
                "mean": mean_val,
                "median": median_val
            })
        else:
            results.append({
                "metric": metric_name,
                "segment": segment,
                "adopters": 0,
                "total_segment": len(seg_top_80),
                "adoption_pct": 0,
                "mean": np.nan,
                "median": np.nan
            })

results_df = pd.DataFrame(results)

# =============================================================================
# CHART: CONFIGURATION DEPTH HEATMAP
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 9))

metric_names = [m[0] for m in metrics]

# Build depth matrix
depth_matrix = []
segment_labels = []

for segment in segments:
    row = []
    seg_results = results_df[results_df["segment"] == segment]
    
    # Build label with adopter count context
    total_in_seg = seg_results["total_segment"].iloc[0]
    segment_labels.append(f"{segment}\n(n={total_in_seg:,})")
    
    for metric_name in metric_names:
        metric_data = seg_results[seg_results["metric"] == metric_name]
        if len(metric_data) > 0 and metric_data["adopters"].iloc[0] > 0:
            row.append(metric_data["mean"].iloc[0])
        else:
            row.append(np.nan)
    
    depth_matrix.append(row)

depth_matrix = np.array(depth_matrix)
masked_depth = np.ma.masked_invalid(depth_matrix)

# Colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)
cmap.set_bad(color='white')

im = ax.imshow(masked_depth, aspect="auto", cmap=cmap)

# Axes
ax.set_xticks(np.arange(len(metric_names)))
ax.set_xticklabels(metric_names, fontsize=12, fontweight='bold')
ax.set_yticks(np.arange(len(segments)))
ax.set_yticklabels(segment_labels, fontsize=11)

# Title
ax.set_title(
    'Configuration Depth Among Adopters: Average Items Per Tour\n'
    'Question: Do suppliers use features shallowly (1-2 items) or deeply (5+ items)?',
    fontsize=15,
    fontweight='bold',
    pad=20
)

# Annotate cells with mean values and adopter counts
for i in range(len(segments)):
    for j in range(len(metric_names)):
        val = depth_matrix[i, j]
        
        if not np.isnan(val):
            # Get adopter count for this cell
            seg = segments[i]
            metric = metric_names[j]
            cell_data = results_df[(results_df["segment"] == seg) & 
                                    (results_df["metric"] == metric)]
            
            if len(cell_data) > 0:
                adopters = int(cell_data["adopters"].iloc[0])
                adoption_pct = cell_data["adoption_pct"].iloc[0]
                
                text_color = "white" if val > np.nanmax(depth_matrix) * 0.5 else GREY_DARK
                
                # Main value
                ax.text(j, i, f"{val:.1f}", 
                       ha="center", va="center",
                       color=text_color, fontsize=14, fontweight="bold")
                
                # Adopter count below
                ax.text(j, i + 0.35, f"({adopters:,} = {adoption_pct:.0f}%)", 
                       ha="center", va="center",
                       color=text_color, fontsize=8, style='italic', alpha=0.9)

# Grid
ax.set_xticks(np.arange(len(metric_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segments)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=3)
ax.tick_params(which="minor", size=0)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Average per Tour (among adopters)", fontsize=12, fontweight='bold')

# Explanation footer
explanation = (
    "Reading: Heritage Preserver suppliers who use individual tiers average 9.7 tiers per tour (2% of segment = 2 suppliers)\n"
    "Calculation: (1) Filter to suppliers with feature, (2) Average items per tour per supplier, (3) Average across suppliers"
)
ax.text(0.5, -0.12, explanation, 
        transform=ax.transAxes,
        fontsize=10,
        ha='center',
        bbox=dict(boxstyle="round,pad=0.8", facecolor=GREY_PALE, alpha=0.8, edgecolor=GREY))

plt.tight_layout()
plt.show()

print("\n✓ Chart: Configuration depth heatmap")

# =============================================================================
# KEY INSIGHTS: SHALLOW VS DEEP ENGAGEMENT
# =============================================================================

print("\n" + "="*100)
print("KEY INSIGHTS: ARE SUPPLIERS CHECKING BOXES OR GOING DEEP?")
print("="*100)

for metric_name in metric_names:
    metric_results = results_df[results_df["metric"] == metric_name]
    with_data = metric_results[metric_results["adopters"] > 0].copy()
    
    if len(with_data) > 0:
        overall_mean = with_data["mean"].mean()
        highest = with_data.loc[with_data["mean"].idxmax()]
        lowest = with_data.loc[with_data["mean"].idxmin()]
        
        # Classify depth
        if overall_mean < 2:
            depth_level = "SHALLOW"
            interpretation = "Suppliers are checking a box, not extracting value"
        elif overall_mean < 4:
            depth_level = "MODERATE"
            interpretation = "Some real configuration, but room for deeper engagement"
        else:
            depth_level = "DEEP"
            interpretation = "Suppliers are actively using the feature"
        
        print(f"\n{metric_name}:")
        print(f"  Overall: {depth_level} (platform avg: {overall_mean:.1f} per tour) - {interpretation}")
        print(f"  Deepest: {highest['segment']} ({highest['mean']:.1f} per tour, {int(highest['adopters']):,} suppliers)")
        print(f"  Shallowest: {lowest['segment']} ({lowest['mean']:.1f} per tour, {int(lowest['adopters']):,} suppliers)")

print("\n" + "="*100)
print("CONCLUSION:")
print("="*100)

# Overall summary
tour_options_mean = results_df[(results_df["metric"] == "Tour Options") & 
                                (results_df["adopters"] > 0)]["mean"].mean()
addons_mean = results_df[(results_df["metric"] == "Add-ons") & 
                          (results_df["adopters"] > 0)]["mean"].mean()
ind_tiers_mean = results_df[(results_df["metric"] == "Individual Tiers") & 
                             (results_df["adopters"] > 0)]["mean"].mean()

if tour_options_mean < 3:
    print("Tour Options: SHALLOW engagement (avg 2-3 options). Suppliers offer minimal choice.")
else:
    print(f"Tour Options: MODERATE engagement (avg {tour_options_mean:.1f} options). Suppliers provide variety.")

if addons_mean < 2:
    print("Add-ons: SHALLOW engagement (avg 1-2 add-ons). Upselling is minimal.")
else:
    print(f"Add-ons: MODERATE engagement (avg {addons_mean:.1f} add-ons). Suppliers monetize extras.")

if ind_tiers_mean > 5:
    print(f"Individual Tiers: DEEP engagement (avg {ind_tiers_mean:.1f} tiers). Complex price segmentation.")
elif ind_tiers_mean > 3:
    print(f"Individual Tiers: MODERATE engagement (avg {ind_tiers_mean:.1f} tiers). Standard tiering.")
else:
    print(f"Individual Tiers: SHALLOW engagement (avg {ind_tiers_mean:.1f} tiers). Basic pricing.")

print("\n✓ Analysis complete: Configuration depth")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 2 REVISED: Managed vs Not Managed - SUPPLIER-LEVEL ADOPTION
# Small/Medium/Large based on 33rd and 67th percentiles
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# is_managed is "yes"/"no"
if "is_managed" in df.columns:
    df["is_managed"] = df["is_managed"].astype(str).str.lower().fillna("no")

# =============================================================================
# ALL 11 FEATURES (CONSISTENT WITH QUESTION 1)
# =============================================================================
df["Multiple tour options"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)
df["Individual pricing with add-ons"] = (df.get("has_individual_pricing", 0) & df.get("has_addons", 0)).astype(int)
df["Group pricing with multiple tiers"] = df.get("has_group_scales", 0)
df["Add-ons with multiple tiers"] = df.get("has_addon_scales", 0)

FEATURES = [
    ("Group pricing", "has_group_pricing"),
    ("Add-ons", "has_addons"),
    ("Multiple tour options (>1)", "Multiple tour options"),
    ("Individual pricing with multiple tiers", "has_individual_scales"),
    ("Individual pricing with add-ons", "Individual pricing with add-ons"),
    ("Group pricing with multiple tiers", "Group pricing with multiple tiers"),
    ("Add-ons with multiple tiers", "Add-ons with multiple tiers"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing & availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

print(f"\n✓ Using all {len(FEATURES)} features for analysis")

# =============================================================================
# AGGREGATE TO SUPPLIER LEVEL FIRST (KEY CHANGE!)
# =============================================================================
print("\n" + "="*100)
print("AGGREGATING TO SUPPLIER LEVEL")
print("="*100)

feature_cols = [col for _, col in FEATURES]

# Step 1: Get supplier-level feature adoption (MAX across all their tours)
# If a supplier uses a feature on ANY tour, they count as adopting it
supplier_features = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg(dict([(col, "max") for col in feature_cols] + [
          ("gmv_l12mo", "sum"),
          ("is_managed", "first")
      ]))
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

print(f"\n✓ Aggregated {len(df):,} tours into {len(supplier_features):,} suppliers")
print(f"  - Each supplier marked as adopting a feature if they use it on ANY tour")
print(f"  - GMV summed across all supplier's tours")

# =============================================================================
# CREATE SIZE BUCKETS BASED ON TERTILES (33rd and 67th percentiles)
# =============================================================================

# Filter to suppliers with GMV > 0
active_suppliers = supplier_features[supplier_features["supplier_gmv_l12mo"] > 0].copy()

# Calculate tertiles
p33 = active_suppliers["supplier_gmv_l12mo"].quantile(0.33)
p67 = active_suppliers["supplier_gmv_l12mo"].quantile(0.67)

print("\n" + "="*100)
print("SIZE BUCKET DEFINITIONS (Based on GMV Tertiles)")
print("="*100)
print(f"Small:  €0 - €{p33:,.0f} (bottom 33%)")
print(f"Medium: €{p33:,.0f} - €{p67:,.0f} (middle 33%)")
print(f"Large:  €{p67:,.0f}+ (top 33%)")
print("="*100)

def assign_size_bucket_tertile(gmv):
    """Assign supplier to size bucket based on tertiles"""
    if gmv <= 0:
        return "No GMV"
    elif gmv <= p33:
        return "Small"
    elif gmv <= p67:
        return "Medium"
    else:
        return "Large"

supplier_features["size_bucket"] = supplier_features["supplier_gmv_l12mo"].apply(assign_size_bucket_tertile)

SIZE_BUCKET_ORDER = ["Small", "Medium", "Large"]

print("\nQUESTION 2: MANAGED VS NOT MANAGED - CONTROLLING FOR SUPPLIER SIZE")
print("Question: At the same supplier size, do managed suppliers adopt different features?")
print("Method: Supplier-level analysis (consistent with Question 1)")
print("="*100)

# Show distribution by size and managed status
print("\nSupplier Distribution by Size Bucket and Managed Status:")
print(f"{'Size Bucket':<15} {'Managed':<15} {'Not Managed':<15} {'Total':<15} {'% Managed':<15}")
print("-" * 75)

for bucket in SIZE_BUCKET_ORDER:
    bucket_suppliers = supplier_features[supplier_features["size_bucket"] == bucket]
    managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "yes"])
    not_managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "no"])
    total = managed + not_managed
    pct_managed = 100 * managed / total if total > 0 else 0
    
    print(f"{bucket:<15} {managed:<15,} {not_managed:<15,} {total:<15,} {pct_managed:<15.1f}%")

total_managed = len(supplier_features[supplier_features["is_managed"] == "yes"])
total_not_managed = len(supplier_features[supplier_features["is_managed"] == "no"])
print("-" * 75)
print(f"{'TOTAL':<15} {total_managed:<15,} {total_not_managed:<15,} {total_managed + total_not_managed:<15,}")

# =============================================================================
# CALCULATE ADOPTION BY SIZE BUCKET AND MANAGED STATUS (SUPPLIER-LEVEL)
# =============================================================================
adoption_data = []

for bucket in SIZE_BUCKET_ORDER:
    # Get suppliers in this size bucket
    bucket_suppliers = supplier_features[supplier_features["size_bucket"] == bucket]
    
    managed = bucket_suppliers[bucket_suppliers["is_managed"] == "yes"]
    not_managed = bucket_suppliers[bucket_suppliers["is_managed"] == "no"]
    
    for status_data, status_label in [
        (managed, "Managed"),
        (not_managed, "Not Managed")
    ]:
        if len(status_data) == 0:
            continue
            
        row = {
            "size_bucket": bucket,
            "management_status": status_label,
            "suppliers": len(status_data),
            "gmv": status_data["supplier_gmv_l12mo"].sum(),
        }
        
        # Calculate % of SUPPLIERS (not tours) that have each feature
        for feat_name, col in FEATURES:
            row[feat_name] = 100.0 * status_data[col].mean()
        
        adoption_data.append(row)

adoption_df = pd.DataFrame(adoption_data)

print("\n" + "="*100)
print("SUPPLIER-LEVEL FEATURE ADOPTION RATES")
print("="*100)

for bucket in SIZE_BUCKET_ORDER:
    print(f"\n{bucket} Suppliers:")
    print(f"{'Feature':<45} {'Managed':<15} {'Not Managed':<15} {'Difference':<15}")
    print("-" * 90)
    
    bucket_data = adoption_df[adoption_df["size_bucket"] == bucket]
    managed_data = bucket_data[bucket_data["management_status"] == "Managed"]
    not_managed_data = bucket_data[bucket_data["management_status"] == "Not Managed"]
    
    if len(managed_data) == 0 or len(not_managed_data) == 0:
        print("  Insufficient data")
        continue
    
    for feat_name, _ in FEATURES:
        m_val = float(managed_data[feat_name].iloc[0])
        nm_val = float(not_managed_data[feat_name].iloc[0])
        diff = m_val - nm_val
        
        print(f"{feat_name:<45} {m_val:>12.1f}% {nm_val:>12.1f}% {diff:>12.1f}pp")

# =============================================================================
# CHART 1: GROUPED BAR CHART - Feature Adoption by Size Bucket
# Shows Managed vs Not Managed side-by-side for each size bucket
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(24, 8), sharey=True)

feature_names = [f[0] for f in FEATURES]

for idx, bucket in enumerate(SIZE_BUCKET_ORDER):
    ax = axes[idx]
    
    bucket_data = adoption_df[adoption_df["size_bucket"] == bucket]
    managed_data = bucket_data[bucket_data["management_status"] == "Managed"]
    not_managed_data = bucket_data[bucket_data["management_status"] == "Not Managed"]
    
    x = np.arange(len(feature_names))
    width = 0.35
    
    managed_values = [float(managed_data[feat].iloc[0]) if len(managed_data) > 0 else 0 for feat in feature_names]
    not_managed_values = [float(not_managed_data[feat].iloc[0]) if len(not_managed_data) > 0 else 0 for feat in feature_names]
    
    bars1 = ax.bar(x - width/2, managed_values, width, label='Managed', 
                   color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    bars2 = ax.bar(x + width/2, not_managed_values, width, label='Not Managed', 
                   color=GREY_LIGHT, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    
    # Add value labels on bars (only if >5%)
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 5:
                ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                       f'{height:.0f}%',
                       ha='center', va='bottom', fontsize=7, fontweight='bold', color=GREY_DARK)
    
    # Get supplier counts
    m_suppliers = int(managed_data["suppliers"].iloc[0]) if len(managed_data) > 0 else 0
    nm_suppliers = int(not_managed_data["suppliers"].iloc[0]) if len(not_managed_data) > 0 else 0
    
    # Get GMV range for this bucket
    bucket_suppliers = supplier_features[supplier_features["size_bucket"] == bucket]
    if len(bucket_suppliers) > 0:
        min_gmv = bucket_suppliers["supplier_gmv_l12mo"].min()
        max_gmv = bucket_suppliers["supplier_gmv_l12mo"].max()
        gmv_range = f"€{min_gmv:,.0f}-€{max_gmv:,.0f}"
    else:
        gmv_range = "N/A"
    
    ax.set_title(f'{bucket}\n{gmv_range}\n{m_suppliers} M | {nm_suppliers} NM', 
                fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if idx == 0:
        ax.set_ylabel('Supplier Adoption Rate (%)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10, loc='upper left')

plt.suptitle(
    'Managed vs Not Managed: Supplier-Level Feature Adoption Across Size Buckets\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV\n'
    'Question: At the same supplier size, does managed support drive different adoption patterns?',
    fontsize=14,
    fontweight='bold',
    y=1.00
)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 1: Supplier-level feature adoption by size bucket (all {len(FEATURES)} features)")

# =============================================================================
# CHART 2: HEATMAP - Adoption Rates by Size x Management Status
# Rows = Features, Columns = Size buckets split by Managed/Not Managed
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 12))

# Build matrix: features (rows) x (size_bucket × management_status) (columns)
column_labels = []
matrix = []

for feat_name, _ in FEATURES:
    row = []
    for bucket in SIZE_BUCKET_ORDER:
        for status in ["Managed", "Not Managed"]:
            data = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                              (adoption_df["management_status"] == status)]
            
            if len(data) > 0:
                row.append(data[feat_name].iloc[0])
            else:
                row.append(0)
            
            # Build column labels (only once)
            if len(matrix) == 0:
                supplier_count = int(data["suppliers"].iloc[0]) if len(data) > 0 else 0
                column_labels.append(f"{bucket}\n{status}\n({supplier_count} suppliers)")
    
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Size × Management combinations
ax.set_xticks(np.arange(len(column_labels)))
ax.set_xticklabels(column_labels, rotation=45, ha='right', fontsize=10)

# Y-axis: Features
ax.set_yticks(np.arange(len(feature_names)))
ax.set_yticklabels(feature_names, fontsize=10, fontweight='bold')

# Title
ax.set_title(
    'Supplier-Level Feature Adoption: Managed vs Not Managed Across Supplier Sizes\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV\n'
    'Question: Does managed support drive adoption when controlling for supplier size?',
    fontsize=13,
    fontweight='bold',
    pad=20
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=9, 
               fontweight="bold")

# Add vertical lines to separate size buckets
for i in range(1, len(SIZE_BUCKET_ORDER)):
    ax.axvline(i * 2 - 0.5, color=GREY_DARK, linewidth=3, alpha=0.7)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Supplier Adoption Rate (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(column_labels)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 2: Heatmap showing supplier-level adoption across size buckets (all {len(FEATURES)} features)")

# =============================================================================
# SUMMARY: WHERE DOES MANAGED SUPPORT MATTER?
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Features Where Managed Support Shows Different Adoption (by size bucket)")
print("="*100)

for bucket in SIZE_BUCKET_ORDER:
    bucket_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                  (adoption_df["management_status"] == "Managed")]
    bucket_not_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                      (adoption_df["management_status"] == "Not Managed")]
    
    if len(bucket_managed) == 0 or len(bucket_not_managed) == 0:
        continue
    
    print(f"\n{bucket}:")
    
    differences = []
    for feat_name, _ in FEATURES:
        m_val = float(bucket_managed[feat_name].iloc[0])
        nm_val = float(bucket_not_managed[feat_name].iloc[0])
        diff = m_val - nm_val
        
        if abs(diff) > 5:  # Show differences > 5pp
            differences.append((feat_name, m_val, nm_val, diff))
    
    if differences:
        differences = sorted(differences, key=lambda x: abs(x[3]), reverse=True)
        for feat, m_val, nm_val, diff in differences:
            direction = "higher" if diff > 0 else "lower"
            
            # Mark as significant if >10pp
            if abs(diff) > 10:
                marker = "*** SIGNIFICANT ***"
            else:
                marker = ""
            
            print(f"  {feat}: Managed {m_val:.1f}% vs Not Managed {nm_val:.1f}% ({diff:+.1f}pp {direction}) {marker}")
    else:
        print(f"  No meaningful differences (all features <5pp gap)")

# =============================================================================
# KEY INSIGHTS
# =============================================================================
print("\n" + "="*100)
print("KEY INSIGHTS")
print("="*100)

print("\n1. SIZE EFFECT vs SUPPORT EFFECT:")
print("   Comparing adoption across Small → Medium → Large buckets vs Managed/Not Managed")

# Calculate size effect (average increase from Small to Large)
size_effect_data = []
for feat_name, _ in FEATURES:
    small_avg = adoption_df[adoption_df["size_bucket"] == "Small"][feat_name].mean()
    large_avg = adoption_df[adoption_df["size_bucket"] == "Large"][feat_name].mean()
    size_effect = large_avg - small_avg
    
    # Calculate support effect (average across all sizes)
    managed_avg = adoption_df[adoption_df["management_status"] == "Managed"][feat_name].mean()
    not_managed_avg = adoption_df[adoption_df["management_status"] == "Not Managed"][feat_name].mean()
    support_effect = managed_avg - not_managed_avg
    
    size_effect_data.append({
        "feature": feat_name,
        "size_effect": size_effect,
        "support_effect": support_effect
    })

size_effect_df = pd.DataFrame(size_effect_data).sort_values("size_effect", ascending=False)

print(f"\n   {'Feature':<45} {'Size Effect':<15} {'Support Effect':<15} {'Dominant Factor':<15}")
print("   " + "-" * 90)

for _, row in size_effect_df.iterrows():
    size_eff = row["size_effect"]
    support_eff = row["support_effect"]
    
    if abs(size_eff) > abs(support_eff) * 2:
        dominant = "SIZE"
    elif abs(support_eff) > abs(size_eff) * 2:
        dominant = "SUPPORT"
    else:
        dominant = "BOTH"
    
    print(f"   {row['feature']:<45} {size_eff:>12.1f}pp {support_eff:>12.1f}pp {dominant:>15}")

print("\n2. FEATURES WHERE MANAGED SUPPORT MATTERS MOST:")
features_where_support_matters = size_effect_df[size_effect_df["support_effect"].abs() > 5].sort_values("support_effect", ascending=False)
if len(features_where_support_matters) > 0:
    for _, row in features_where_support_matters.head(5).iterrows():
        print(f"   - {row['feature']}: {row['support_effect']:+.1f}pp higher for managed")
else:
    print("   - No features show >5pp managed support advantage")

print("\n3. FEATURES WHERE SIZE MATTERS MOST:")
features_where_size_matters = size_effect_df[size_effect_df["size_effect"].abs() > 10].sort_values("size_effect", ascending=False)
if len(features_where_size_matters) > 0:
    for _, row in features_where_size_matters.head(5).iterrows():
        print(f"   - {row['feature']}: {row['size_effect']:+.1f}pp increase from Small to Large")
else:
    print("   - No features show >10pp size effect")

print("\n" + "="*100)
print(f"✓ Analysis complete: Managed vs Not Managed (SUPPLIER-LEVEL) with all {len(FEATURES)} features")
print("="*100)
